<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/research%2Fpost-e50-axial-level-pairing/notebooks%2Fpost_e50%2F67C_postE50_axial_level_pairing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 67C -- Post-E50 -- Axial Level Pairing (Sagittal absolute level <-> axial slices)

**Roadmap:** 67A (relative localization) -> 67B (RSNA absolute anchor audit, cerrado) -> 67B1
(orientation + geometry-only window audit) -> 67B2 (**scorer supervisado, LOCKED VALIDATION
completa, cerrado formalmente**) -> **67C (este notebook)**.

## Contexto congelado de 67B2 (no se modifica, solo se cita)

- Base commit de esta rama: `ea2ac297bc470209fdd9dde1ef67ce2d32a771b0`
  (`research/post-e50-absolute-lumbar-window-scorer`).
- Sagittal segmentation checkpoint (frozen, sin cambios desde Notebook 67):
  SHA-256 `cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944`.
- Absolute lumbar window scorer VALIDATED: `absolute_lumbar_window_scorer_validated.pt`,
  SHA-256 `bb9e9d812972305cbc3dc87b96b95f78b7c9de4afc8ac4e6f071b35ae6fffd2c`.
- 67B2 locked validation: **296 estudios** (outer `validation` split completo, una sola corrida).
- `AUTOMATIC_DISC_LOCALIZATION_VALIDATED = True` (67B2, no se toca aqui).

**Ninguno de estos artefactos se modifica en 67C.** `67C` NO reentrena el scorer, NO reejecuta la
locked validation de 67B2, NO accede a `internal_test`, NO accede al test oficial de RSNA, NO toca
`main`, Backend, ni Frontend.

## Objetivo de 67C

Resolver el pairing entre los 5 niveles discales absolutos (`L1-L2 ... L5-S1`, ya resueltos por el
pipeline congelado 67A+67B1+67B2) y los cortes/imagenes **axiales** correspondientes. **Primera
etapa: solucion deterministica GT-free basada exclusivamente en geometria DICOM** -- sin entrenar
ningun modelo todavia.

## Metodologia (4 sub-etapas)

- **67C-A** -- Inventario de geometria axial (outer TRAIN). *Este notebook, en esta revision.*
- **67C-B** -- Desarrollo de pairing geometry-only (outer TRAIN unicamente).
- **67C-C** -- Congelar la regla de pairing / abstencion (antes de tocar outer validation).
- **67C-D** -- Locked E2E validation (unica corrida, GPU L4, sobre outer `validation`).

Durante 67C-A/B se usa **exclusivamente outer TRAIN**. Las labels axiales de outer `validation`
permanecen bloqueadas hasta `67C-C` (freeze). `internal_test` permanece completamente bloqueado en
todas las etapas. `67C_AXIAL_PAIRING_VALIDATED = False` hasta completar la locked validation de
`67C-D`.

## 0 -- Entorno / configuracion deterministica (67C-A)

In [1]:
# ============================================================
# 0 -- Environment / deterministic configuration (67C-A)
# ============================================================

import copy
import hashlib
import json
import math
import os
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Mapping

import numpy as np
import pandas as pd

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

PFI_DRIVE_ROOT = Path(
    os.environ.get(
        "PFI_DRIVE_ROOT",
        "/content/drive/MyDrive/PFI_MVP" if IN_COLAB else "./PFI_MVP",
    )
)

RSNA_ROOT = Path(
    os.environ.get(
        "PFI_RSNA_ROOT",
        str(PFI_DRIVE_ROOT / "data" / "RSNA_LUMBAR_DISC"),
    )
)

# Frozen 67B2 artifacts -- referenced for identity verification ONLY, never modified, never
# reloaded/retrained here. 67C-A does not need the sagittal checkpoint or the scorer at all (pure
# geometry inventory), but the expected hashes are declared up front for later stages (67C-D).
EXPECTED_SAGITTAL_CHECKPOINT_SHA256 = "cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944"
EXPECTED_SCORER_SHA256 = "bb9e9d812972305cbc3dc87b96b95f78b7c9de4afc8ac4e6f071b35ae6fffd2c"
BASE_COMMIT_67B2 = "ea2ac297bc470209fdd9dde1ef67ce2d32a771b0"

RESULTS_DIR = PFI_DRIVE_ROOT / "results" / "post_e50" / "67C"
METRICS_DIR = PFI_DRIVE_ROOT / "metrics" / "post_e50" / "67C"
for _p in (RESULTS_DIR, METRICS_DIR):
    _p.mkdir(parents=True, exist_ok=True)

EXPECTED_CSV_SHA256 = {
    "train.csv": "f0c9e06486bcddcd83b1ee1d95ab9db8e028d7c3c9fcbada950f0b8e4d828528",
    "train_label_coordinates.csv": "416fb434b5bdbf69814210d03dd791dff5ef9a010bac004f5f68e8b79e852635",
    "train_series_descriptions.csv": "bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33",
}

# --- 67C stage/scope control flags ---
RSNA_INTERNAL_TEST_LOCKED = True
OFFICIAL_TEST_ACCESSED = False
ACCESS_OUTER_VALIDATION_AXIAL_GT = False  # <-- stays False until 67C-C freeze is complete.
RUN_LOCKED_VALIDATION = False             # <-- 67C-D, not executed in 67C-A/B.
HEURISTIC_RULES_FROZEN = False            # <-- pairing rule may still be adjusted using outer-TRAIN-internal DEV.
TRAINING_PERFORMED = False                # <-- 67C-A/B/C never train anything.
AUTOMATIC_DISC_LOCALIZATION_VALIDATED = True  # unchanged from 67B2 -- 67C does not touch this flag's meaning.
AXIAL_PAIRING_VALIDATED_67C = False       # <-- this notebook's own outcome flag (spec: "67C_AXIAL_PAIRING_VALIDATED";
                                           # renamed to a valid Python identifier -- cannot start with a digit).
                                           # Stays False until 67C-D locked validation.

print("PFI_DRIVE_ROOT:", PFI_DRIVE_ROOT)
print("RSNA_ROOT:", RSNA_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("METRICS_DIR:", METRICS_DIR)
print("BASE_COMMIT_67B2:", BASE_COMMIT_67B2)
print("RSNA_INTERNAL_TEST_LOCKED:", RSNA_INTERNAL_TEST_LOCKED)
print("OFFICIAL_TEST_ACCESSED:", OFFICIAL_TEST_ACCESSED)
print("ACCESS_OUTER_VALIDATION_AXIAL_GT:", ACCESS_OUTER_VALIDATION_AXIAL_GT)
print("RUN_LOCKED_VALIDATION:", RUN_LOCKED_VALIDATION)
print("AXIAL_PAIRING_VALIDATED_67C:", AXIAL_PAIRING_VALIDATED_67C)

assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False
assert ACCESS_OUTER_VALIDATION_AXIAL_GT is False
assert RUN_LOCKED_VALIDATION is False
assert TRAINING_PERFORMED is False

IN_COLAB: True


MessageError: Error: credential propagation was unsuccessful

## 1 -- Dependencias (CPU-first; SimpleITK/scipy/pydicom, sin GPU en 67C-A)

In [ ]:
# ============================================================
# 1 -- Dependencies (67C-A is CPU-only: no model load, no GPU needed)
# ============================================================

def _importable(name):
    try:
        module = __import__(name)
        return getattr(module, "__version__", "unknown")
    except ImportError:
        return None

required = {
    "SimpleITK": _importable("SimpleITK"),
    "scipy": _importable("scipy"),
    "pydicom": _importable("pydicom"),
}
print(required)

missing = [k for k, v in required.items() if v is None]
if missing:
    packages = []
    if "SimpleITK" in missing:
        packages.append("SimpleITK")
    if "scipy" in missing:
        packages.append("scipy")
    if "pydicom" in missing:
        packages.append("pydicom")
    if packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])

import SimpleITK as sitk
import scipy
import pydicom

print("SimpleITK:", sitk.Version_VersionString())
print("scipy:", scipy.__version__)
print("pydicom:", pydicom.__version__)
print("67C-A is CPU-first: no torch model is loaded, no GPU is requested in this stage.")

{'SimpleITK': None, 'scipy': '1.16.3', 'pydicom': None}
SimpleITK: 2.5.6
scipy: 1.16.3
pydicom: 3.0.2
67C-A is CPU-first: no torch model is loaded, no GPU is requested in this stage.


## 2 -- Identidad del dataset + outer split (reconstruido identico a 67B/67B1/67B2)

In [ ]:
# ============================================================
# 2 -- Dataset identity + outer split (must reproduce 67B2 exactly: seed=2026, 1382/296/297)
# ============================================================

CANONICAL_LEVELS = ["L1-L2", "L2-L3", "L3-L4", "L4-L5", "L5-S1"]
PRIMARY_REFERENCE_CONDITION = "Spinal Canal Stenosis"
PRIMARY_REFERENCE_SERIES = "Sagittal T2/STIR"


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def opaque_id(raw_value) -> str:
    return hashlib.sha256(str(raw_value).encode("utf-8")).hexdigest()[:12]


def normalize_level(raw):
    if raw is None:
        return None
    try:
        if pd.isna(raw):
            return None
    except Exception:
        pass
    text = str(raw).strip().upper().replace("/", "-").replace("_", "-")
    return text if text in CANONICAL_LEVELS else None


def build_reproducible_study_split(study_ids, seed=2026, train_frac=0.70, validation_frac=0.15):
    ordered = sorted(str(x) for x in study_ids)
    rng = np.random.default_rng(seed)
    shuffled = list(ordered)
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = int(n * train_frac)
    n_val = int(n * validation_frac)
    return {
        "train": sorted(shuffled[:n_train]),
        "validation": sorted(shuffled[n_train:n_train + n_val]),
        "internal_test": sorted(shuffled[n_train + n_val:]),
    }


warnings: list[str] = []

GATE_A_RSNA_dataset_structure = "NOT_RUN"
train_df = pd.DataFrame()
coords_df = pd.DataFrame(columns=["study_id", "series_id", "level", "condition", "instance_number", "x", "y"])
series_df = pd.DataFrame(columns=["study_id", "series_id", "series_description"])
outer_split = {"train": [], "validation": [], "internal_test": []}
TRAIN_STUDY_IDS: set = set()

_csv_files_present = all((RSNA_ROOT / f).is_file() for f in EXPECTED_CSV_SHA256)

if _csv_files_present:
    for filename, expected in EXPECTED_CSV_SHA256.items():
        path = RSNA_ROOT / filename
        actual = sha256_file(path)
        print(filename, actual, "MATCH" if actual == expected else "MISMATCH")
        assert actual == expected, f"{filename} SHA mismatch -- STOP, do not proceed on an unverified dataset."

    train_df = pd.read_csv(RSNA_ROOT / "train.csv", dtype={"study_id": "string"})
    coords_df = pd.read_csv(RSNA_ROOT / "train_label_coordinates.csv", dtype={"study_id": "string", "series_id": "string"})
    series_df = pd.read_csv(RSNA_ROOT / "train_series_descriptions.csv", dtype={"study_id": "string", "series_id": "string"})

    print("shapes:", train_df.shape, coords_df.shape, series_df.shape)
    assert train_df.shape == (1975, 26)
    assert coords_df.shape == (48692, 7)
    assert series_df.shape == (6294, 3)

    coords_df = coords_df.copy()
    coords_df["level_raw"] = coords_df["level"]
    coords_df["level_normalized"] = coords_df["level_raw"].apply(normalize_level)

    outer_split = build_reproducible_study_split(series_df["study_id"].dropna().unique())
    print({k: len(v) for k, v in outer_split.items()})

    assert len(outer_split["train"]) == 1382
    assert len(outer_split["validation"]) == 296
    assert len(outer_split["internal_test"]) == 297
    assert set(outer_split["train"]).isdisjoint(outer_split["validation"])
    assert set(outer_split["train"]).isdisjoint(outer_split["internal_test"])
    assert set(outer_split["validation"]).isdisjoint(outer_split["internal_test"])
    print("Outer split leakage audit: PASS -- identical reconstruction to 67B/67B1/67B2 (seed=2026).")

    TRAIN_STUDY_IDS = set(outer_split["train"])
    GATE_A_RSNA_dataset_structure = "PASS"
    print("67C-A/B scope: outer TRAIN only (n=1382). validation (296) and internal_test (297) untouched.")
else:
    warnings.append("RSNA CSVs not found in this environment -- GATE_A_RSNA_dataset_structure NOT_RUN (graceful local degradation, no fabricated evidence).")
    coords_df["level_raw"] = pd.Series(dtype=object)
    coords_df["level_normalized"] = pd.Series(dtype=object)
    print("RSNA CSVs not found -- GATE_A_RSNA_dataset_structure: NOT_RUN")

print("GATE_A_RSNA_dataset_structure:", GATE_A_RSNA_dataset_structure)

train.csv f0c9e06486bcddcd83b1ee1d95ab9db8e028d7c3c9fcbada950f0b8e4d828528 MATCH
train_label_coordinates.csv 416fb434b5bdbf69814210d03dd791dff5ef9a010bac004f5f68e8b79e852635 MATCH
train_series_descriptions.csv bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33 MATCH
shapes: (1975, 26) (48692, 7) (6294, 3)
{'train': 1382, 'validation': 296, 'internal_test': 297}
Outer split leakage audit: PASS -- identical reconstruction to 67B/67B1/67B2 (seed=2026).
67C-A/B scope: outer TRAIN only (n=1382). validation (296) and internal_test (297) untouched.
GATE_A_RSNA_dataset_structure: PASS


## 3 -- Inventario dinamico de `series_description` (outer TRAIN, sin asumir nombres)

Scope: **outer TRAIN unicamente** (1382 studies) -- las labels axiales de `validation`/`internal_test`
no se tocan aqui, ni siquiera para este inventario descriptivo.

In [ ]:
# ============================================================
# 3 -- Dynamic series_description inventory (outer TRAIN only)
# ============================================================

series_df_train = pd.DataFrame(columns=series_df.columns)
coords_df_train = pd.DataFrame(columns=coords_df.columns)
series_description_counts = pd.Series(dtype="int64")

if GATE_A_RSNA_dataset_structure == "PASS":
    series_df_train = series_df[series_df["study_id"].isin(TRAIN_STUDY_IDS)].copy()
    coords_df_train = coords_df[coords_df["study_id"].isin(TRAIN_STUDY_IDS)].copy()

    series_description_counts = series_df_train["series_description"].value_counts(dropna=False)
    print("Distinct series_description values observed in outer TRAIN (series-level counts):")
    print(series_description_counts.to_string())
    print()
    print("Total series rows (outer TRAIN):", len(series_df_train))
    print("Distinct studies with at least one series (outer TRAIN):", series_df_train["study_id"].nunique())
else:
    warnings.append("Series inventory (step 3) NOT_RUN: GATE_A did not PASS in this run.")
    print("GATE_A did not PASS -- series_description inventory NOT_RUN.")

Distinct series_description values observed in outer TRAIN (series-level counts):
series_description
Axial T2            1632
Sagittal T1         1387
Sagittal T2/STIR    1381

Total series rows (outer TRAIN): 4400
Distinct studies with at least one series (outer TRAIN): 1382


## 4 -- Identificar series axiales usadas por las referencias RSNA (sin asumir el nombre)

No se filtra por un string de `series_description` hardcodeado. Se construye el crosstab real
`condition x series_description` (join `coords_df_train` + `series_df_train` por
`study_id`+`series_id`) y se imprime completo -- la clasificacion "axial" se deriva de que el valor
observado de `series_description` contenga la subcadena `AXIAL` (dato real, no asumido), y se
imprime explicitamente cuales valores fueron clasificados asi y con que conteo.

In [ ]:
# ============================================================
# 4 -- Axial series identification (data-driven, never a silent hardcoded name)
# ============================================================

axial_series_descriptions_observed: list = []
reference_joined_train = pd.DataFrame()
axial_reference_rows_train = pd.DataFrame(columns=list(coords_df_train.columns) + ["series_description"])


def is_axial_series_description(desc) -> bool:
    # Data-driven substring check against the ACTUAL observed series_description values -- never a
    # single hardcoded exact-match string. Printed explicitly below so the classification is
    # auditable, not silent.
    return "AXIAL" in str(desc).strip().upper()


if GATE_A_RSNA_dataset_structure == "PASS":
    _series_lookup_train = series_df_train[["study_id", "series_id", "series_description"]].drop_duplicates()
    reference_joined_train = coords_df_train.merge(_series_lookup_train, on=["study_id", "series_id"], how="left")

    print("Distinct condition values observed (outer TRAIN, GT reference rows):")
    print(sorted(reference_joined_train["condition"].dropna().unique().tolist()))
    print()

    condition_series_crosstab = pd.crosstab(reference_joined_train["condition"], reference_joined_train["series_description"], dropna=False)
    print("condition x series_description crosstab (outer TRAIN, full, nothing filtered):")
    print(condition_series_crosstab.to_string())
    print()

    axial_series_descriptions_observed = sorted(
        d for d in series_description_counts.index if isinstance(d, str) and is_axial_series_description(d)
    )
    print("series_description values classified as AXIAL (substring match on real observed values):")
    for _d in axial_series_descriptions_observed:
        print(f"  - {_d!r}: {int(series_description_counts.get(_d, 0))} series rows (outer TRAIN)")
    print()

    axial_reference_rows_train = reference_joined_train[reference_joined_train["series_description"].isin(axial_series_descriptions_observed)].copy()
    print("GT reference rows (outer TRAIN) landing on an AXIAL series_description:", len(axial_reference_rows_train))
    print("Conditions observed among those axial-series reference rows:", sorted(axial_reference_rows_train["condition"].dropna().unique().tolist()))
    print("Studies (outer TRAIN) with at least one axial-series GT reference row:", axial_reference_rows_train["study_id"].nunique())
else:
    warnings.append("Axial series identification (step 4) NOT_RUN: GATE_A did not PASS in this run.")
    print("GATE_A did not PASS -- axial series identification NOT_RUN.")

Distinct condition values observed (outer TRAIN, GT reference rows):
['Left Neural Foraminal Narrowing', 'Left Subarticular Stenosis', 'Right Neural Foraminal Narrowing', 'Right Subarticular Stenosis', 'Spinal Canal Stenosis']

condition x series_description crosstab (outer TRAIN, full, nothing filtered):
series_description                Axial T2  Sagittal T1  Sagittal T2/STIR
condition                                                                
Left Neural Foraminal Narrowing          0         6905                 0
Left Subarticular Stenosis            6722            0                 0
Right Neural Foraminal Narrowing         0         6905                 0
Right Subarticular Stenosis           6725            0                 0
Spinal Canal Stenosis                    0            5              6811

series_description values classified as AXIAL (substring match on real observed values):
  - 'Axial T2': 1632 series rows (outer TRAIN)

GT reference rows (outer TRAIN) landi

## 5 -- Tabla anonimizada por study (outer TRAIN) -- geometria axial + metadata

Solo hashes opacos (12 caracteres SHA-256) -- **nunca** `study_id`/`series_id` crudos, ni en
memoria persistida ni en ningun print. `RSNA_AVAILABLE` controla si esta celda puede ejecutar
(requiere `train_images/` real); si no esta disponible, degrada a `NOT_RUN` explicito.

In [ ]:
# ============================================================
# 5 -- Per-study anonymized axial geometry + metadata inventory (outer TRAIN only)
# ============================================================

RSNA_AVAILABLE = (RSNA_ROOT / "train_images").is_dir()
print("RSNA_AVAILABLE:", RSNA_AVAILABLE)

GATE_67C_A_AXIAL_INVENTORY = "NOT_RUN"
axial_inventory_rows = []       # one row per outer-TRAIN study -- opaque only
metadata_error_rows = []        # one row per detected duplicate/incomplete GT reference -- opaque only

if RSNA_AVAILABLE:
    train_images_root = RSNA_ROOT / "train_images"
    _t0 = time.time()

    for _i, study_id in enumerate(sorted(TRAIN_STUDY_IDS)):
        study_opaque = opaque_id(study_id)

        study_axial_series = series_df_train[
            (series_df_train["study_id"] == study_id)
            & (series_df_train["series_description"].isin(axial_series_descriptions_observed))
        ]
        n_axial_series = len(study_axial_series)

        slices_per_series = []
        for _, srow in study_axial_series.iterrows():
            series_dir = train_images_root / str(study_id) / str(srow["series_id"])
            n_slices = len(list(series_dir.glob("*.dcm"))) if series_dir.is_dir() else 0
            slices_per_series.append(n_slices)

        study_axial_gt = axial_reference_rows_train[axial_reference_rows_train["study_id"] == study_id]

        seen_series_condition_level = set()
        distinct_levels_seen = set()
        conditions_present = set()
        unrecognized_level_count = 0
        incomplete_metadata_count = 0
        duplicate_count = 0

        for _, r in study_axial_gt.iterrows():
            condition = r.get("condition")
            conditions_present.add(condition)
            level_norm = r.get("level_normalized")
            series_id_r = r.get("series_id")

            if pd.isna(r.get("instance_number")) or pd.isna(r.get("x")) or pd.isna(r.get("y")):
                incomplete_metadata_count += 1

            if level_norm is None:
                unrecognized_level_count += 1
                continue

            dedup_key = (series_id_r, condition, level_norm)
            if dedup_key in seen_series_condition_level:
                duplicate_count += 1
                metadata_error_rows.append({"study_id_opaque": study_opaque, "error_type": "duplicate_level", "condition": condition, "level_normalized": level_norm})
            else:
                seen_series_condition_level.add(dedup_key)
                distinct_levels_seen.add(level_norm)

        distinct_levels_available = len(distinct_levels_seen)

        axial_inventory_rows.append({
            "study_id_opaque": study_opaque,
            "n_axial_series": n_axial_series,
            "slices_per_axial_series": slices_per_series,
            "gt_axial_levels_available": distinct_levels_available,
            "conditions_present": sorted(c for c in conditions_present if c is not None),
            "axial_series_descriptions": sorted(study_axial_series["series_description"].dropna().unique().tolist()),
            "unrecognized_level_count": unrecognized_level_count,
            "incomplete_metadata_count": incomplete_metadata_count,
            "duplicate_level_count": duplicate_count,
        })

        if (_i + 1) % 200 == 0 or (_i + 1) == len(TRAIN_STUDY_IDS):
            print(f"Axial inventory: {_i + 1}/{len(TRAIN_STUDY_IDS)} studies | elapsed={time.time() - _t0:.1f}s")

    total_with_axial = sum(1 for r in axial_inventory_rows if r["n_axial_series"] > 0)
    total_metadata_errors = sum(r["unrecognized_level_count"] + r["incomplete_metadata_count"] + r["duplicate_level_count"] for r in axial_inventory_rows)
    GATE_67C_A_AXIAL_INVENTORY = "PASS" if axial_inventory_rows else "FAIL"

    print()
    print("Studies (outer TRAIN) with >=1 axial series:", total_with_axial, "/", len(TRAIN_STUDY_IDS))
    print("Total metadata-error events (unrecognized + incomplete + duplicate):", total_metadata_errors)
else:
    print("RSNA train_images/ not found in this environment -- axial inventory NOT_RUN (graceful degradation, no fabricated evidence).")

RSNA_AVAILABLE: True
Axial inventory: 200/1382 studies | elapsed=149.4s
Axial inventory: 400/1382 studies | elapsed=297.5s
Axial inventory: 600/1382 studies | elapsed=443.5s
Axial inventory: 800/1382 studies | elapsed=592.0s
Axial inventory: 1000/1382 studies | elapsed=739.6s
Axial inventory: 1200/1382 studies | elapsed=884.9s
Axial inventory: 1382/1382 studies | elapsed=1017.6s

Studies (outer TRAIN) with >=1 axial series: 1382 / 1382
Total metadata-error events (unrecognized + incomplete + duplicate): 0


## Artefactos 67C-A (`results/post_e50/67C/`, `metrics/post_e50/67C/`) -- solo opacos, no sobrescribe 67B2

In [ ]:
# ============================================================
# Artifacts -- opaque only, never overwrite any 67B/67B1/67B2 path
# ============================================================

ALLOWED_WRITE_ROOTS_67C = [RESULTS_DIR, METRICS_DIR]


def safe_write_text(path: Path, content: str) -> None:
    resolved = path.resolve()
    if not any(str(resolved).startswith(str(root.resolve())) for root in ALLOWED_WRITE_ROOTS_67C):
        raise PermissionError(f"Refusing to write outside 67C's allowed roots: {resolved}")
    if "C:\\Users\\" in content or "/Users/" in content:
        raise ValueError("Refusing to persist content containing a local user path.")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def _rows_to_csv_text(rows: list) -> str:
    if not rows:
        return ""
    return pd.DataFrame(rows).to_csv(index=False)


safe_write_text(RESULTS_DIR / "67C_A_axial_inventory.csv", _rows_to_csv_text(axial_inventory_rows))
safe_write_text(RESULTS_DIR / "67C_A_metadata_errors.csv", _rows_to_csv_text(metadata_error_rows))

series_description_counts_dict = {str(k): int(v) for k, v in series_description_counts.items()}
axial_series_descriptions_with_counts = {d: int(series_description_counts.get(d, 0)) for d in axial_series_descriptions_observed}
conditions_axial_observed = sorted(c for c in axial_reference_rows_train["condition"].dropna().unique().tolist())

pfi_67c_a_summary = {
    "notebook": "67C_postE50_axial_level_pairing",
    "stage": "67C-A",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "base_commit_67b2": BASE_COMMIT_67B2,
    "expected_sagittal_checkpoint_sha256": EXPECTED_SAGITTAL_CHECKPOINT_SHA256,
    "expected_scorer_sha256": EXPECTED_SCORER_SHA256,
    "automatic_disc_localization_validated": AUTOMATIC_DISC_LOCALIZATION_VALIDATED,
    "axial_pairing_validated_67c": AXIAL_PAIRING_VALIDATED_67C,
    "rsna_available": RSNA_AVAILABLE,
    "outer_split_counts": {k: len(v) for k, v in outer_split.items()},
    "gate_67c_a_axial_inventory": GATE_67C_A_AXIAL_INVENTORY,
    "series_descriptions_observed": series_description_counts_dict,
    "studies_with_axial_train": int(sum(1 for r in axial_inventory_rows if r["n_axial_series"] > 0)),
    "axial_series_observed_train": int(sum(r["n_axial_series"] for r in axial_inventory_rows)),
    "gt_axial_rows_train": int(len(axial_reference_rows_train)),
    "gt_axial_studies_train": int(axial_reference_rows_train["study_id"].nunique()) if RSNA_AVAILABLE else 0,
    "levels_available_distribution": ({str(k): int(v) for k, v in pd.Series([r["gt_axial_levels_available"] for r in axial_inventory_rows]).value_counts().to_dict().items()} if axial_inventory_rows else {}),
    "conditions_axial_observed": conditions_axial_observed,
    "axial_series_descriptions_observed": axial_series_descriptions_with_counts,
    "metadata_error_events": len(metadata_error_rows),
    "validation_axial_labels_accessed": False,
    "internal_test_accessed": False,
    "official_test_accessed": False,
    "training_performed": False,
}
safe_write_text(METRICS_DIR / "67C_A_summary.json", json.dumps(pfi_67c_a_summary, indent=2, default=str))

print("Artifacts written under:")
print(" -", RESULTS_DIR / "67C_A_axial_inventory.csv")
print(" -", RESULTS_DIR / "67C_A_metadata_errors.csv")
print(" -", METRICS_DIR / "67C_A_summary.json")

Artifacts written under:
 - /content/drive/MyDrive/PFI_MVP/results/post_e50/67C/67C_A_axial_inventory.csv
 - /content/drive/MyDrive/PFI_MVP/results/post_e50/67C/67C_A_metadata_errors.csv
 - /content/drive/MyDrive/PFI_MVP/metrics/post_e50/67C/67C_A_summary.json


## 67C-A -- Cierre (imprime el bloque de finalizacion requerido)

In [ ]:
# ============================================================
# 67C-A CLOSING BLOCK
# ============================================================

print("=" * 60)
print("67C-A AXIAL INVENTORY COMPLETE")
print("=" * 60)
print()
print("series descriptions:", series_description_counts_dict)
print()
print("studies con axial:", pfi_67c_a_summary["studies_with_axial_train"], "/", len(TRAIN_STUDY_IDS))
print("series axiales:", pfi_67c_a_summary["axial_series_observed_train"])
print("GT axial rows:", pfi_67c_a_summary["gt_axial_rows_train"])
print("GT axial studies:", pfi_67c_a_summary["gt_axial_studies_train"])
print("levels disponibles (distribucion por study):", pfi_67c_a_summary["levels_available_distribution"])
print("condiciones axiales:", pfi_67c_a_summary["conditions_axial_observed"])
print("errores de metadata:", pfi_67c_a_summary["metadata_error_events"])
print()
print("validation axial labels accessed:", pfi_67c_a_summary["validation_axial_labels_accessed"])
print("internal test accessed:", pfi_67c_a_summary["internal_test_accessed"])
print("official test accessed:", pfi_67c_a_summary["official_test_accessed"])
print()
print("GATE_67C_A_AXIAL_INVENTORY:", GATE_67C_A_AXIAL_INVENTORY)
print("AXIAL_PAIRING_VALIDATED_67C:", AXIAL_PAIRING_VALIDATED_67C, "(unchanged -- stays False until 67C-D locked validation)")
print("AUTOMATIC_DISC_LOCALIZATION_VALIDATED:", AUTOMATIC_DISC_LOCALIZATION_VALIDATED, "(unchanged from 67B2)")

67C-A AXIAL INVENTORY COMPLETE

series descriptions: {'Axial T2': 1632, 'Sagittal T1': 1387, 'Sagittal T2/STIR': 1381}

studies con axial: 1382 / 1382
series axiales: 1632
GT axial rows: 13447
GT axial studies: 1381
levels disponibles (distribucion por study): {'5': 1261, '4': 65, '3': 53, '2': 1, '0': 1, '1': 1}
condiciones axiales: ['Left Subarticular Stenosis', 'Right Subarticular Stenosis']
errores de metadata: 0

validation axial labels accessed: False
internal test accessed: False
official test accessed: False

GATE_67C_A_AXIAL_INVENTORY: PASS
AXIAL_PAIRING_VALIDATED_67C: False (unchanged -- stays False until 67C-D locked validation)
AUTOMATIC_DISC_LOCALIZATION_VALIDATED: True (unchanged from 67B2)


# 67C-B -- Desarrollo de pairing geometria-only (outer TRAIN unicamente)

Algoritmo GT-free: `nivel discal absoluto sagital -> punto fisico XYZ -> geometria individual de
cada slice axial DICOM -> slice/bloque axial correspondiente`. **Exclusivamente outer TRAIN.** No se
accede a GT axial de `validation`, no se accede a `internal_test`, no se accede al test oficial. No
se entrena ninguna red neuronal. Los flags de scope (`ACCESS_OUTER_VALIDATION_AXIAL_GT`,
`RUN_LOCKED_VALIDATION`, `AXIAL_PAIRING_VALIDATED_67C`) permanecen `False`.

Los 5 `ABSOLUTE_LEVEL_REFERENCE_POINT` sagitales de RSNA se usan aqui como **herramienta de
desarrollo geometrico** (para construir y auditar la regla) -- nunca como feature del runtime final,
nunca disponibles en inferencia real. El GT axial se usa unicamente **post-hoc**, despues de que la
prediccion GT-free ya fue producida.

In [ ]:
assert ACCESS_OUTER_VALIDATION_AXIAL_GT is False
assert RUN_LOCKED_VALIDATION is False
assert AXIAL_PAIRING_VALIDATED_67C is False
print("67C-B scope safety: PASS (outer TRAIN only, no validation/internal_test/official-test access, no training).")

67C-B scope safety: PASS (outer TRAIN only, no validation/internal_test/official-test access, no training).


## Geometria DICOM base (misma formula validada en 65/66/67/67A/67B/67B1)

`dicom_pixel_to_patient_xyz` -- cross-validada contra SimpleITK en notebooks previos, redefinida
aqui explicitamente (67C no importa 67B1/67B2 como modulo).

In [ ]:
def dicom_pixel_to_patient_xyz(column_index: float, row_index: float, pixel_spacing: tuple, image_position_patient: tuple, image_orientation_patient: tuple) -> np.ndarray:
    # P = IPP + column_index * column_spacing * row_cosines + row_index * row_spacing * column_cosines
    row_cosines = np.array(image_orientation_patient[0:3], dtype=np.float64)
    column_cosines = np.array(image_orientation_patient[3:6], dtype=np.float64)
    origin = np.array(image_position_patient, dtype=np.float64)
    row_spacing, column_spacing = pixel_spacing
    return origin + column_index * column_spacing * row_cosines + row_index * row_spacing * column_cosines


def check_xy_inside_frame(column_index: float, row_index: float, rows: int, columns: int) -> bool:
    return (0 <= column_index < columns) and (0 <= row_index < rows)


def check_geometry_finite(*values: float) -> bool:
    return all(np.isfinite(v) for v in values)


def check_orientation_valid(image_orientation_patient: tuple) -> bool:
    if len(image_orientation_patient) != 6:
        return False
    row_cosines = np.array(image_orientation_patient[0:3], dtype=np.float64)
    column_cosines = np.array(image_orientation_patient[3:6], dtype=np.float64)
    return bool(abs(np.linalg.norm(row_cosines) - 1.0) < 1e-2 and abs(np.linalg.norm(column_cosines) - 1.0) < 1e-2 and abs(float(np.dot(row_cosines, column_cosines))) < 1e-2)


# Synthetic self-tests (independent of RSNA availability):
_identity_xyz = dicom_pixel_to_patient_xyz(10.0, 20.0, (1.0, 1.0), (0.0, 0.0, 0.0), (1.0, 0.0, 0.0, 0.0, 1.0, 0.0))
assert np.allclose(_identity_xyz, [10.0, 20.0, 0.0]), "dicom_pixel_to_patient_xyz self-test FAILED (identity)"
_sitk_check_image = sitk.Image(64, 64, 1, sitk.sitkUInt8)
_sitk_check_image.SetSpacing((0.5, 0.7, 1.0))
_sitk_check_image.SetOrigin((100.0, -50.0, 25.0))
_sitk_check_image.SetDirection((0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0))
_sitk_xyz = np.array(_sitk_check_image.TransformContinuousIndexToPhysicalPoint((5.0, 7.0, 0.0)))
_manual_xyz = dicom_pixel_to_patient_xyz(5.0, 7.0, (0.7, 0.5), (100.0, -50.0, 25.0), (0.0, 1.0, 0.0, 1.0, 0.0, 0.0))
assert np.allclose(_sitk_xyz, _manual_xyz, atol=1e-6), "dicom_pixel_to_patient_xyz self-test FAILED (SimpleITK cross-check)"
assert check_xy_inside_frame(10, 10, 64, 64) is True and check_xy_inside_frame(-1, 10, 64, 64) is False, "check_xy_inside_frame self-test FAILED"
assert check_geometry_finite(1.0, 2.0) is True and check_geometry_finite(1.0, float("nan")) is False, "check_geometry_finite self-test FAILED"
assert check_orientation_valid((1.0, 0.0, 0.0, 0.0, 1.0, 0.0)) is True, "check_orientation_valid self-test FAILED"
print("dicom_pixel_to_patient_xyz (cross-validated against SimpleITK) / check_xy_inside_frame / check_geometry_finite / check_orientation_valid: synthetic self-tests PASSED.")

dicom_pixel_to_patient_xyz (cross-validated against SimpleITK) / check_xy_inside_frame / check_geometry_finite / check_orientation_valid: synthetic self-tests PASSED.


## 1 -- Split interno de desarrollo (67C_train=80% / 67C_dev=20%, seed=2026, study-level)

Determinista, study-level, sin leakage. Scope = **outer TRAIN unicamente** (`TRAIN_ONLY`) -- outer
`validation`/`internal_test` no participan de este split interno. El DEV interno **nunca** entrena
un modelo (67C-B es geometry-only): se usa solo para elegir/fijar reglas geometricas y de
abstencion, nunca para tunear contra outer validation.

In [ ]:
def build_internal_67c_split(study_ids, seed=2026, train_frac=0.80):
    ordered = sorted(str(x) for x in study_ids)
    rng = np.random.default_rng(seed)
    shuffled = list(ordered)
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = int(round(n * train_frac))
    return {"67C_train": sorted(shuffled[:n_train]), "67C_dev": sorted(shuffled[n_train:])}


_syn_ids = [f"s{i}" for i in range(100)]
_syn_split_a = build_internal_67c_split(_syn_ids)
_syn_split_b = build_internal_67c_split(_syn_ids)
assert _syn_split_a == _syn_split_b, "build_internal_67c_split self-test FAILED (not deterministic)"
assert set(_syn_split_a["67C_train"]) | set(_syn_split_a["67C_dev"]) == set(_syn_ids), "build_internal_67c_split self-test FAILED (coverage)"
assert set(_syn_split_a["67C_train"]).isdisjoint(_syn_split_a["67C_dev"]), "build_internal_67c_split self-test FAILED (leakage)"
assert len(_syn_split_a["67C_train"]) == 80 and len(_syn_split_a["67C_dev"]) == 20, "build_internal_67c_split self-test FAILED (80/20 ratio)"
print("build_internal_67c_split: synthetic self-tests PASSED (deterministic, disjoint, 80/20).")

internal_67c_split = {"67C_train": [], "67C_dev": []}
if GATE_A_RSNA_dataset_structure == "PASS":
    internal_67c_split = build_internal_67c_split(sorted(TRAIN_STUDY_IDS))
    assert set(internal_67c_split["67C_train"]).issubset(TRAIN_STUDY_IDS)
    assert set(internal_67c_split["67C_dev"]).issubset(TRAIN_STUDY_IDS)
    assert set(internal_67c_split["67C_train"]).isdisjoint(internal_67c_split["67C_dev"])
    assert set(internal_67c_split["67C_train"]) | set(internal_67c_split["67C_dev"]) == TRAIN_STUDY_IDS
    print("67C internal split counts:", {k: len(v) for k, v in internal_67c_split.items()})
else:
    warnings.append("67C internal split NOT_RUN: GATE_A did not PASS in this run.")
    print("GATE_A did not PASS -- 67C internal split NOT_RUN.")

internal_67c_split_manifest = {
    "seed": 2026,
    "outer_scope": "TRAIN_ONLY",
    "train_frac": 0.80,
    "counts": {k: len(v) for k, v in internal_67c_split.items()},
    "67C_train_opaque": sorted(opaque_id(sid) for sid in internal_67c_split["67C_train"]),
    "67C_dev_opaque": sorted(opaque_id(sid) for sid in internal_67c_split["67C_dev"]),
}
print("67C_internal_split_manifest ready (opaque only). counts:", internal_67c_split_manifest["counts"])

build_internal_67c_split: synthetic self-tests PASSED (deterministic, disjoint, 80/20).
67C internal split counts: {'67C_train': 1106, '67C_dev': 276}
67C_internal_split_manifest ready (opaque only). counts: {'67C_train': 1106, '67C_dev': 276}


## 2 -- Puntos discales sagitales de desarrollo (`ABSOLUTE_LEVEL_REFERENCE_POINT`, outer TRAIN)

Herramienta de desarrollo geometrico -- **nunca** feature del runtime final, **nunca** disponible en
inferencia real. QC explicito por study: 5 niveles completos, IOP valido, coordenadas dentro de
imagen, XYZ finito, niveles unicos (sin duplicados).

In [ ]:
def resolve_dicom_instance_path(train_images_root: Path, study_id: str, series_id: str, instance_number: int):
    series_dir = train_images_root / str(study_id) / str(series_id)
    if not series_dir.is_dir():
        return None
    candidate = series_dir / f"{instance_number}.dcm"
    if candidate.is_file():
        return candidate
    matches = list(series_dir.glob(f"*{instance_number}*.dcm"))
    return matches[0] if matches else None


def resolve_sagittal_gt_disc_xyz_for_study(study_id_raw: str, train_images_root: Path) -> dict:
    # Development tool ONLY -- resolves the 5 ABSOLUTE_LEVEL_REFERENCE_POINT (Spinal Canal
    # Stenosis, Sagittal T2/STIR) to physical XYZ for one outer-TRAIN study. Returns
    # {"xyz": {level: np.ndarray}, "qc": {...}}. Never called for validation/internal_test studies.
    rows_this_study = reference_joined_train[
        (reference_joined_train["study_id"] == study_id_raw)
        & (reference_joined_train["condition"] == PRIMARY_REFERENCE_CONDITION)
        & (reference_joined_train["series_description"] == PRIMARY_REFERENCE_SERIES)
    ]

    xyz_by_level = {}
    qc = {
        "five_levels_complete": False, "iop_valid": True, "coords_inside_image": True,
        "xyz_finite": True, "levels_unique": True, "rows_seen": int(len(rows_this_study)),
    }

    seen_levels = []
    for _, row in rows_this_study.iterrows():
        level = row.get("level_normalized")
        if level not in CANONICAL_LEVELS:
            continue
        if level in seen_levels:
            qc["levels_unique"] = False
            continue
        seen_levels.append(level)

        series_id_raw = row["series_id"]
        instance_number = int(row["instance_number"])
        dcm_path = resolve_dicom_instance_path(train_images_root, study_id_raw, series_id_raw, instance_number)
        if dcm_path is None:
            continue
        try:
            ds = pydicom.dcmread(str(dcm_path), stop_before_pixels=True)
        except Exception:
            continue
        required = ["Rows", "Columns", "PixelSpacing", "ImagePositionPatient", "ImageOrientationPatient"]
        if not all(hasattr(ds, tag) for tag in required):
            continue

        px_spacing = tuple(float(v) for v in ds.PixelSpacing)
        ipp = tuple(float(v) for v in ds.ImagePositionPatient)
        iop = tuple(float(v) for v in ds.ImageOrientationPatient)
        x_col = float(row["x"])
        y_row = float(row["y"])

        if not check_orientation_valid(iop):
            qc["iop_valid"] = False
            continue
        if not check_xy_inside_frame(x_col, y_row, int(ds.Rows), int(ds.Columns)):
            qc["coords_inside_image"] = False
            continue

        xyz = dicom_pixel_to_patient_xyz(x_col, y_row, px_spacing, ipp, iop)
        if not np.all(np.isfinite(xyz)):
            qc["xyz_finite"] = False
            continue

        xyz_by_level[level] = xyz

    qc["five_levels_complete"] = len(xyz_by_level) == len(CANONICAL_LEVELS)
    qc["qc_pass"] = bool(qc["five_levels_complete"] and qc["iop_valid"] and qc["coords_inside_image"] and qc["xyz_finite"] and qc["levels_unique"])
    return {"xyz": xyz_by_level, "qc": qc}


print("resolve_sagittal_gt_disc_xyz_for_study defined (development tool only -- outer TRAIN scope, caller-enforced).")

resolve_sagittal_gt_disc_xyz_for_study defined (development tool only -- outer TRAIN scope, caller-enforced).


## 3 -- Geometria axial por slice (cada slice usa su propia geometria, nunca SliceLocation/InstanceNumber)

No se trata una serie axial como un unico plano: cada slice lee su propio `ImagePositionPatient` /
`ImageOrientationPatient` / `PixelSpacing`. Anomalias registradas explicitamente: metadata faltante,
IOP invalido, PixelSpacing invalido, dimensiones invalidas, `InstanceNumber` duplicado.

In [ ]:
import tempfile


def _build_synthetic_dicom_dataset(pixel_spacing, image_position_patient, image_orientation_patient, rows=64, columns=64) -> "pydicom.Dataset":
    from pydicom.dataset import Dataset, FileMetaDataset
    from pydicom.uid import ExplicitVRLittleEndian, generate_uid

    file_meta = FileMetaDataset()
    file_meta.MediaStorageSOPClassUID = generate_uid()
    file_meta.MediaStorageSOPInstanceUID = generate_uid()
    file_meta.TransferSyntaxUID = ExplicitVRLittleEndian

    ds = Dataset()
    ds.file_meta = file_meta
    ds.is_little_endian = True
    ds.is_implicit_VR = False
    ds.Rows = rows
    ds.Columns = columns
    ds.PixelSpacing = [str(round(float(pixel_spacing[0]), 6)), str(round(float(pixel_spacing[1]), 6))]
    ds.ImagePositionPatient = [str(round(float(v), 6)) for v in image_position_patient]
    ds.ImageOrientationPatient = [str(round(float(v), 6)) for v in image_orientation_patient]
    ds.SOPInstanceUID = file_meta.MediaStorageSOPInstanceUID
    ds.SOPClassUID = file_meta.MediaStorageSOPClassUID
    ds.PixelData = np.zeros((rows, columns), dtype=np.uint16).tobytes()
    ds.BitsAllocated = 16
    ds.BitsStored = 16
    ds.HighBit = 15
    ds.PixelRepresentation = 0
    ds.SamplesPerPixel = 1
    ds.PhotometricInterpretation = "MONOCHROME2"
    return ds


AXIAL_ANOMALY_TYPES = (
    "MISSING_METADATA", "INVALID_IOP", "INVALID_PIXEL_SPACING", "INVALID_DIMENSIONS", "DUPLICATE_INSTANCE_NUMBER",
)


def read_axial_slice_geometry(dcm_path: Path) -> dict:
    # Reads ONE axial DICOM's own geometry -- never reused across other slices in the same series.
    try:
        ds = pydicom.dcmread(str(dcm_path), stop_before_pixels=True)
    except Exception as exc:
        return {"status": "READ_ERROR", "anomaly": "MISSING_METADATA", "detail": f"{type(exc).__name__}: {exc}"}

    required = ["Rows", "Columns", "PixelSpacing", "ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]
    if not all(hasattr(ds, tag) for tag in required):
        return {"status": "INVALID", "anomaly": "MISSING_METADATA", "detail": "one or more required DICOM tags absent"}

    try:
        rows, columns = int(ds.Rows), int(ds.Columns)
        pixel_spacing = tuple(float(v) for v in ds.PixelSpacing)
        ipp = tuple(float(v) for v in ds.ImagePositionPatient)
        iop = tuple(float(v) for v in ds.ImageOrientationPatient)
        instance_number = int(ds.InstanceNumber)
    except (TypeError, ValueError) as exc:
        return {"status": "INVALID", "anomaly": "MISSING_METADATA", "detail": f"{type(exc).__name__}: {exc}"}

    if rows <= 0 or columns <= 0:
        return {"status": "INVALID", "anomaly": "INVALID_DIMENSIONS", "detail": f"rows={rows} columns={columns}"}
    if len(pixel_spacing) != 2 or not check_geometry_finite(*pixel_spacing) or any(v <= 0 for v in pixel_spacing):
        return {"status": "INVALID", "anomaly": "INVALID_PIXEL_SPACING", "detail": str(pixel_spacing)}
    if not check_orientation_valid(iop) or not check_geometry_finite(*iop, *ipp):
        return {"status": "INVALID", "anomaly": "INVALID_IOP", "detail": str(iop)}

    row_cosines = np.array(iop[0:3], dtype=np.float64)
    column_cosines = np.array(iop[3:6], dtype=np.float64)
    normal = np.cross(row_cosines, column_cosines)
    normal_norm = float(np.linalg.norm(normal))
    if normal_norm <= 1e-9 or not np.all(np.isfinite(normal)):
        return {"status": "INVALID", "anomaly": "INVALID_IOP", "detail": "degenerate cross product (row/col cosines nearly parallel)"}
    normal = normal / normal_norm

    return {
        "status": "OK", "anomaly": None,
        "instance_number": instance_number, "rows": rows, "columns": columns,
        "pixel_spacing": pixel_spacing, "ipp": np.array(ipp, dtype=np.float64), "iop": iop,
        "row_cosines": row_cosines, "column_cosines": column_cosines, "normal": normal,
    }


def audit_axial_series_geometry(dcm_paths: list) -> dict:
    slices = []
    anomalies = []
    instance_numbers_seen = {}

    for p in dcm_paths:
        geom = read_axial_slice_geometry(p)
        if geom["status"] != "OK":
            anomalies.append({"anomaly_type": geom["anomaly"], "detail": geom.get("detail")})
            continue
        instance_numbers_seen.setdefault(geom["instance_number"], 0)
        instance_numbers_seen[geom["instance_number"]] += 1
        slices.append(geom)

    for instance_number, count in instance_numbers_seen.items():
        if count > 1:
            anomalies.append({"anomaly_type": "DUPLICATE_INSTANCE_NUMBER", "detail": f"instance_number={instance_number} appears {count} times"})

    return {"slices": slices, "anomalies": anomalies, "n_dicom_paths": len(dcm_paths), "n_valid_slices": len(slices)}


# Synthetic self-tests (independent of RSNA availability):
_syn_dir = Path(tempfile.mkdtemp())
_syn_axial_paths = []
for _i, _z in enumerate([0.0, 5.0, 5.0, 10.0]):  # deliberate duplicate InstanceNumber via z=5.0 twice below
    _ds = _build_synthetic_dicom_dataset(pixel_spacing=(0.8, 0.8), image_position_patient=(0.0, 0.0, _z), image_orientation_patient=(1.0, 0.0, 0.0, 0.0, 1.0, 0.0))
    _ds.InstanceNumber = 2 if _i == 2 else _i + 1  # instance_number=2 duplicated (index 1 and index 2)
    _path = _syn_dir / f"axial_{_i}.dcm"
    _ds.save_as(str(_path), enforce_file_format=True)
    _syn_axial_paths.append(_path)

_syn_audit = audit_axial_series_geometry(_syn_axial_paths)
assert _syn_audit["n_valid_slices"] == 4, "audit_axial_series_geometry self-test FAILED (expected 4 valid synthetic slices)"
assert any(a["anomaly_type"] == "DUPLICATE_INSTANCE_NUMBER" for a in _syn_audit["anomalies"]), "audit_axial_series_geometry self-test FAILED (expected a DUPLICATE_INSTANCE_NUMBER anomaly)"
assert np.allclose(_syn_audit["slices"][0]["normal"], [0.0, 0.0, 1.0]) or np.allclose(_syn_audit["slices"][0]["normal"], [0.0, 0.0, -1.0]), "audit_axial_series_geometry self-test FAILED (expected axis-aligned normal for identity IOP)"
for _p in _syn_axial_paths:
    _p.unlink()
_syn_dir.rmdir()
print("read_axial_slice_geometry / audit_axial_series_geometry: synthetic self-tests PASSED (valid slices, duplicate InstanceNumber anomaly detected).")

read_axial_slice_geometry / audit_axial_series_geometry: synthetic self-tests PASSED (valid slices, duplicate InstanceNumber anomaly detected).


## 4 -- Punto -> plano (signed distance, proyeccion, FOV)

`signed_distance_mm = dot(P - O, n)`; `plane_distance_mm = abs(signed_distance_mm)`;
`P_projected = P - signed_distance_mm * n`, convertido a `row/col` DICOM de ESE slice
especificamente (nunca se reutiliza la geometria de otro slice).

In [ ]:
def project_point_to_axial_slice(P: np.ndarray, slice_geom: dict) -> dict:
    O = slice_geom["ipp"]
    n = slice_geom["normal"]
    row_spacing, column_spacing = slice_geom["pixel_spacing"]
    row_cosines = slice_geom["row_cosines"]
    column_cosines = slice_geom["column_cosines"]

    signed_distance_mm = float(np.dot(P - O, n))
    plane_distance_mm = abs(signed_distance_mm)
    P_projected = P - signed_distance_mm * n

    diff = P_projected - O
    projected_col_px = float(np.dot(diff, row_cosines)) / column_spacing
    projected_row_px = float(np.dot(diff, column_cosines)) / row_spacing

    inside_fov = check_xy_inside_frame(projected_col_px, projected_row_px, slice_geom["rows"], slice_geom["columns"])
    distance_to_border_px = min(
        projected_col_px, slice_geom["columns"] - 1 - projected_col_px,
        projected_row_px, slice_geom["rows"] - 1 - projected_row_px,
    )

    return {
        "signed_distance_mm": signed_distance_mm, "plane_distance_mm": plane_distance_mm,
        "projected_row_px": projected_row_px, "projected_col_px": projected_col_px,
        "inside_fov": bool(inside_fov), "distance_to_border_px": float(distance_to_border_px),
    }


# --- Synthetic self-tests (Section 4 audit -- all 6 required cases) ---
_id_slice = {
    "ipp": np.array([0.0, 0.0, 0.0]), "normal": np.array([0.0, 0.0, 1.0]),
    "pixel_spacing": (1.0, 1.0), "row_cosines": np.array([1.0, 0.0, 0.0]), "column_cosines": np.array([0.0, 1.0, 0.0]),
    "rows": 64, "columns": 64,
}

# 1) point exactly on the plane -> zero distance.
_r1 = project_point_to_axial_slice(np.array([10.0, 20.0, 0.0]), _id_slice)
assert abs(_r1["signed_distance_mm"]) < 1e-9 and abs(_r1["plane_distance_mm"]) < 1e-9, "project_point_to_axial_slice self-test FAILED (case 1: on-plane point should have zero distance)"
assert abs(_r1["projected_col_px"] - 10.0) < 1e-9 and abs(_r1["projected_row_px"] - 20.0) < 1e-9, "project_point_to_axial_slice self-test FAILED (case 1: projected row/col should equal the in-plane offset)"

# 2) known distance -> point offset 7.5mm along the normal.
_r2 = project_point_to_axial_slice(np.array([10.0, 20.0, 7.5]), _id_slice)
assert abs(_r2["plane_distance_mm"] - 7.5) < 1e-9, "project_point_to_axial_slice self-test FAILED (case 2: expected plane_distance_mm=7.5)"
assert abs(_r2["projected_col_px"] - 10.0) < 1e-9 and abs(_r2["projected_row_px"] - 20.0) < 1e-9, "project_point_to_axial_slice self-test FAILED (case 2: projection should ignore the normal-direction offset)"

# 3) projection inside FOV.
assert project_point_to_axial_slice(np.array([30.0, 30.0, 0.0]), _id_slice)["inside_fov"] is True, "project_point_to_axial_slice self-test FAILED (case 3: expected inside_fov=True)"

# 4) projection outside FOV.
assert project_point_to_axial_slice(np.array([100.0, 100.0, 0.0]), _id_slice)["inside_fov"] is False, "project_point_to_axial_slice self-test FAILED (case 4: expected inside_fov=False)"

# 5) inverted normal produces the SAME absolute distance (sign flips, magnitude does not).
_flipped_slice = dict(_id_slice, normal=-_id_slice["normal"])
_r5a = project_point_to_axial_slice(np.array([10.0, 20.0, 7.5]), _id_slice)
_r5b = project_point_to_axial_slice(np.array([10.0, 20.0, 7.5]), _flipped_slice)
assert abs(_r5a["plane_distance_mm"] - _r5b["plane_distance_mm"]) < 1e-9, "project_point_to_axial_slice self-test FAILED (case 5: inverted normal should preserve plane_distance_mm)"
assert _r5a["signed_distance_mm"] * _r5b["signed_distance_mm"] < 0, "project_point_to_axial_slice self-test FAILED (case 5: inverted normal should flip the sign of signed_distance_mm)"

# 6) different IOP per slice handled independently (a tilted slice must not reuse the identity slice's geometry).
_tilted_slice = {
    "ipp": np.array([5.0, -3.0, 2.0]), "normal": np.array([0.0, -0.342, 0.940]) / np.linalg.norm([0.0, -0.342, 0.940]),
    "pixel_spacing": (0.7, 0.9), "row_cosines": np.array([1.0, 0.0, 0.0]), "column_cosines": np.array([0.0, 0.9396926, 0.3420201]),
    "rows": 128, "columns": 128,
}
_r6_identity = project_point_to_axial_slice(np.array([10.0, 20.0, 0.0]), _id_slice)
_r6_tilted = project_point_to_axial_slice(np.array([10.0, 20.0, 0.0]), _tilted_slice)
assert _r6_identity["plane_distance_mm"] != _r6_tilted["plane_distance_mm"] or _r6_identity["projected_col_px"] != _r6_tilted["projected_col_px"], "project_point_to_axial_slice self-test FAILED (case 6: identity and tilted slices must be evaluated independently, not share geometry)"

print("project_point_to_axial_slice: synthetic self-tests PASSED (on-plane, known-distance, inside/outside FOV, normal-sign invariance, independent per-slice IOP).")

project_point_to_axial_slice: synthetic self-tests PASSED (on-plane, known-distance, inside/outside FOV, normal-sign invariance, independent per-slice IOP).


## 5 -- Candidatos por nivel (study x level x TODOS los slices de TODAS las series axiales)

Regla base (sin threshold arbitrario): priorizar `inside_fov=True`, luego minima
`plane_distance_mm`. Solo opaque IDs en las filas persistidas.

In [ ]:
def build_pairing_candidates_for_study(study_id_raw: str, gt_xyz_by_level: dict, train_images_root: Path, geometry_log: list = None) -> list:
    study_opaque = opaque_id(study_id_raw)
    study_axial_series = series_df_train[
        (series_df_train["study_id"] == study_id_raw)
        & (series_df_train["series_description"].isin(axial_series_descriptions_observed))
    ]

    candidates = []
    for _, srow in study_axial_series.iterrows():
        series_id_raw = srow["series_id"]
        series_description = srow["series_description"]
        series_dir = train_images_root / str(study_id_raw) / str(series_id_raw)
        if not series_dir.is_dir():
            continue
        series_opaque = opaque_id(series_id_raw)

        for dcm_path in sorted(series_dir.glob("*.dcm")):
            geom = read_axial_slice_geometry(dcm_path)
            if geom["status"] != "OK":
                continue

            if geometry_log is not None:
                geometry_log.append({
                    "study_id_opaque": study_opaque, "series_id_opaque": series_opaque, "series_description": series_description,
                    "instance_number": geom["instance_number"], "rows": geom["rows"], "columns": geom["columns"],
                    "pixel_spacing_row_mm": geom["pixel_spacing"][0], "pixel_spacing_col_mm": geom["pixel_spacing"][1],
                    "normal_x": float(geom["normal"][0]), "normal_y": float(geom["normal"][1]), "normal_z": float(geom["normal"][2]),
                })

            for level, P in gt_xyz_by_level.items():
                proj = project_point_to_axial_slice(P, geom)
                candidates.append({
                    "study_id_opaque": study_opaque, "level": level,
                    "series_id_opaque": series_opaque, "series_description": series_description,
                    "instance_number": geom["instance_number"],
                    "plane_distance_mm": proj["plane_distance_mm"], "signed_plane_distance_mm": proj["signed_distance_mm"],
                    "inside_fov": proj["inside_fov"],
                    "projected_row_px": proj["projected_row_px"], "projected_col_px": proj["projected_col_px"],
                    "rows": geom["rows"], "columns": geom["columns"],
                })
    return candidates


def select_baseline_a(candidates_for_level: list) -> dict:
    # Base rule (Section 5 audit): prioritize inside_fov=True, then minimum plane_distance_mm.
    # No arbitrary threshold. Returns None if no candidates exist for this level.
    if not candidates_for_level:
        return None
    ranked = sorted(candidates_for_level, key=lambda c: (not c["inside_fov"], c["plane_distance_mm"]))
    return ranked[0]


# Synthetic self-test: inside_fov=True must always outrank inside_fov=False, regardless of distance.
_syn_candidates = [
    {"series_id_opaque": "a", "instance_number": 1, "plane_distance_mm": 50.0, "inside_fov": False},
    {"series_id_opaque": "b", "instance_number": 2, "plane_distance_mm": 2.0, "inside_fov": True},
    {"series_id_opaque": "c", "instance_number": 3, "plane_distance_mm": 1.0, "inside_fov": False},
]
_syn_best = select_baseline_a(_syn_candidates)
assert _syn_best["series_id_opaque"] == "b", "select_baseline_a self-test FAILED (inside_fov=True must outrank a smaller out-of-FOV distance)"
assert select_baseline_a([]) is None, "select_baseline_a self-test FAILED (empty candidate list should return None)"
print("build_pairing_candidates_for_study / select_baseline_a: defined and self-tested (inside_fov priority, then min plane_distance_mm; no threshold).")

build_pairing_candidates_for_study / select_baseline_a: defined and self-tested (inside_fov priority, then min plane_distance_mm; no threshold).


## 6 -- Diagnostico de bloques axiales (orientacion por serie -- diagnostico, no requisito del algoritmo)

Se mide primero si nearest-plane-distance + FOV ya alcanza; el clustering de orientacion es
diagnostico secundario, **no** se asume necesario para el algoritmo final.

In [ ]:
def audit_axial_series_blocks(slices: list) -> dict:
    if len(slices) < 2:
        return {"n_slices": len(slices), "angle_between_consecutive_deg": [], "orientation_clusters": 1 if slices else 0, "instance_number_range": None}

    ordered = sorted(slices, key=lambda s: s["instance_number"])
    angles = []
    for i in range(len(ordered) - 1):
        n1, n2 = ordered[i]["normal"], ordered[i + 1]["normal"]
        cos_angle = float(np.clip(np.dot(n1, n2), -1.0, 1.0))
        angles.append(float(np.degrees(np.arccos(abs(cos_angle)))))  # abs(): a flipped normal is the same physical plane orientation

    # Approximate orientation clustering: a new cluster starts whenever the angle to the previous
    # slice's normal exceeds a small numerical tolerance (NOT a clinical/anatomical threshold --
    # just distinguishing "same block" from "genuinely different angulation").
    ANGLE_CLUSTER_TOLERANCE_DEG = 1.0
    n_clusters = 1 + sum(1 for a in angles if a > ANGLE_CLUSTER_TOLERANCE_DEG)

    instance_numbers = [s["instance_number"] for s in ordered]
    return {
        "n_slices": len(ordered),
        "angle_between_consecutive_deg": angles,
        "orientation_clusters": n_clusters,
        "instance_number_range": [min(instance_numbers), max(instance_numbers)],
    }


_syn_same_block = [{"instance_number": i, "normal": np.array([0.0, 0.0, 1.0])} for i in range(5)]
_syn_diag_same = audit_axial_series_blocks(_syn_same_block)
assert _syn_diag_same["orientation_clusters"] == 1, "audit_axial_series_blocks self-test FAILED (identical normals should form a single cluster)"

_syn_two_blocks = [{"instance_number": i, "normal": np.array([0.0, 0.0, 1.0])} for i in range(3)] + [{"instance_number": i, "normal": np.array([0.0, 0.5735764, 0.8191520])} for i in range(3, 6)]  # ~35deg tilt
_syn_diag_two = audit_axial_series_blocks(_syn_two_blocks)
assert _syn_diag_two["orientation_clusters"] == 2, "audit_axial_series_blocks self-test FAILED (expected 2 orientation clusters for a tilted second block)"
print("audit_axial_series_blocks: synthetic self-tests PASSED (single-block and two-block detection). Diagnostic only -- not used by baseline_A/baseline_B selection.")

audit_axial_series_blocks: synthetic self-tests PASSED (single-block and two-block detection). Diagnostic only -- not used by baseline_A/baseline_B selection.


## 7 -- GT axial POST-HOC (nunca usado para producir la prediccion)

Metrica primaria: **EXACT LEVEL ASSIGNMENT ACCURACY** -- el slice/bloque elegido, ¿pertenece a la
serie axial correcta para ese nivel? No se exige `InstanceNumber` exacto (multiples slices pueden
representar correctamente el mismo nivel). Metricas secundarias (solo dentro de la misma serie):
exact series hit, exact slice hit, +/-1, +/-2, diferencia absoluta de `InstanceNumber`.

In [ ]:
def resolve_axial_gt_reference_for_study(study_id_raw: str) -> dict:
    rows = axial_reference_rows_train[axial_reference_rows_train["study_id"] == study_id_raw]
    gt_by_level = {}
    for _, r in rows.iterrows():
        level = r.get("level_normalized")
        if level not in CANONICAL_LEVELS:
            continue
        instance_number = int(r["instance_number"]) if pd.notna(r.get("instance_number")) else None
        gt_by_level.setdefault(level, []).append({
            "series_id_opaque": opaque_id(r["series_id"]), "instance_number": instance_number, "condition": r.get("condition"),
        })
    return gt_by_level


def evaluate_prediction_against_gt(predicted_candidate, gt_rows_for_level: list) -> dict:
    # POST-HOC ONLY -- called strictly after baseline_A/baseline_B already picked predicted_candidate.
    if predicted_candidate is None or not gt_rows_for_level:
        return {"exact_level_assignment": None, "exact_series_hit": None, "exact_slice_hit": None, "instance_diff_abs": None, "within_1": None, "within_2": None}

    predicted_series = predicted_candidate["series_id_opaque"]
    matching_gt_in_series = [g for g in gt_rows_for_level if g["series_id_opaque"] == predicted_series]

    if not matching_gt_in_series:
        return {"exact_level_assignment": False, "exact_series_hit": False, "exact_slice_hit": None, "instance_diff_abs": None, "within_1": None, "within_2": None}

    diffs = [abs(predicted_candidate["instance_number"] - g["instance_number"]) for g in matching_gt_in_series if g["instance_number"] is not None]
    if not diffs:
        return {"exact_level_assignment": True, "exact_series_hit": True, "exact_slice_hit": None, "instance_diff_abs": None, "within_1": None, "within_2": None}

    min_diff = min(diffs)
    return {
        "exact_level_assignment": True, "exact_series_hit": True,
        "exact_slice_hit": bool(min_diff == 0), "instance_diff_abs": int(min_diff),
        "within_1": bool(min_diff <= 1), "within_2": bool(min_diff <= 2),
    }


# Synthetic self-tests (GT plays a POST-HOC role only -- never chooses the candidate):
_gt_syn = [{"series_id_opaque": "s1", "instance_number": 10, "condition": "X"}]
_pred_match = {"series_id_opaque": "s1", "instance_number": 11}
_eval_match = evaluate_prediction_against_gt(_pred_match, _gt_syn)
assert _eval_match["exact_level_assignment"] is True and _eval_match["within_1"] is True and _eval_match["exact_slice_hit"] is False, "evaluate_prediction_against_gt self-test FAILED (expected series match, +/-1 hit, not exact-slice)"

_pred_wrong_series = {"series_id_opaque": "s2", "instance_number": 10}
_eval_wrong = evaluate_prediction_against_gt(_pred_wrong_series, _gt_syn)
assert _eval_wrong["exact_level_assignment"] is False, "evaluate_prediction_against_gt self-test FAILED (expected exact_level_assignment=False for wrong series)"

assert evaluate_prediction_against_gt(None, _gt_syn)["exact_level_assignment"] is None, "evaluate_prediction_against_gt self-test FAILED (no prediction should yield None, never False)"
print("resolve_axial_gt_reference_for_study / evaluate_prediction_against_gt: defined and self-tested (post-hoc only, GT never selects the candidate).")

resolve_axial_gt_reference_for_study / evaluate_prediction_against_gt: defined and self-tested (post-hoc only, GT never selects the candidate).


## Ejecucion end-to-end sobre 67C_train / 67C_dev (BASELINE_A) -- gated por RSNA_AVAILABLE

Corre `resolve_sagittal_gt_disc_xyz_for_study` (dev tool) -> `build_pairing_candidates_for_study`
-> `select_baseline_a` -> `resolve_axial_gt_reference_for_study` (POST-HOC) ->
`evaluate_prediction_against_gt` sobre cada study de `67C_train` y `67C_dev`. Ningun paso de
seleccion mira GT axial. `MAX_STUDIES_67C_B=None` procesa todos los studies elegibles.

In [ ]:
# ============================================================
# 67C-B — FAST DICOM GEOMETRY I/O OVERRIDE
#
# EXECUTION-ONLY OPTIMIZATION.
#
# Scientific method is UNCHANGED:
# - same read_axial_slice_geometry()
# - same build_pairing_candidates_for_study()
# - same point -> plane projection
# - same FOV rule
# - same GT post-hoc evaluation
# - same TRAIN / DEV split
# - same metrics
#
# What changes:
# 1. Reads axial DICOM metadata concurrently.
# 2. Reads each axial DICOM geometry only ONCE per runtime.
# 3. Keeps geometry in RAM for reuse.
# 4. BASELINE_B reuses DEV geometry instead of reopening DICOMs.
# 5. Memoizes sagittal GT and axial GT resolver results.
#
# Run AFTER all 67C-B helper/function-definition cells
# and BEFORE the heavy execution cell that starts with:
#
#     MAX_STUDIES_67C_B = None
#
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import threading
import time
import os
import copy


# ============================================================
# 0 — SAFETY CONTRACT
# ============================================================

assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False
assert ACCESS_OUTER_VALIDATION_AXIAL_GT is False
assert RUN_LOCKED_VALIDATION is False
assert TRAINING_PERFORMED is False
assert AXIAL_PAIRING_VALIDATED_67C is False


_required_fast_symbols = [
    "read_axial_slice_geometry",
    "build_pairing_candidates_for_study",
    "resolve_sagittal_gt_disc_xyz_for_study",
    "resolve_axial_gt_reference_for_study",
    "series_df_train",
    "axial_series_descriptions_observed",
    "internal_67c_split",
    "opaque_id",
]

_missing_fast_symbols = [
    x
    for x in _required_fast_symbols
    if x not in globals()
]

assert not _missing_fast_symbols, (
    "Run the previous 67C-B definition cells first. "
    f"Missing: {_missing_fast_symbols}"
)


FAST_TRAIN_IMAGES_ROOT = (
    RSNA_ROOT
    / "train_images"
)

assert FAST_TRAIN_IMAGES_ROOT.is_dir(), (
    f"Missing train_images: {FAST_TRAIN_IMAGES_ROOT}"
)


print(
    "=============================================="
)
print(
    "67C-B FAST I/O OVERRIDE"
)
print(
    "=============================================="
)


# ============================================================
# 1 — Preserve ORIGINAL scientific functions exactly once
# ============================================================

if (
    "_67C_FAST_ORIGINALS_CAPTURED"
    not in globals()
):

    _67C_FAST_ORIGINALS_CAPTURED = True

    _67C_ORIGINAL_READ_AXIAL_GEOMETRY = (
        read_axial_slice_geometry
    )

    _67C_ORIGINAL_BUILD_CANDIDATES = (
        build_pairing_candidates_for_study
    )

    _67C_ORIGINAL_RESOLVE_SAGITTAL_GT = (
        resolve_sagittal_gt_disc_xyz_for_study
    )

    _67C_ORIGINAL_RESOLVE_AXIAL_GT = (
        resolve_axial_gt_reference_for_study
    )

    print(
        "Original scientific functions captured: PASS"
    )

else:

    print(
        "Original scientific functions already preserved."
    )


# ============================================================
# 2 — Persistent in-memory caches
# ============================================================

# High-RAM runtime is ideal here.
#
# Geometry objects are small compared with pixel arrays:
# we only cache metadata / vectors, never image pixels.

if "_67C_GEOMETRY_CACHE" not in globals():
    _67C_GEOMETRY_CACHE = {}

if "_67C_SAGITTAL_GT_CACHE" not in globals():
    _67C_SAGITTAL_GT_CACHE = {}

if "_67C_AXIAL_GT_CACHE" not in globals():
    _67C_AXIAL_GT_CACHE = {}


_67C_GEOMETRY_CACHE_LOCK = threading.Lock()


# ============================================================
# 3 — I/O worker pool
# ============================================================

# Google Drive FUSE usually benefits strongly from concurrent
# small-file metadata reads.
#
# 16 is conservative enough not to flood Drive.
# This does NOT affect scientific output.

GEOMETRY_IO_WORKERS = 16


# If this optimization cell is re-run, replace only the
# executor; keep already-read geometry cache.

if "_67C_GEOMETRY_EXECUTOR" in globals():

    try:
        _67C_GEOMETRY_EXECUTOR.shutdown(
            wait=False,
            cancel_futures=True,
        )
    except Exception:
        pass


_67C_GEOMETRY_EXECUTOR = ThreadPoolExecutor(
    max_workers=GEOMETRY_IO_WORKERS
)


print(
    "Geometry I/O workers:",
    GEOMETRY_IO_WORKERS,
)


# ============================================================
# 4 — Stable in-memory path key
# ============================================================

def _67c_geometry_path_key(
    dcm_path,
):

    # Never persisted.
    # Raw path remains runtime-only.
    return str(
        Path(dcm_path)
    )


# ============================================================
# 5 — Cached drop-in replacement for DICOM geometry reader
# ============================================================

def _67c_cached_read_axial_slice_geometry(
    dcm_path,
):

    key = _67c_geometry_path_key(
        dcm_path
    )

    cached = _67C_GEOMETRY_CACHE.get(
        key
    )

    if cached is not None:
        return cached


    # Fallback for any path that was not prewarmed.
    #
    # IMPORTANT:
    # calls the ORIGINAL scientific reader.

    result = (
        _67C_ORIGINAL_READ_AXIAL_GEOMETRY(
            Path(dcm_path)
        )
    )


    with _67C_GEOMETRY_CACHE_LOCK:

        _67C_GEOMETRY_CACHE[
            key
        ] = result


    return result


# ============================================================
# 6 — Resolve axial DICOM paths for ONE study
# ============================================================

def _67c_collect_axial_paths_for_study(
    study_id_raw,
    train_images_root,
):

    sid = str(
        study_id_raw
    )


    study_axial_series = (
        series_df_train[
            (
                series_df_train[
                    "study_id"
                ]
                .astype(str)
                == sid
            )
            &
            (
                series_df_train[
                    "series_description"
                ]
                .isin(
                    axial_series_descriptions_observed
                )
            )
        ]
    )


    paths = []


    for _, srow in study_axial_series.iterrows():

        series_id_raw = str(
            srow[
                "series_id"
            ]
        )

        series_dir = (
            Path(
                train_images_root
            )
            / sid
            / series_id_raw
        )


        if not series_dir.is_dir():
            continue


        paths.extend(
            sorted(
                series_dir.glob(
                    "*.dcm"
                )
            )
        )


    # deterministic de-duplication
    unique = {}

    for p in paths:
        unique[
            str(p)
        ] = p


    return [
        unique[k]
        for k
        in sorted(
            unique
        )
    ]


# ============================================================
# 7 — Parallel prewarm for ONE study
# ============================================================

_67C_FAST_STATS = {
    "studies_prewarmed": 0,
    "dicoms_requested": 0,
    "dicoms_read_from_drive": 0,
    "dicoms_cache_hits": 0,
    "prewarm_seconds": 0.0,
}


def _67c_prewarm_study_geometry(
    study_id_raw,
    train_images_root,
):

    t0 = time.time()


    paths = (
        _67c_collect_axial_paths_for_study(
            study_id_raw,
            train_images_root,
        )
    )


    missing = []

    cache_hits = 0


    for path in paths:

        key = (
            _67c_geometry_path_key(
                path
            )
        )


        if (
            key
            in _67C_GEOMETRY_CACHE
        ):

            cache_hits += 1

        else:

            missing.append(
                path
            )


    # --------------------------------------------------------
    # Read every missing DICOM geometry concurrently.
    #
    # Each worker calls the ORIGINAL reader.
    # --------------------------------------------------------

    futures = {

        _67C_GEOMETRY_EXECUTOR.submit(
            _67C_ORIGINAL_READ_AXIAL_GEOMETRY,
            path,
        ): path

        for path in missing
    }


    new_results = {}


    for future in as_completed(
        futures
    ):

        path = futures[
            future
        ]

        # Do NOT fabricate graceful values here.
        # The original function already implements its
        # audited error semantics.

        result = future.result()

        new_results[
            _67c_geometry_path_key(
                path
            )
        ] = result


    if new_results:

        with _67C_GEOMETRY_CACHE_LOCK:

            _67C_GEOMETRY_CACHE.update(
                new_results
            )


    elapsed = (
        time.time()
        - t0
    )


    _67C_FAST_STATS[
        "studies_prewarmed"
    ] += 1

    _67C_FAST_STATS[
        "dicoms_requested"
    ] += len(
        paths
    )

    _67C_FAST_STATS[
        "dicoms_read_from_drive"
    ] += len(
        missing
    )

    _67C_FAST_STATS[
        "dicoms_cache_hits"
    ] += cache_hits

    _67C_FAST_STATS[
        "prewarm_seconds"
    ] += elapsed


    return {
        "n_paths": len(
            paths
        ),
        "n_new_reads": len(
            missing
        ),
        "n_cache_hits": cache_hits,
        "elapsed_s": elapsed,
    }


# ============================================================
# 8 — Fast wrapper around ORIGINAL candidate builder
# ============================================================

_67C_FAST_BUILD_COUNT = 0
_67C_FAST_BUILD_T0 = time.time()


def _67c_fast_build_pairing_candidates_for_study(
    study_id_raw,
    gt_xyz_by_level,
    train_images_root,
    geometry_log=None,
):

    global _67C_FAST_BUILD_COUNT


    # First load ALL axial geometries for the study
    # concurrently.

    _67c_prewarm_study_geometry(
        study_id_raw,
        train_images_root,
    )


    # Then call EXACTLY the original candidate builder.
    #
    # Its global read_axial_slice_geometry now points to
    # the cache-backed reader, therefore its scientific
    # implementation is unchanged but no Drive read occurs.

    output = (
        _67C_ORIGINAL_BUILD_CANDIDATES(
            study_id_raw,
            gt_xyz_by_level,
            train_images_root,
            geometry_log=geometry_log,
        )
    )


    _67C_FAST_BUILD_COUNT += 1


    if (
        _67C_FAST_BUILD_COUNT % 100
        == 0
    ):

        elapsed = (
            time.time()
            - _67C_FAST_BUILD_T0
        )

        stats = (
            _67C_FAST_STATS
        )


        print(
            "[FAST I/O]"
            f" studies={_67C_FAST_BUILD_COUNT}"
            f" | geometry cache={len(_67C_GEOMETRY_CACHE)} DICOMs"
            f" | Drive reads={stats['dicoms_read_from_drive']}"
            f" | cache hits={stats['dicoms_cache_hits']}"
            f" | elapsed={elapsed/60:.1f} min"
        )


    return output


# ============================================================
# 9 — GT resolver memoization
#
# Same exact resolver; only avoids doing the same work twice.
# ============================================================

def _67c_fast_resolve_sagittal_gt(
    study_id_raw,
):

    key = opaque_id(
        study_id_raw
    )


    if (
        key
        not in _67C_SAGITTAL_GT_CACHE
    ):

        _67C_SAGITTAL_GT_CACHE[
            key
        ] = (
            _67C_ORIGINAL_RESOLVE_SAGITTAL_GT(
                study_id_raw
            )
        )


    return (
        _67C_SAGITTAL_GT_CACHE[
            key
        ]
    )


def _67c_fast_resolve_axial_gt(
    study_id_raw,
):

    key = opaque_id(
        study_id_raw
    )


    if (
        key
        not in _67C_AXIAL_GT_CACHE
    ):

        _67C_AXIAL_GT_CACHE[
            key
        ] = (
            _67C_ORIGINAL_RESOLVE_AXIAL_GT(
                study_id_raw
            )
        )


    return (
        _67C_AXIAL_GT_CACHE[
            key
        ]
    )


# ============================================================
# 10 — Install execution-only overrides
# ============================================================

read_axial_slice_geometry = (
    _67c_cached_read_axial_slice_geometry
)

build_pairing_candidates_for_study = (
    _67c_fast_build_pairing_candidates_for_study
)

resolve_sagittal_gt_disc_xyz_for_study = (
    _67c_fast_resolve_sagittal_gt
)

resolve_axial_gt_reference_for_study = (
    _67c_fast_resolve_axial_gt
)


# ============================================================
# 11 — Tiny hard parity check
#
# Read a few DICOMs using ORIGINAL reader and compare against
# cached path.
#
# This does NOT access validation.
# ============================================================

_parity_study = (
    internal_67c_split[
        "67C_train"
    ][0]
)


_parity_paths = (
    _67c_collect_axial_paths_for_study(
        _parity_study,
        FAST_TRAIN_IMAGES_ROOT,
    )
)


assert len(
    _parity_paths
) > 0, (
    "No axial DICOMs found in parity TRAIN study."
)


_parity_paths = (
    _parity_paths[:3]
)


for _path in _parity_paths:

    serial = (
        _67C_ORIGINAL_READ_AXIAL_GEOMETRY(
            _path
        )
    )

    cached = (
        _67c_cached_read_axial_slice_geometry(
            _path
        )
    )


    assert (
        serial.get(
            "status"
        )
        ==
        cached.get(
            "status"
        )
    )


    if (
        serial.get(
            "status"
        )
        == "OK"
    ):

        assert (
            serial[
                "instance_number"
            ]
            ==
            cached[
                "instance_number"
            ]
        )

        assert (
            serial[
                "rows"
            ]
            ==
            cached[
                "rows"
            ]
        )

        assert (
            serial[
                "columns"
            ]
            ==
            cached[
                "columns"
            ]
        )

        assert np.allclose(
            serial[
                "pixel_spacing"
            ],
            cached[
                "pixel_spacing"
            ],
        )

        assert np.allclose(
            serial[
                "ipp"
            ],
            cached[
                "ipp"
            ],
        )

        assert np.allclose(
            serial[
                "iop"
            ],
            cached[
                "iop"
            ],
        )

        assert np.allclose(
            serial[
                "normal"
            ],
            cached[
                "normal"
            ],
        )


print(
    "Geometry cache parity:",
    "PASS",
)


# ============================================================
# 12 — Warm-up benchmark
#
# Preloads a few TRAIN studies now.
# They remain cached when the main cell begins.
# ============================================================

WARMUP_STUDIES = 8


_warmup_ids = (
    internal_67c_split[
        "67C_train"
    ][:WARMUP_STUDIES]
)


_warmup_t0 = time.time()

_warmup_dicom_count = 0


for _sid in _warmup_ids:

    info = (
        _67c_prewarm_study_geometry(
            _sid,
            FAST_TRAIN_IMAGES_ROOT,
        )
    )

    _warmup_dicom_count += (
        info[
            "n_paths"
        ]
    )


_warmup_elapsed = (
    time.time()
    - _warmup_t0
)


print()
print(
    "=============================================="
)

print(
    "67C-B FAST I/O READY"
)

print(
    "=============================================="
)

print(
    "Workers:",
    GEOMETRY_IO_WORKERS,
)

print(
    "Warmup studies:",
    WARMUP_STUDIES,
)

print(
    "Warmup axial DICOMs:",
    _warmup_dicom_count,
)

print(
    "Warmup time:",
    round(
        _warmup_elapsed,
        2,
    ),
    "sec",
)

print(
    "Geometry cache entries:",
    len(
        _67C_GEOMETRY_CACHE
    ),
)

print()

print(
    "Candidate builder scientific code:"
    " ORIGINAL / UNCHANGED"
)

print(
    "Axial geometry reader:"
    " ORIGINAL / PARALLEL-CACHED"
)

print(
    "Outer validation axial GT accessed: False"
)

print(
    "Internal test accessed: False"
)

print(
    "Official test accessed: False"
)

print()

print(
    "NOW RERUN ONLY THE HEAVY 67C-B EXECUTION CELL."
)

67C-B FAST I/O OVERRIDE
Original scientific functions captured: PASS
Geometry I/O workers: 16
Geometry cache parity: PASS

67C-B FAST I/O READY
Workers: 16
Warmup studies: 8
Warmup axial DICOMs: 268
Warmup time: 1.36 sec
Geometry cache entries: 268

Candidate builder scientific code: ORIGINAL / UNCHANGED
Axial geometry reader: ORIGINAL / PARALLEL-CACHED
Outer validation axial GT accessed: False
Internal test accessed: False
Official test accessed: False

NOW RERUN ONLY THE HEAVY 67C-B EXECUTION CELL.


In [ ]:
# ============================================================
# 67C-B FAST I/O — FINAL GT SIGNATURE FIX
# ============================================================

# Sagittal GT original:
#   (study_id_raw, train_images_root)
#
# Axial GT original:
#   (study_id_raw)
#
# Scientific behavior unchanged.
# Existing geometry cache preserved.


def _67c_fast_resolve_sagittal_gt(
    study_id_raw,
    train_images_root,
):
    key = opaque_id(study_id_raw)

    if key not in _67C_SAGITTAL_GT_CACHE:
        _67C_SAGITTAL_GT_CACHE[key] = (
            _67C_ORIGINAL_RESOLVE_SAGITTAL_GT(
                study_id_raw,
                train_images_root,
            )
        )

    return _67C_SAGITTAL_GT_CACHE[key]


def _67c_fast_resolve_axial_gt(
    study_id_raw,
):
    key = opaque_id(study_id_raw)

    if key not in _67C_AXIAL_GT_CACHE:
        _67C_AXIAL_GT_CACHE[key] = (
            _67C_ORIGINAL_RESOLVE_AXIAL_GT(
                study_id_raw
            )
        )

    return _67C_AXIAL_GT_CACHE[key]


# Install corrected wrappers.
resolve_sagittal_gt_disc_xyz_for_study = (
    _67c_fast_resolve_sagittal_gt
)

resolve_axial_gt_reference_for_study = (
    _67c_fast_resolve_axial_gt
)


# ============================================================
# HARD SMOKE TEST — ONE TRAIN STUDY
# ============================================================

_test_sid = internal_67c_split["67C_train"][0]

_test_root = RSNA_ROOT / "train_images"


_test_sag = (
    resolve_sagittal_gt_disc_xyz_for_study(
        _test_sid,
        _test_root,
    )
)

_test_ax = (
    resolve_axial_gt_reference_for_study(
        _test_sid
    )
)


assert isinstance(_test_sag, dict)
assert isinstance(_test_ax, dict)


print()
print("==============================================")
print("67C-B FINAL GT SIGNATURE FIX: PASS")
print("==============================================")

print("Sagittal GT signature: PASS")
print("Axial GT signature: PASS")

print(
    "Geometry cache preserved:",
    len(_67C_GEOMETRY_CACHE),
    "DICOMs",
)

print(
    "Sagittal GT cache:",
    len(_67C_SAGITTAL_GT_CACHE),
    "studies",
)

print(
    "Axial GT cache:",
    len(_67C_AXIAL_GT_CACHE),
    "studies",
)

print()

print("Outer validation axial GT accessed: False")
print("Internal test accessed: False")
print("Official test accessed: False")

print()
print("NOW RERUN THE HEAVY 67C-B CELL.")


67C-B FINAL GT SIGNATURE FIX: PASS
Sagittal GT signature: PASS
Axial GT signature: PASS
Geometry cache preserved: 268 DICOMs
Sagittal GT cache: 2 studies
Axial GT cache: 1 studies

Outer validation axial GT accessed: False
Internal test accessed: False
Official test accessed: False

NOW RERUN THE HEAVY 67C-B CELL.


In [ ]:
MAX_STUDIES_67C_B = None  # None => process all eligible studies in each cohort.

pairing_prediction_rows = []       # opaque only -- one row per (study, level)
pairing_candidate_summary_rows = []  # opaque only -- candidate-count/ambiguity diagnostics per (study, level)
study_skip_reasons = []            # opaque only -- studies excluded before candidate generation
axial_slice_geometry_rows = []     # opaque only -- one row per unique axial DICOM slice touched
sagittal_gt_disc_xyz_rows = []     # opaque only -- one row per (study, level) dev-tool GT XYZ (never a runtime feature)
all_pairing_candidates_rows = []   # opaque only -- full candidate table (study x level x axial slice), cache for 67C-C

RSNA_AVAILABLE_67C_B = (RSNA_ROOT / "train_images").is_dir() if GATE_A_RSNA_dataset_structure == "PASS" else False
print("RSNA_AVAILABLE_67C_B:", RSNA_AVAILABLE_67C_B)

if RSNA_AVAILABLE_67C_B:
    train_images_root = RSNA_ROOT / "train_images"
    for cohort_name in ("67C_train", "67C_dev"):
        cohort_study_ids = internal_67c_split[cohort_name]
        if MAX_STUDIES_67C_B is not None:
            cohort_study_ids = cohort_study_ids[:MAX_STUDIES_67C_B]

        _t0 = time.time()
        for _i, study_id_raw in enumerate(cohort_study_ids):
            study_opaque = opaque_id(study_id_raw)

            gt_dev = resolve_sagittal_gt_disc_xyz_for_study(study_id_raw, train_images_root)
            if not gt_dev["qc"]["qc_pass"]:
                study_skip_reasons.append({"study_id_opaque": study_opaque, "cohort": cohort_name, "reason": "SAGITTAL_GT_QC_FAILED", "qc": gt_dev["qc"]})
                continue

            gt_xyz_by_level = gt_dev["xyz"]
            for _lv, _xyz in gt_xyz_by_level.items():
                sagittal_gt_disc_xyz_rows.append({
                    "study_id_opaque": study_opaque, "cohort": cohort_name, "level": _lv,
                    "x_mm": float(_xyz[0]), "y_mm": float(_xyz[1]), "z_mm": float(_xyz[2]),
                })
            candidates_all_levels = build_pairing_candidates_for_study(study_id_raw, gt_xyz_by_level, train_images_root, geometry_log=axial_slice_geometry_rows)
            for _c in candidates_all_levels:
                all_pairing_candidates_rows.append({**_c, "cohort": cohort_name})

            if not candidates_all_levels:
                study_skip_reasons.append({"study_id_opaque": study_opaque, "cohort": cohort_name, "reason": "NO_AXIAL_CANDIDATES", "qc": gt_dev["qc"]})
                continue

            axial_gt_by_level = resolve_axial_gt_reference_for_study(study_id_raw)

            # series_id_opaque -> set of levels for which this series is a GT axial series (used
            # only to build the diagnostic 5x5 confusion matrix -- never fed back into selection).
            series_to_gt_levels: dict = {}
            for _lv, _gt_rows in axial_gt_by_level.items():
                for _g in _gt_rows:
                    series_to_gt_levels.setdefault(_g["series_id_opaque"], set()).add(_lv)

            for level in CANONICAL_LEVELS:
                level_candidates = [c for c in candidates_all_levels if c["level"] == level]
                predicted = select_baseline_a(level_candidates)
                gt_rows_for_level = axial_gt_by_level.get(level, [])
                eval_result = evaluate_prediction_against_gt(predicted, gt_rows_for_level)

                ranked = sorted(level_candidates, key=lambda c: (not c["inside_fov"], c["plane_distance_mm"]))
                distance_1 = ranked[0]["plane_distance_mm"] if len(ranked) >= 1 else None
                distance_2 = ranked[1]["plane_distance_mm"] if len(ranked) >= 2 else None
                margin_mm = (distance_2 - distance_1) if (distance_1 is not None and distance_2 is not None) else None
                ratio = (distance_1 / max(distance_2, 1e-6)) if (distance_1 is not None and distance_2 is not None) else None

                confusion_true_level = None
                if predicted is not None:
                    _matched_levels = series_to_gt_levels.get(predicted["series_id_opaque"])
                    if not _matched_levels:
                        confusion_true_level = "NONE"
                    elif len(_matched_levels) == 1:
                        confusion_true_level = next(iter(_matched_levels))
                    else:
                        confusion_true_level = "MULTIPLE"

                pairing_prediction_rows.append({
                    "study_id_opaque": study_opaque, "cohort": cohort_name, "level": level,
                    "n_candidates": len(level_candidates),
                    "n_axial_series_for_study": len(set(c["series_id_opaque"] for c in candidates_all_levels)),
                    "selected_series_id_opaque": predicted["series_id_opaque"] if predicted else None,
                    "selected_instance_number": predicted["instance_number"] if predicted else None,
                    "selected_plane_distance_mm": predicted["plane_distance_mm"] if predicted else None,
                    "selected_inside_fov": predicted["inside_fov"] if predicted else None,
                    "distance_margin_mm": margin_mm, "distance_ratio": ratio,
                    "has_axial_gt_for_level": len(gt_rows_for_level) > 0,
                    "confusion_true_level": confusion_true_level,
                    **eval_result,
                })

                pairing_candidate_summary_rows.append({
                    "study_id_opaque": study_opaque, "cohort": cohort_name, "level": level,
                    "n_candidates": len(level_candidates), "distance_1_mm": distance_1, "distance_2_mm": distance_2,
                    "margin_mm": margin_mm,
                })

            if (_i + 1) % 100 == 0 or (_i + 1) == len(cohort_study_ids):
                print(f"[{cohort_name}] {_i + 1}/{len(cohort_study_ids)} studies | elapsed={time.time() - _t0:.1f}s")
else:
    warnings.append("67C-B end-to-end pairing NOT_RUN: RSNA train_images/ unavailable, or GATE_A did not PASS in this run.")
    print("RSNA_AVAILABLE_67C_B is False -- pairing execution NOT_RUN (graceful degradation, no fabricated evidence).")

RSNA_AVAILABLE_67C_B: True
[67C_train] 100/1106 studies | elapsed=18.0s
[FAST I/O] studies=100 | geometry cache=4301 DICOMs | Drive reads=4298 | cache hits=271 | elapsed=3.1 min
[67C_train] 200/1106 studies | elapsed=126.3s
[FAST I/O] studies=200 | geometry cache=8227 DICOMs | Drive reads=8224 | cache hits=271 | elapsed=5.1 min
[67C_train] 300/1106 studies | elapsed=281.5s
[FAST I/O] studies=300 | geometry cache=12431 DICOMs | Drive reads=12428 | cache hits=271 | elapsed=7.7 min
[67C_train] 400/1106 studies | elapsed=461.2s
[FAST I/O] studies=400 | geometry cache=16831 DICOMs | Drive reads=16828 | cache hits=271 | elapsed=10.9 min
[67C_train] 500/1106 studies | elapsed=606.1s
[FAST I/O] studies=500 | geometry cache=20653 DICOMs | Drive reads=20650 | cache hits=271 | elapsed=13.3 min
[67C_train] 600/1106 studies | elapsed=751.5s
[FAST I/O] studies=600 | geometry cache=24978 DICOMs | Drive reads=24975 | cache hits=271 | elapsed=16.0 min
[67C_train] 700/1106 studies | elapsed=904.6s
[FAST

## 8 -- Metricas (TRAIN / DEV por separado) -- nunca fabricadas si no hay evidencia

Metrica primaria: **exact level assignment accuracy**. El resto son diagnosticos. Si no hay filas
evaluables, cada metrica reporta `None` explicitamente -- nunca un valor inventado.

In [ ]:
def _percentile_or_none(values: list, q: float):
    clean = [v for v in values if v is not None and np.isfinite(v)]
    return float(np.percentile(clean, q)) if clean else None


def compute_67c_b_metrics(rows: list, cohort_name: str, all_cohort_study_ids: list) -> dict:
    cohort_rows = [r for r in rows if r["cohort"] == cohort_name]
    evaluable_rows = [r for r in cohort_rows if r["exact_level_assignment"] is not None]

    studies_total = len(all_cohort_study_ids)
    studies_evaluable = len(set(r["study_id_opaque"] for r in cohort_rows))
    level_rows_evaluable = len(evaluable_rows)
    coverage = (len(cohort_rows) / (studies_total * len(CANONICAL_LEVELS))) if studies_total else None

    exact_level_accuracy = (sum(1 for r in evaluable_rows if r["exact_level_assignment"]) / level_rows_evaluable) if level_rows_evaluable else None

    accuracy_by_level = {}
    for level in CANONICAL_LEVELS:
        level_rows = [r for r in evaluable_rows if r["level"] == level]
        accuracy_by_level[level] = (sum(1 for r in level_rows if r["exact_level_assignment"]) / len(level_rows)) if level_rows else None

    confusion_matrix = {intended: {actual: 0 for actual in CANONICAL_LEVELS + ["NONE", "MULTIPLE"]} for intended in CANONICAL_LEVELS}
    for r in cohort_rows:
        if r.get("confusion_true_level") is not None:
            confusion_matrix[r["level"]][r["confusion_true_level"]] = confusion_matrix[r["level"]].get(r["confusion_true_level"], 0) + 1

    plane_distances = [r["selected_plane_distance_mm"] for r in cohort_rows if r["selected_plane_distance_mm"] is not None]
    median_plane_distance_mm = _percentile_or_none(plane_distances, 50)
    p90_plane_distance_mm = _percentile_or_none(plane_distances, 90)
    p95_plane_distance_mm = _percentile_or_none(plane_distances, 95)

    fov_rejection_count = sum(1 for r in cohort_rows if r["selected_inside_fov"] is False)
    no_candidate_count = sum(1 for r in cohort_rows if r["n_candidates"] == 0)
    metadata_failure_count = sum(1 for r in study_skip_reasons if r["cohort"] == cohort_name)

    accuracy_by_n_series = {}
    for n_series in sorted(set(r["n_axial_series_for_study"] for r in evaluable_rows)):
        subset = [r for r in evaluable_rows if r["n_axial_series_for_study"] == n_series]
        accuracy_by_n_series[str(n_series)] = sum(1 for r in subset if r["exact_level_assignment"]) / len(subset)

    accuracy_by_n_candidates_bucket = {}
    _buckets = [(0, 0), (1, 5), (6, 15), (16, 10**9)]
    for lo, hi in _buckets:
        subset = [r for r in evaluable_rows if lo <= r["n_candidates"] <= hi]
        label = f"{lo}-{hi if hi < 10**9 else 'inf'}"
        accuracy_by_n_candidates_bucket[label] = (sum(1 for r in subset if r["exact_level_assignment"]) / len(subset)) if subset else None

    correct_distances = [r["selected_plane_distance_mm"] for r in evaluable_rows if r["exact_level_assignment"] and r["selected_plane_distance_mm"] is not None]
    incorrect_distances = [r["selected_plane_distance_mm"] for r in evaluable_rows if not r["exact_level_assignment"] and r["selected_plane_distance_mm"] is not None]
    correct_margins = [r["distance_margin_mm"] for r in evaluable_rows if r["exact_level_assignment"] and r["distance_margin_mm"] is not None]
    incorrect_margins = [r["distance_margin_mm"] for r in evaluable_rows if not r["exact_level_assignment"] and r["distance_margin_mm"] is not None]

    within_1_rows = [r for r in evaluable_rows if r.get("within_1") is not None]
    within_2_rows = [r for r in evaluable_rows if r.get("within_2") is not None]
    exact_slice_rows = [r for r in evaluable_rows if r.get("exact_slice_hit") is not None]

    return {
        "cohort": cohort_name,
        "studies_total": studies_total,
        "studies_evaluable": studies_evaluable,
        "level_rows_evaluable": level_rows_evaluable,
        "coverage": coverage,
        "exact_level_assignment_accuracy": exact_level_accuracy,
        "accuracy_by_level": accuracy_by_level,
        "confusion_matrix_5x5": confusion_matrix,
        "median_plane_distance_mm": median_plane_distance_mm,
        "p90_plane_distance_mm": p90_plane_distance_mm,
        "p95_plane_distance_mm": p95_plane_distance_mm,
        "fov_rejection_count": fov_rejection_count,
        "no_candidate_count": no_candidate_count,
        "metadata_failure_count": metadata_failure_count,
        "accuracy_by_n_axial_series": accuracy_by_n_series,
        "accuracy_by_n_candidates_bucket": accuracy_by_n_candidates_bucket,
        "exact_series_hit_rate": (sum(1 for r in evaluable_rows if r["exact_series_hit"]) / level_rows_evaluable) if level_rows_evaluable else None,
        "exact_slice_hit_rate": (sum(1 for r in exact_slice_rows if r["exact_slice_hit"]) / len(exact_slice_rows)) if exact_slice_rows else None,
        "within_1_rate": (sum(1 for r in within_1_rows if r["within_1"]) / len(within_1_rows)) if within_1_rows else None,
        "within_2_rate": (sum(1 for r in within_2_rows if r["within_2"]) / len(within_2_rows)) if within_2_rows else None,
        "median_instance_diff_abs": _percentile_or_none([r["instance_diff_abs"] for r in evaluable_rows if r.get("instance_diff_abs") is not None], 50),
        "plane_distance_correct_vs_incorrect": {
            "correct_median": _percentile_or_none(correct_distances, 50), "incorrect_median": _percentile_or_none(incorrect_distances, 50),
        },
        "distance_margin_correct_vs_incorrect": {
            "correct_median": _percentile_or_none(correct_margins, 50), "incorrect_median": _percentile_or_none(incorrect_margins, 50),
        },
    }


pairing_67c_b_train_metrics = compute_67c_b_metrics(pairing_prediction_rows, "67C_train", internal_67c_split["67C_train"])
pairing_67c_b_dev_metrics = compute_67c_b_metrics(pairing_prediction_rows, "67C_dev", internal_67c_split["67C_dev"])

print("67C_B_train_metrics.exact_level_assignment_accuracy:", pairing_67c_b_train_metrics["exact_level_assignment_accuracy"])
print("67C_B_dev_metrics.exact_level_assignment_accuracy:", pairing_67c_b_dev_metrics["exact_level_assignment_accuracy"])

67C_B_train_metrics.exact_level_assignment_accuracy: 0.9921320284014584
67C_B_dev_metrics.exact_level_assignment_accuracy: 0.9907763259031515


## 9 -- Ambiguedad (solo DEV interno) -- curvas descriptivas, ningun threshold congelado todavia

`distance_margin_mm = distance_2 - distance_1`; `distance_ratio = distance_1 / max(distance_2, eps)`.
Curvas puramente descriptivas sobre `67C_dev` -- **nunca** se mira outer `validation`. El congelamiento
real del threshold/regla de abstencion ocurre en `67C-C`, no aqui.

In [ ]:
def compute_max_distance_curve(rows: list, thresholds: list) -> list:
    curve = []
    for t in thresholds:
        accepted = [r for r in rows if r["selected_inside_fov"] and r["selected_plane_distance_mm"] is not None and r["selected_plane_distance_mm"] <= t]
        coverage = len(accepted) / len(rows) if rows else None
        accuracy = (sum(1 for r in accepted if r["exact_level_assignment"]) / len(accepted)) if accepted else None
        curve.append({"max_plane_distance_mm": t, "coverage": coverage, "accuracy": accuracy, "n_accepted": len(accepted)})
    return curve


def compute_min_margin_curve(rows: list, thresholds: list) -> list:
    curve = []
    for t in thresholds:
        accepted = [r for r in rows if r["selected_inside_fov"] and r["distance_margin_mm"] is not None and r["distance_margin_mm"] >= t]
        coverage = len(accepted) / len(rows) if rows else None
        accuracy = (sum(1 for r in accepted if r["exact_level_assignment"]) / len(accepted)) if accepted else None
        curve.append({"min_margin_mm": t, "coverage": coverage, "accuracy": accuracy, "n_accepted": len(accepted)})
    return curve


def compute_combined_curve(rows: list, distance_thresholds: list, margin_thresholds: list) -> list:
    curve = []
    for dt in distance_thresholds:
        for mt in margin_thresholds:
            accepted = [
                r for r in rows
                if r["selected_inside_fov"]
                and r["selected_plane_distance_mm"] is not None and r["selected_plane_distance_mm"] <= dt
                and r["distance_margin_mm"] is not None and r["distance_margin_mm"] >= mt
            ]
            coverage = len(accepted) / len(rows) if rows else None
            accuracy = (sum(1 for r in accepted if r["exact_level_assignment"]) / len(accepted)) if accepted else None
            curve.append({"max_plane_distance_mm": dt, "min_margin_mm": mt, "coverage": coverage, "accuracy": accuracy, "n_accepted": len(accepted)})
    return curve


dev_evaluable_rows = [r for r in pairing_prediction_rows if r["cohort"] == "67C_dev" and r["exact_level_assignment"] is not None]

selective_curve_A, selective_curve_B, selective_curve_C = [], [], []
best_descriptive_operating_point = None

if dev_evaluable_rows:
    _dev_distances = sorted(r["selected_plane_distance_mm"] for r in dev_evaluable_rows if r["selected_plane_distance_mm"] is not None)
    _dev_margins = sorted(r["distance_margin_mm"] for r in dev_evaluable_rows if r["distance_margin_mm"] is not None)

    _distance_thresholds = sorted(set(_percentile_or_none(_dev_distances, q) for q in (10, 25, 50, 75, 90, 100)) - {None}) if _dev_distances else []
    _margin_thresholds = sorted(set(_percentile_or_none(_dev_margins, q) for q in (0, 10, 25, 50, 75, 90)) - {None}) if _dev_margins else []

    selective_curve_A = compute_max_distance_curve(dev_evaluable_rows, _distance_thresholds)
    selective_curve_B = compute_min_margin_curve(dev_evaluable_rows, _margin_thresholds)
    selective_curve_C = compute_combined_curve(dev_evaluable_rows, _distance_thresholds, _margin_thresholds)

    # Descriptive only: report the combined operating point with the highest accuracy among those
    # with coverage >= 0.5 -- this is NOT a frozen rule, just a descriptive summary for 67C-C to
    # consider later. Never applied back to selection, never evaluated on outer validation.
    _candidates_ge_50pct_coverage = [c for c in selective_curve_C if c["coverage"] is not None and c["coverage"] >= 0.5 and c["accuracy"] is not None]
    if _candidates_ge_50pct_coverage:
        best_descriptive_operating_point = max(_candidates_ge_50pct_coverage, key=lambda c: c["accuracy"])

    print("Descriptive curve A (max plane distance) points:", len(selective_curve_A))
    print("Descriptive curve B (min margin) points:", len(selective_curve_B))
    print("Descriptive curve C (combined) points:", len(selective_curve_C))
    print("Best descriptive DEV operating point (coverage>=0.5, informational only):", best_descriptive_operating_point)
else:
    warnings.append("Ambiguity/threshold descriptive curves NOT_RUN: no evaluable DEV rows in this run.")
    print("No evaluable DEV rows -- ambiguity descriptive curves NOT_RUN.")

Descriptive curve A (max plane distance) points: 6
Descriptive curve B (min margin) points: 6
Descriptive curve C (combined) points: 36
Best descriptive DEV operating point (coverage>=0.5, informational only): {'max_plane_distance_mm': 23.818094078633493, 'min_margin_mm': 1.994024515702412, 'coverage': 0.5003843197540354, 'accuracy': 1.0, 'n_accepted': 651}


## 10 -- Experimento DEV: asignacion conjunta (BASELINE_B) vs nearest independiente (BASELINE_A)

`BASELINE_A` (nearest plane distance + FOV, independiente por nivel) vs `BASELINE_B` (asignacion
global Hungarian 5-niveles <-> slices candidatos, cada slice usado a lo sumo una vez -- previene
reutilizar un slice incompatible entre niveles). **No se impone monotonicidad por `InstanceNumber`**
-- la unica restriccion es la no-reutilizacion geometrica que da la asignacion biparita en si misma.
Si `BASELINE_A` ya es suficientemente bueno, se prefiere por simplicidad (Section 10 audit).

In [ ]:
from scipy.optimize import linear_sum_assignment as _linear_sum_assignment

FOV_VIOLATION_PENALTY_MM = 10_000.0  # not a clinical/anatomical threshold -- a large constant so an
                                      # out-of-FOV candidate is only chosen when unavoidable.


def select_baseline_b(candidates_all_levels: list) -> dict:
    per_level_candidate_by_key = {lv: {} for lv in CANONICAL_LEVELS}
    for c in candidates_all_levels:
        key = (c["series_id_opaque"], c["instance_number"])
        per_level_candidate_by_key[c["level"]][key] = c

    slice_keys = sorted(set(k for lv in CANONICAL_LEVELS for k in per_level_candidate_by_key[lv].keys()))
    if not slice_keys:
        return {lv: None for lv in CANONICAL_LEVELS}

    cost = np.full((len(CANONICAL_LEVELS), len(slice_keys)), FOV_VIOLATION_PENALTY_MM * 10.0)
    for i, level in enumerate(CANONICAL_LEVELS):
        for j, key in enumerate(slice_keys):
            c = per_level_candidate_by_key[level].get(key)
            if c is not None:
                cost[i, j] = c["plane_distance_mm"] + (0.0 if c["inside_fov"] else FOV_VIOLATION_PENALTY_MM)

    row_ind, col_ind = _linear_sum_assignment(cost)
    result = {lv: None for lv in CANONICAL_LEVELS}
    for r, cidx in zip(row_ind, col_ind):
        level = CANONICAL_LEVELS[r]
        key = slice_keys[cidx]
        result[level] = per_level_candidate_by_key[level].get(key)
    return result


# Synthetic self-test: two adjacent levels both have the SAME nearest slice as their independent
# top choice -- BASELINE_A (independent) would double-book it; BASELINE_B must split them.
_syn_shared_slice = ("series_x", 10)
_syn_candidates_shared = [
    {"level": "L1-L2", "series_id_opaque": "series_x", "instance_number": 10, "plane_distance_mm": 1.0, "inside_fov": True},
    {"level": "L1-L2", "series_id_opaque": "series_y", "instance_number": 11, "plane_distance_mm": 3.0, "inside_fov": True},
    {"level": "L2-L3", "series_id_opaque": "series_x", "instance_number": 10, "plane_distance_mm": 1.5, "inside_fov": True},
    {"level": "L2-L3", "series_id_opaque": "series_z", "instance_number": 12, "plane_distance_mm": 4.0, "inside_fov": True},
]
_syn_b_result = select_baseline_b(_syn_candidates_shared)
_chosen_keys = [(v["series_id_opaque"], v["instance_number"]) for v in [_syn_b_result["L1-L2"], _syn_b_result["L2-L3"]] if v is not None]
assert len(set(_chosen_keys)) == len(_chosen_keys), "select_baseline_b self-test FAILED (two levels must not be assigned the same candidate slice)"
print("select_baseline_b: synthetic self-test PASSED (shared best-candidate slice is split across levels, never double-booked).")

baseline_comparison_rows = []
if RSNA_AVAILABLE_67C_B:
    for study_id_raw in internal_67c_split["67C_dev"]:
        study_opaque = opaque_id(study_id_raw)
        gt_dev = resolve_sagittal_gt_disc_xyz_for_study(study_id_raw, train_images_root)
        if not gt_dev["qc"]["qc_pass"]:
            continue
        candidates_all_levels = build_pairing_candidates_for_study(study_id_raw, gt_dev["xyz"], train_images_root)
        if not candidates_all_levels:
            continue

        axial_gt_by_level = resolve_axial_gt_reference_for_study(study_id_raw)
        baseline_b_result = select_baseline_b(candidates_all_levels)

        for level in CANONICAL_LEVELS:
            level_candidates = [c for c in candidates_all_levels if c["level"] == level]
            pred_a = select_baseline_a(level_candidates)
            pred_b = baseline_b_result[level]
            gt_rows_for_level = axial_gt_by_level.get(level, [])
            eval_a = evaluate_prediction_against_gt(pred_a, gt_rows_for_level)
            eval_b = evaluate_prediction_against_gt(pred_b, gt_rows_for_level)
            baseline_comparison_rows.append({
                "study_id_opaque": study_opaque, "level": level,
                "baseline_a_exact_level_assignment": eval_a["exact_level_assignment"],
                "baseline_b_exact_level_assignment": eval_b["exact_level_assignment"],
            })

    _comparable = [r for r in baseline_comparison_rows if r["baseline_a_exact_level_assignment"] is not None and r["baseline_b_exact_level_assignment"] is not None]
    baseline_a_dev_accuracy = (sum(1 for r in _comparable if r["baseline_a_exact_level_assignment"]) / len(_comparable)) if _comparable else None
    baseline_b_dev_accuracy = (sum(1 for r in _comparable if r["baseline_b_exact_level_assignment"]) / len(_comparable)) if _comparable else None
    print("BASELINE_A DEV exact-level accuracy (this comparison subset):", baseline_a_dev_accuracy)
    print("BASELINE_B DEV exact-level accuracy (this comparison subset):", baseline_b_dev_accuracy)
    if baseline_a_dev_accuracy is not None and baseline_b_dev_accuracy is not None:
        print("Preference (Section 10 audit): BASELINE_A preferred by simplicity unless BASELINE_B shows a clear improvement." if baseline_a_dev_accuracy >= baseline_b_dev_accuracy - 1e-9 else "BASELINE_B shows higher DEV accuracy in this comparison -- consider for 67C-C freeze discussion.")
else:
    baseline_a_dev_accuracy = None
    baseline_b_dev_accuracy = None
    warnings.append("BASELINE_A vs BASELINE_B DEV comparison NOT_RUN: RSNA unavailable in this run.")
    print("RSNA_AVAILABLE_67C_B is False -- BASELINE_A vs BASELINE_B comparison NOT_RUN.")

select_baseline_b: synthetic self-test PASSED (shared best-candidate slice is split across levels, never double-booked).
[FAST I/O] studies=1400 | geometry cache=54729 DICOMs | Drive reads=54726 | cache hits=3521 | elapsed=34.7 min
[FAST I/O] studies=1500 | geometry cache=54729 DICOMs | Drive reads=54726 | cache hits=7623 | elapsed=34.7 min
BASELINE_A DEV exact-level accuracy (this comparison subset): 0.9907763259031515
BASELINE_B DEV exact-level accuracy (this comparison subset): 0.9907763259031515
Preference (Section 10 audit): BASELINE_A preferred by simplicity unless BASELINE_B shows a clear improvement.


## Artefactos 67C-B (caches en `results/post_e50/67C/`, metricas en `metrics/post_e50/67C/`)

Solo opaque IDs. Nunca sobrescribe artefactos de 67B/67B1/67B2 (rutas distintas por diseno).

In [ ]:
def _rows_to_csv_text(rows: list) -> str:
    if not rows:
        return ""
    return pd.DataFrame(rows).to_csv(index=False)


safe_write_text(RESULTS_DIR / "axial_slice_geometry_outer_train.csv", _rows_to_csv_text(axial_slice_geometry_rows))
safe_write_text(RESULTS_DIR / "sagittal_gt_disc_xyz_outer_train.csv", _rows_to_csv_text(sagittal_gt_disc_xyz_rows))
safe_write_text(RESULTS_DIR / "pairing_candidates_outer_train.csv", _rows_to_csv_text(all_pairing_candidates_rows))
safe_write_text(RESULTS_DIR / "pairing_predictions_outer_train.csv", _rows_to_csv_text(pairing_prediction_rows))

safe_write_text(METRICS_DIR / "67C_internal_split_manifest.json", json.dumps(internal_67c_split_manifest, indent=2, default=str))
safe_write_text(METRICS_DIR / "67C_B_train_metrics.json", json.dumps(pairing_67c_b_train_metrics, indent=2, default=str))
safe_write_text(METRICS_DIR / "67C_B_dev_metrics.json", json.dumps(pairing_67c_b_dev_metrics, indent=2, default=str))
safe_write_text(METRICS_DIR / "67C_B_dev_selective_curve.csv", _rows_to_csv_text(selective_curve_C))

print("Artifacts written under:")
for _f in ("axial_slice_geometry_outer_train.csv", "sagittal_gt_disc_xyz_outer_train.csv", "pairing_candidates_outer_train.csv", "pairing_predictions_outer_train.csv"):
    print(" -", RESULTS_DIR / _f)
for _f in ("67C_internal_split_manifest.json", "67C_B_train_metrics.json", "67C_B_dev_metrics.json", "67C_B_dev_selective_curve.csv"):
    print(" -", METRICS_DIR / _f)

Artifacts written under:
 - /content/drive/MyDrive/PFI_MVP/results/post_e50/67C/axial_slice_geometry_outer_train.csv
 - /content/drive/MyDrive/PFI_MVP/results/post_e50/67C/sagittal_gt_disc_xyz_outer_train.csv
 - /content/drive/MyDrive/PFI_MVP/results/post_e50/67C/pairing_candidates_outer_train.csv
 - /content/drive/MyDrive/PFI_MVP/results/post_e50/67C/pairing_predictions_outer_train.csv
 - /content/drive/MyDrive/PFI_MVP/metrics/post_e50/67C/67C_internal_split_manifest.json
 - /content/drive/MyDrive/PFI_MVP/metrics/post_e50/67C/67C_B_train_metrics.json
 - /content/drive/MyDrive/PFI_MVP/metrics/post_e50/67C/67C_B_dev_metrics.json
 - /content/drive/MyDrive/PFI_MVP/metrics/post_e50/67C/67C_B_dev_selective_curve.csv


## 67C-B -- Cierre (imprime el bloque de finalizacion requerido)

In [ ]:
def _fmt(value, digits=4):
    return round(value, digits) if isinstance(value, (int, float)) else value


print("=" * 46)
print("67C-B GEOMETRY PAIRING DEVELOPMENT COMPLETE")
print("=" * 46)
print()
print("67C TRAIN studies:", len(internal_67c_split["67C_train"]))
print("67C DEV studies:", len(internal_67c_split["67C_dev"]))
print()
print("TRAIN:")
print("  evaluable levels:", pairing_67c_b_train_metrics["level_rows_evaluable"])
print("  coverage:", _fmt(pairing_67c_b_train_metrics["coverage"]))
print("  exact level accuracy:", _fmt(pairing_67c_b_train_metrics["exact_level_assignment_accuracy"]))
print()
print("DEV:")
print("  evaluable levels:", pairing_67c_b_dev_metrics["level_rows_evaluable"])
print("  coverage:", _fmt(pairing_67c_b_dev_metrics["coverage"]))
print("  exact level accuracy:", _fmt(pairing_67c_b_dev_metrics["exact_level_assignment_accuracy"]))
print()
print("DEV accuracy by level:")
for _level in CANONICAL_LEVELS:
    print(f"  {_level}:", _fmt(pairing_67c_b_dev_metrics["accuracy_by_level"].get(_level)))
print()
print("DEV plane distance:")
print("  median:", _fmt(pairing_67c_b_dev_metrics["median_plane_distance_mm"]))
print("  p95:", _fmt(pairing_67c_b_dev_metrics["p95_plane_distance_mm"]))
print()
print("DEV ambiguity:")
print("  median winner distance:", _fmt(pairing_67c_b_dev_metrics["plane_distance_correct_vs_incorrect"]["correct_median"]))
print("  median margin:", _fmt(pairing_67c_b_dev_metrics["distance_margin_correct_vs_incorrect"]["correct_median"]))
print()
print("Best descriptive DEV operating points:")
print(" ", best_descriptive_operating_point)
print()
INTERNAL_TEST_ACCESSED = False  # 67C-B never touches internal_test; RSNA_INTERNAL_TEST_LOCKED stays True throughout.
print("Outer validation axial GT accessed:", ACCESS_OUTER_VALIDATION_AXIAL_GT)
print("Internal test accessed:", INTERNAL_TEST_ACCESSED)
print("Official test accessed:", OFFICIAL_TEST_ACCESSED)
print()
print("AXIAL_PAIRING_VALIDATED_67C:", AXIAL_PAIRING_VALIDATED_67C)
print()
print("NEXT:")
print("67C-C -- freeze deterministic pairing + abstention policy")

67C-B GEOMETRY PAIRING DEVELOPMENT COMPLETE

67C TRAIN studies: 1106
67C DEV studies: 276

TRAIN:
  evaluable levels: 5211
  coverage: 0.9575
  exact level accuracy: 0.9921

DEV:
  evaluable levels: 1301
  coverage: 0.9529
  exact level accuracy: 0.9908

DEV accuracy by level:
  L1-L2: 0.996
  L2-L3: 0.9962
  L3-L4: 0.981
  L4-L5: 0.9848
  L5-S1: 0.9962

DEV plane distance:
  median: 1.0662
  p95: 2.2882

DEV ambiguity:
  median winner distance: 1.0599
  median margin: 2.0058

Best descriptive DEV operating points:
  {'max_plane_distance_mm': 23.818094078633493, 'min_margin_mm': 1.994024515702412, 'coverage': 0.5003843197540354, 'accuracy': 1.0, 'n_accepted': 651}

Outer validation axial GT accessed: False
Internal test accessed: False
Official test accessed: False

AXIAL_PAIRING_VALIDATED_67C: False

NEXT:
67C-C -- freeze deterministic pairing + abstention policy


In [ ]:
# ============================================================
# 67C-C — PRE-FREEZE DEV OPERATING-POINT AUDIT
#
# Purpose:
# choose the selective operating point using INTERNAL DEV only.
#
# DOES NOT:
# - access outer validation
# - access internal_test
# - access official test
# - retrain anything
# - freeze anything yet
# ============================================================

import numpy as np
import pandas as pd
import json


# ============================================================
# 0 — SAFETY
# ============================================================

assert ACCESS_OUTER_VALIDATION_AXIAL_GT is False
assert RUN_LOCKED_VALIDATION is False
assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False
assert AXIAL_PAIRING_VALIDATED_67C is False

assert baseline_a_dev_accuracy is not None
assert baseline_b_dev_accuracy is not None

# Baseline B must not be preferred unless it actually improves.
assert baseline_a_dev_accuracy >= baseline_b_dev_accuracy - 1e-12


# ============================================================
# 1 — Freeze architecture choice, NOT thresholds
# ============================================================

PAIRING_METHOD_SELECTED = "BASELINE_A_NEAREST_PLANE_PLUS_FOV"

print(
    "Pairing method selected for 67C-C:",
    PAIRING_METHOD_SELECTED,
)

print(
    "BASELINE_A DEV:",
    baseline_a_dev_accuracy,
)

print(
    "BASELINE_B DEV:",
    baseline_b_dev_accuracy,
)


# ============================================================
# 2 — DEV evaluable rows
# ============================================================

dev_rows = pd.DataFrame([
    r
    for r in pairing_prediction_rows
    if (
        r["cohort"] == "67C_dev"
        and
        r["exact_level_assignment"] is not None
    )
]).copy()


assert len(dev_rows) == 1301

dev_rows["correct"] = (
    dev_rows[
        "exact_level_assignment"
    ].astype(bool)
)


raw_correct = int(
    dev_rows["correct"].sum()
)

raw_errors = int(
    (~dev_rows["correct"]).sum()
)

raw_accuracy = float(
    dev_rows["correct"].mean()
)


print()
print("DEV evaluable levels:", len(dev_rows))
print("DEV correct:", raw_correct)
print("DEV errors:", raw_errors)
print("DEV raw accuracy:", raw_accuracy)


# ============================================================
# 3 — Diagnostic error distributions
# ============================================================

for correctness in [True, False]:

    subset = dev_rows[
        dev_rows["correct"] == correctness
    ]

    label = (
        "CORRECT"
        if correctness
        else "INCORRECT"
    )

    print()
    print(label)
    print("  n:", len(subset))

    if len(subset):

        print(
            "  distance median:",
            float(
                subset[
                    "selected_plane_distance_mm"
                ].median()
            )
        )

        print(
            "  distance p95:",
            float(
                subset[
                    "selected_plane_distance_mm"
                ].quantile(0.95)
            )
        )

        print(
            "  margin median:",
            float(
                subset[
                    "distance_margin_mm"
                ].median()
            )
        )

        print(
            "  margin p05:",
            float(
                subset[
                    "distance_margin_mm"
                ].quantile(0.05)
            )
        )

        print(
            "  inside FOV:",
            int(
                subset[
                    "selected_inside_fov"
                ].fillna(False).sum()
            ),
            "/",
            len(subset),
        )


# ============================================================
# 4 — Fine DEV-only selective grid
#
# We deliberately inspect HIGH-COVERAGE regimes.
# We are NOT maximizing accuracy blindly because that produced
# the undesirable ~50% coverage / 100% accuracy point.
# ============================================================

valid_distance = (
    dev_rows[
        "selected_plane_distance_mm"
    ]
    .dropna()
    .astype(float)
)

valid_margin = (
    dev_rows[
        "distance_margin_mm"
    ]
    .dropna()
    .astype(float)
)


# Dense but inexpensive quantile grid.
distance_quantiles = np.linspace(
    0.50,
    1.00,
    51,
)

margin_quantiles = np.linspace(
    0.00,
    0.50,
    51,
)


distance_thresholds = sorted(
    set(
        float(
            valid_distance.quantile(q)
        )
        for q
        in distance_quantiles
    )
)

margin_thresholds = sorted(
    set(
        float(
            valid_margin.quantile(q)
        )
        for q
        in margin_quantiles
    )
)


grid_rows = []


for max_distance in distance_thresholds:

    for min_margin in margin_thresholds:

        accepted_mask = (
            dev_rows[
                "selected_inside_fov"
            ].fillna(False).astype(bool)
            &
            dev_rows[
                "selected_plane_distance_mm"
            ].notna()
            &
            (
                dev_rows[
                    "selected_plane_distance_mm"
                ]
                <= max_distance
            )
            &
            dev_rows[
                "distance_margin_mm"
            ].notna()
            &
            (
                dev_rows[
                    "distance_margin_mm"
                ]
                >= min_margin
            )
        )

        accepted = dev_rows[
            accepted_mask
        ]

        n_accepted = int(
            len(accepted)
        )

        coverage = (
            n_accepted
            / len(dev_rows)
        )

        accuracy = (
            float(
                accepted[
                    "correct"
                ].mean()
            )
            if n_accepted
            else None
        )

        errors = (
            int(
                (
                    ~accepted[
                        "correct"
                    ]
                ).sum()
            )
            if n_accepted
            else 0
        )

        grid_rows.append({
            "max_plane_distance_mm":
                max_distance,

            "min_margin_mm":
                min_margin,

            "coverage":
                coverage,

            "accuracy":
                accuracy,

            "accepted":
                n_accepted,

            "errors":
                errors,
        })


grid_df = pd.DataFrame(
    grid_rows
)


# ============================================================
# 5 — Best accuracy while preserving requested coverage
# ============================================================

coverage_targets = [
    0.95,
    0.90,
    0.85,
    0.80,
    0.75,
]


print()
print(
    "=============================================="
)

print(
    "HIGH-COVERAGE DEV OPERATING POINTS"
)

print(
    "=============================================="
)


best_by_coverage = {}


for target in coverage_targets:

    eligible = grid_df[
        (
            grid_df[
                "coverage"
            ]
            >= target
        )
        &
        grid_df[
            "accuracy"
        ].notna()
    ].copy()


    if eligible.empty:

        best_by_coverage[
            str(target)
        ] = None

        print(
            f"coverage >= {target:.0%}: NONE"
        )

        continue


    # Primary: highest accuracy.
    # Secondary: highest coverage.
    # Tertiary: simplest / looser thresholds.

    eligible = eligible.sort_values(
        [
            "accuracy",
            "coverage",
            "max_plane_distance_mm",
            "min_margin_mm",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )


    best = eligible.iloc[0]


    result = {
        "coverage_target":
            target,

        "max_plane_distance_mm":
            float(
                best[
                    "max_plane_distance_mm"
                ]
            ),

        "min_margin_mm":
            float(
                best[
                    "min_margin_mm"
                ]
            ),

        "coverage":
            float(
                best[
                    "coverage"
                ]
            ),

        "accuracy":
            float(
                best[
                    "accuracy"
                ]
            ),

        "accepted":
            int(
                best[
                    "accepted"
                ]
            ),

        "errors":
            int(
                best[
                    "errors"
                ]
            ),
    }


    best_by_coverage[
        str(target)
    ] = result


    print()

    print(
        f"coverage >= {target:.0%}"
    )

    print(
        "  max distance:",
        result[
            "max_plane_distance_mm"
        ]
    )

    print(
        "  min margin:",
        result[
            "min_margin_mm"
        ]
    )

    print(
        "  actual coverage:",
        result[
            "coverage"
        ]
    )

    print(
        "  accuracy:",
        result[
            "accuracy"
        ]
    )

    print(
        "  accepted:",
        result[
            "accepted"
        ]
    )

    print(
        "  errors:",
        result[
            "errors"
        ]
    )


# ============================================================
# 6 — Highest coverage achievable at target precision
# ============================================================

accuracy_targets = [
    0.990,
    0.995,
    0.9975,
    1.000,
]


print()
print(
    "=============================================="
)

print(
    "HIGH-PRECISION DEV OPERATING POINTS"
)

print(
    "=============================================="
)


best_by_accuracy = {}


for target in accuracy_targets:

    eligible = grid_df[
        (
            grid_df[
                "accuracy"
            ]
            >= target
        )
    ].copy()


    if eligible.empty:

        best_by_accuracy[
            str(target)
        ] = None

        print(
            f"accuracy >= {target:.2%}: NONE"
        )

        continue


    eligible = eligible.sort_values(
        [
            "coverage",
            "accuracy",
        ],
        ascending=[
            False,
            False,
        ],
    )


    best = eligible.iloc[0]


    result = {
        "accuracy_target":
            target,

        "max_plane_distance_mm":
            float(
                best[
                    "max_plane_distance_mm"
                ]
            ),

        "min_margin_mm":
            float(
                best[
                    "min_margin_mm"
                ]
            ),

        "coverage":
            float(
                best[
                    "coverage"
                ]
            ),

        "accuracy":
            float(
                best[
                    "accuracy"
                ]
            ),

        "accepted":
            int(
                best[
                    "accepted"
                ]
            ),

        "errors":
            int(
                best[
                    "errors"
                ]
            ),
    }


    best_by_accuracy[
        str(target)
    ] = result


    print()

    print(
        f"accuracy >= {target:.2%}"
    )

    print(
        "  max distance:",
        result[
            "max_plane_distance_mm"
        ]
    )

    print(
        "  min margin:",
        result[
            "min_margin_mm"
        ]
    )

    print(
        "  coverage:",
        result[
            "coverage"
        ]
    )

    print(
        "  actual accuracy:",
        result[
            "accuracy"
        ]
    )

    print(
        "  accepted:",
        result[
            "accepted"
        ]
    )

    print(
        "  errors:",
        result[
            "errors"
        ]
    )


# ============================================================
# 7 — Save PRE-FREEZE diagnostic only
# ============================================================

PRE_FREEZE_GRID_PATH = (
    METRICS_DIR
    / "67C_C_pre_freeze_dev_grid.csv"
)

PRE_FREEZE_SUMMARY_PATH = (
    METRICS_DIR
    / "67C_C_pre_freeze_summary.json"
)


safe_write_text(
    PRE_FREEZE_GRID_PATH,
    grid_df.to_csv(
        index=False
    ),
)


pre_freeze_summary = {

    "stage":
        "67C-C_PRE_FREEZE_AUDIT",

    "pairing_method_selected":
        PAIRING_METHOD_SELECTED,

    "baseline_a_dev_accuracy":
        baseline_a_dev_accuracy,

    "baseline_b_dev_accuracy":
        baseline_b_dev_accuracy,

    "dev_evaluable_levels":
        int(
            len(dev_rows)
        ),

    "raw_accuracy":
        raw_accuracy,

    "raw_errors":
        raw_errors,

    "best_by_minimum_coverage":
        best_by_coverage,

    "best_by_minimum_accuracy":
        best_by_accuracy,

    "outer_validation_axial_gt_accessed":
        False,

    "internal_test_accessed":
        False,

    "official_test_accessed":
        False,

    "rule_frozen":
        False,

    "AXIAL_PAIRING_VALIDATED_67C":
        False,
}


safe_write_text(
    PRE_FREEZE_SUMMARY_PATH,
    json.dumps(
        pre_freeze_summary,
        indent=2,
    ),
)


print()
print(
    "=============================================="
)

print(
    "67C-C PRE-FREEZE AUDIT COMPLETE"
)

print(
    "=============================================="
)

print(
    "Pairing method:",
    PAIRING_METHOD_SELECTED,
)

print(
    "Raw DEV:",
    f"{raw_correct}/{len(dev_rows)}",
    "=",
    raw_accuracy,
)

print()

print(
    "NO RULE FROZEN YET"
)

print(
    "Outer validation axial GT accessed: False"
)

print(
    "Internal test accessed: False"
)

print(
    "Official test accessed: False"
)

print(
    "AXIAL_PAIRING_VALIDATED_67C: False"
)

Pairing method selected for 67C-C: BASELINE_A_NEAREST_PLANE_PLUS_FOV
BASELINE_A DEV: 0.9907763259031515
BASELINE_B DEV: 0.9907763259031515

DEV evaluable levels: 1301
DEV correct: 1289
DEV errors: 12
DEV raw accuracy: 0.9907763259031515

CORRECT
  n: 1289
  distance median: 1.0598899299829156
  distance p95: 2.2115100774883953
  margin median: 2.0057703718907156
  margin p05: 0.12869869093380165
  inside FOV: 1289 / 1289

INCORRECT
  n: 12
  distance median: 0.4724652536652725
  distance p95: 1.8223437182280757
  margin median: 0.09490776272666346
  margin p05: 0.010795264229038157
  inside FOV: 12 / 12

HIGH-COVERAGE DEV OPERATING POINTS

coverage >= 95%
  max distance: 23.818094078633493
  min margin: 0.1172779103904702
  actual coverage: 0.9500384319754035
  accuracy: 0.9959546925566343
  accepted: 1236
  errors: 5

coverage >= 90%
  max distance: 23.818094078633493
  min margin: 0.26162020362301863
  actual coverage: 0.9100691775557264
  accuracy: 0.9966216216216216
  accepted: 118

In [ ]:
# ============================================================
# 67C-C — STRICT SLICE / BLOCK METRIC AUDIT
#
# Development-only audit on internal DEV.
#
# No DICOM reread.
# No outer validation.
# No internal_test.
# No official test.
# No rule freeze yet.
# ============================================================

import numpy as np
import pandas as pd
import json


# ------------------------------------------------------------
# 0 — Safety
# ------------------------------------------------------------

assert ACCESS_OUTER_VALIDATION_AXIAL_GT is False
assert RUN_LOCKED_VALIDATION is False
assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False
assert AXIAL_PAIRING_VALIDATED_67C is False


# ------------------------------------------------------------
# 1 — DEV rows already produced by 67C-B
# ------------------------------------------------------------

dev = pd.DataFrame([
    r
    for r in pairing_prediction_rows
    if (
        r["cohort"] == "67C_dev"
        and
        r["exact_level_assignment"] is not None
    )
]).copy()


assert len(dev) == 1301


# ------------------------------------------------------------
# 2 — Strict metrics
#
# IMPORTANT:
#
# exact_level_assignment:
#     predicted AXIAL SERIES matches GT series for that level
#
# strict_exact_slice:
#     correct series AND exact GT instance
#
# strict_within_1:
#     correct series AND <= 1 InstanceNumber from GT
#
# strict_within_2:
#     correct series AND <= 2 InstanceNumbers from GT
#
# Wrong-series predictions count as FALSE in all strict metrics.
# ------------------------------------------------------------

def strict_metric(row, field):

    if not bool(row["exact_level_assignment"]):
        return False

    value = row.get(field)

    if pd.isna(value):
        return None

    return bool(value)


dev["strict_exact_slice"] = dev.apply(
    lambda r: strict_metric(
        r,
        "exact_slice_hit",
    ),
    axis=1,
)

dev["strict_within_1"] = dev.apply(
    lambda r: strict_metric(
        r,
        "within_1",
    ),
    axis=1,
)

dev["strict_within_2"] = dev.apply(
    lambda r: strict_metric(
        r,
        "within_2",
    ),
    axis=1,
)


def metric_summary(series):

    valid = series.dropna()

    if len(valid) == 0:
        return {
            "n": 0,
            "hits": 0,
            "rate": None,
        }

    hits = int(
        valid.astype(bool).sum()
    )

    return {
        "n": int(len(valid)),
        "hits": hits,
        "rate": float(
            hits / len(valid)
        ),
    }


series_level = metric_summary(
    dev["exact_level_assignment"]
)

exact_slice = metric_summary(
    dev["strict_exact_slice"]
)

within_1 = metric_summary(
    dev["strict_within_1"]
)

within_2 = metric_summary(
    dev["strict_within_2"]
)


# ------------------------------------------------------------
# 3 — Instance error distribution
# ------------------------------------------------------------

instance_diff = (
    pd.to_numeric(
        dev["instance_diff_abs"],
        errors="coerce",
    )
    .dropna()
)


instance_stats = {

    "n":
        int(len(instance_diff)),

    "median":
        float(
            instance_diff.median()
        )
        if len(instance_diff)
        else None,

    "p75":
        float(
            instance_diff.quantile(0.75)
        )
        if len(instance_diff)
        else None,

    "p90":
        float(
            instance_diff.quantile(0.90)
        )
        if len(instance_diff)
        else None,

    "p95":
        float(
            instance_diff.quantile(0.95)
        )
        if len(instance_diff)
        else None,

    "max":
        float(
            instance_diff.max()
        )
        if len(instance_diff)
        else None,
}


# ------------------------------------------------------------
# 4 — Per-level strict metrics
# ------------------------------------------------------------

per_level = {}


for level in CANONICAL_LEVELS:

    d = dev[
        dev["level"] == level
    ]

    per_level[level] = {

        "n":
            int(len(d)),

        "series_level_accuracy":
            metric_summary(
                d[
                    "exact_level_assignment"
                ]
            )["rate"],

        "exact_slice_rate":
            metric_summary(
                d[
                    "strict_exact_slice"
                ]
            )["rate"],

        "within_1_rate":
            metric_summary(
                d[
                    "strict_within_1"
                ]
            )["rate"],

        "within_2_rate":
            metric_summary(
                d[
                    "strict_within_2"
                ]
            )["rate"],
    }


# ------------------------------------------------------------
# 5 — How often does one axial series represent several levels?
#
# This tells us how informative "series-level accuracy" actually is.
# ------------------------------------------------------------

dev_study_ids = set(
    internal_67c_split[
        "67C_dev"
    ]
)


gt_dev_axial = (
    axial_reference_rows_train[
        axial_reference_rows_train[
            "study_id"
        ].isin(
            dev_study_ids
        )
    ]
    .copy()
)


gt_dev_axial = gt_dev_axial[
    gt_dev_axial[
        "level_normalized"
    ].isin(
        CANONICAL_LEVELS
    )
]


series_level_map = (
    gt_dev_axial[
        [
            "study_id",
            "series_id",
            "level_normalized",
        ]
    ]
    .drop_duplicates()
)


levels_per_series = (
    series_level_map
    .groupby(
        [
            "study_id",
            "series_id",
        ]
    )[
        "level_normalized"
    ]
    .nunique()
)


max_levels_per_series_by_study = (
    levels_per_series
    .groupby(
        level=0
    )
    .max()
)


shared_series_distribution = (
    max_levels_per_series_by_study
    .value_counts()
    .sort_index()
    .to_dict()
)


studies_multi_level_series = int(
    (
        max_levels_per_series_by_study
        >= 2
    ).sum()
)


studies_with_gt = int(
    len(
        max_levels_per_series_by_study
    )
)


multi_level_series_rate = (
    studies_multi_level_series
    / studies_with_gt
    if studies_with_gt
    else None
)


# ------------------------------------------------------------
# 6 — Print authoritative audit
# ------------------------------------------------------------

print(
    "=" * 60
)

print(
    "67C-C STRICT SLICE / BLOCK METRIC AUDIT"
)

print(
    "=" * 60
)

print()

print(
    "DEV evaluable levels:",
    len(dev),
)

print()

print(
    "SERIES-LEVEL:"
)

print(
    "  exact-level assignment:",
    series_level,
)

print()

print(
    "STRICT SLICE/BLOCK:"
)

print(
    "  exact slice:",
    exact_slice,
)

print(
    "  within ±1 instance:",
    within_1,
)

print(
    "  within ±2 instances:",
    within_2,
)

print()

print(
    "INSTANCE DIFFERENCE:"
)

for k, v in instance_stats.items():
    print(
        f"  {k}:",
        v,
    )

print()

print(
    "STRICT METRICS BY LEVEL:"
)

for level in CANONICAL_LEVELS:

    print(
        f"  {level}:",
        per_level[level],
    )

print()

print(
    "AXIAL SERIES SHARING DIAGNOSTIC:"
)

print(
    "  DEV studies with axial GT:",
    studies_with_gt,
)

print(
    "  studies where one axial series covers >=2 levels:",
    studies_multi_level_series,
)

print(
    "  rate:",
    multi_level_series_rate,
)

print(
    "  max-levels-per-series distribution:",
    {
        int(k): int(v)
        for k, v
        in shared_series_distribution.items()
    },
)


# ------------------------------------------------------------
# 7 — Persist audit
# ------------------------------------------------------------

slice_audit = {

    "stage":
        "67C-C_STRICT_SLICE_AUDIT",

    "pairing_method":
        "BASELINE_A_NEAREST_PLANE_PLUS_FOV",

    "dev_evaluable_levels":
        int(len(dev)),

    "series_level_accuracy":
        series_level,

    "strict_exact_slice":
        exact_slice,

    "strict_within_1":
        within_1,

    "strict_within_2":
        within_2,

    "instance_difference":
        instance_stats,

    "per_level":
        per_level,

    "axial_series_sharing": {

        "studies_with_axial_gt":
            studies_with_gt,

        "studies_with_multi_level_series":
            studies_multi_level_series,

        "multi_level_series_rate":
            multi_level_series_rate,

        "max_levels_per_series_distribution":
            {
                str(k): int(v)
                for k, v
                in shared_series_distribution.items()
            },
    },

    "outer_validation_axial_gt_accessed":
        False,

    "internal_test_accessed":
        False,

    "official_test_accessed":
        False,

    "rule_frozen":
        False,

    "AXIAL_PAIRING_VALIDATED_67C":
        False,
}


safe_write_text(
    METRICS_DIR
    / "67C_C_strict_slice_audit.json",

    json.dumps(
        slice_audit,
        indent=2,
    ),
)


print()
print(
    "=" * 60
)

print(
    "67C-C STRICT AUDIT COMPLETE"
)

print(
    "=" * 60
)

print(
    "NO RULE FROZEN YET"
)

print(
    "Outer validation axial GT accessed: False"
)

print(
    "Internal test accessed: False"
)

print(
    "Official test accessed: False"
)

print(
    "AXIAL_PAIRING_VALIDATED_67C: False"
)

67C-C STRICT SLICE / BLOCK METRIC AUDIT

DEV evaluable levels: 1301

SERIES-LEVEL:
  exact-level assignment: {'n': 1301, 'hits': 1289, 'rate': 0.9907763259031515}

STRICT SLICE/BLOCK:
  exact slice: {'n': 1301, 'hits': 921, 'rate': 0.7079169869331283}
  within ±1 instance: {'n': 1301, 'hits': 1254, 'rate': 0.9638739431206764}
  within ±2 instances: {'n': 1301, 'hits': 1280, 'rate': 0.9838585703305149}

INSTANCE DIFFERENCE:
  n: 1289
  median: 0.0
  p75: 1.0
  p90: 1.0
  p95: 1.0
  max: 6.0

STRICT METRICS BY LEVEL:
  L1-L2: {'n': 253, 'series_level_accuracy': 0.9960474308300395, 'exact_slice_rate': 0.7984189723320159, 'within_1_rate': 0.9723320158102767, 'within_2_rate': 0.9920948616600791}
  L2-L3: {'n': 260, 'series_level_accuracy': 0.9961538461538462, 'exact_slice_rate': 0.7884615384615384, 'within_1_rate': 0.9884615384615385, 'within_2_rate': 0.9961538461538462}
  L3-L4: {'n': 263, 'series_level_accuracy': 0.9809885931558935, 'exact_slice_rate': 0.688212927756654, 'within_1_rate': 

In [ ]:
# ============================================================
# 67C-C — STRICT SELECTIVE POLICY AUDIT
#
# Final DEV-only audit before rule freeze.
#
# NO DICOM reads
# NO outer validation
# NO internal_test
# NO official test
# ============================================================

import numpy as np
import pandas as pd


assert ACCESS_OUTER_VALIDATION_AXIAL_GT is False
assert RUN_LOCKED_VALIDATION is False
assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False
assert AXIAL_PAIRING_VALIDATED_67C is False


# 'dev' comes from the previous strict-slice audit.
assert "dev" in globals()
assert len(dev) == 1301

for col in [
    "strict_exact_slice",
    "strict_within_1",
    "strict_within_2",
]:
    assert col in dev.columns


# Thresholds already observed during the DEV-only audit.
# 0.0 = raw/no extra abstention.
MARGIN_THRESHOLDS = [
    0.0,
    0.0904461716448477,
    0.1172779103904702,
    0.26162020362301863,
    0.5041318346544338,
    0.6291502342730732,
    1.6271208388227905,
]


def bool_rate(series):
    valid = series.dropna()

    if len(valid) == 0:
        return None

    return float(
        valid.astype(bool).mean()
    )


rows = []


for threshold in MARGIN_THRESHOLDS:

    accepted_mask = (
        dev["selected_inside_fov"]
        .fillna(False)
        .astype(bool)
        &
        dev["distance_margin_mm"].notna()
        &
        (
            dev["distance_margin_mm"]
            >= threshold
        )
    )

    accepted = dev[
        accepted_mask
    ].copy()

    coverage = (
        len(accepted)
        / len(dev)
    )

    series_acc = bool_rate(
        accepted[
            "exact_level_assignment"
        ]
    )

    exact_slice_acc = bool_rate(
        accepted[
            "strict_exact_slice"
        ]
    )

    within_1_acc = bool_rate(
        accepted[
            "strict_within_1"
        ]
    )

    within_2_acc = bool_rate(
        accepted[
            "strict_within_2"
        ]
    )

    p95_plane_distance = (
        float(
            accepted[
                "selected_plane_distance_mm"
            ].quantile(0.95)
        )
        if len(accepted)
        else None
    )

    rows.append({
        "min_margin_mm": threshold,
        "accepted": int(len(accepted)),
        "coverage": coverage,
        "series_accuracy": series_acc,
        "exact_slice_accuracy": exact_slice_acc,
        "within_1_accuracy": within_1_acc,
        "within_2_accuracy": within_2_acc,
        "p95_selected_plane_distance_mm":
            p95_plane_distance,
    })


strict_selective_df = pd.DataFrame(
    rows
)


print("=" * 78)
print("67C-C STRICT SELECTIVE DEV AUDIT")
print("=" * 78)
print()

print(
    strict_selective_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ============================================================
# Suggested candidate:
#
# Highest strict ±2 accuracy among policies retaining >=90%
# DEV coverage.
#
# Tie breakers:
# 1) ±1 accuracy
# 2) series accuracy
# 3) higher coverage
#
# This is still DEV-only. Validation remains unopened.
# ============================================================

eligible = strict_selective_df[
    strict_selective_df[
        "coverage"
    ] >= 0.90
].copy()


assert len(eligible) > 0


eligible = eligible.sort_values(
    [
        "within_2_accuracy",
        "within_1_accuracy",
        "series_accuracy",
        "coverage",
    ],
    ascending=[
        False,
        False,
        False,
        False,
    ],
)


suggested = eligible.iloc[0]


print()
print("=" * 78)
print("SUGGESTED 67C-C FREEZE CANDIDATE")
print("=" * 78)

for k, v in suggested.items():
    print(f"{k}: {v}")


print()
print("NO RULE FROZEN YET")
print("Outer validation axial GT accessed: False")
print("Internal test accessed: False")
print("Official test accessed: False")
print("AXIAL_PAIRING_VALIDATED_67C: False")

67C-C STRICT SELECTIVE DEV AUDIT

 min_margin_mm  accepted  coverage  series_accuracy  exact_slice_accuracy  within_1_accuracy  within_2_accuracy  p95_selected_plane_distance_mm
      0.000000      1301  1.000000         0.990776              0.707917           0.963874           0.983859                        2.209716
      0.090446      1249  0.960031         0.995196              0.714171           0.971177           0.988791                        2.170115
      0.117278      1236  0.950038         0.995955              0.714401           0.971683           0.989482                        2.166685
      0.261620      1184  0.910069         0.996622              0.722973           0.973818           0.990709                        2.132042
      0.504132      1118  0.859339         0.997317              0.733453           0.977639           0.992844                        2.029928
      0.629150      1079  0.829362         0.998146              0.738647           0.977757          

In [ ]:
# ============================================================
# 67C-C — FINAL DETERMINISTIC PAIRING POLICY FREEZE
#
# This cell formally freezes the axial pairing method,
# abstention rule and 67C-D validation gates BEFORE any
# outer-validation axial GT is accessed.
# ============================================================

import json
from datetime import datetime, timezone


# ============================================================
# 0 — HARD SAFETY CHECK
# ============================================================

assert ACCESS_OUTER_VALIDATION_AXIAL_GT is False
assert RUN_LOCKED_VALIDATION is False
assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False
assert AXIAL_PAIRING_VALIDATED_67C is False
assert TRAINING_PERFORMED is False

assert baseline_a_dev_accuracy is not None
assert baseline_b_dev_accuracy is not None

# Baseline B produced no improvement.
assert abs(
    baseline_a_dev_accuracy
    - baseline_b_dev_accuracy
) < 1e-12


# ============================================================
# 1 — FROZEN METHOD
# ============================================================

FROZEN_67C_PAIRING_METHOD = (
    "BASELINE_A_NEAREST_PLANE_PLUS_FOV"
)

FROZEN_67C_REQUIRE_INSIDE_FOV = True

# Exact DEV-selected operating point.
FROZEN_67C_MIN_MARGIN_MM = (
    0.26162020362301863
)

FROZEN_67C_MAX_PLANE_DISTANCE_MM = (
    23.818094078633493
)


# ============================================================
# 2 — EXPECTED DEV REFERENCE
#
# These are NOT validation metrics.
# They are recorded only to prove exactly which DEV operating
# point was frozen.
# ============================================================

FROZEN_67C_DEV_REFERENCE = {

    "evaluable_levels":
        1301,

    "accepted":
        1184,

    "coverage":
        0.9100691775557264,

    "series_accuracy":
        0.9966216216216216,

    "exact_slice_accuracy":
        0.722972972972973,

    "within_1_accuracy":
        0.9738175675675675,

    "within_2_accuracy":
        0.9907094594594594,

    "p95_selected_plane_distance_mm":
        2.132042274077186,
}


# ============================================================
# 3 — RECOMPUTE FROZEN POLICY ON DEV AND ASSERT NO DRIFT
# ============================================================

assert "dev" in globals()
assert len(dev) == 1301


_frozen_accept_mask = (

    dev[
        "selected_inside_fov"
    ]
    .fillna(False)
    .astype(bool)

    &

    dev[
        "selected_plane_distance_mm"
    ]
    .notna()

    &

    (
        dev[
            "selected_plane_distance_mm"
        ]
        <= FROZEN_67C_MAX_PLANE_DISTANCE_MM
    )

    &

    dev[
        "distance_margin_mm"
    ]
    .notna()

    &

    (
        dev[
            "distance_margin_mm"
        ]
        >= FROZEN_67C_MIN_MARGIN_MM
    )
)


_frozen_dev = dev[
    _frozen_accept_mask
].copy()


def _frozen_bool_rate(series):

    valid = series.dropna()

    assert len(valid) > 0

    return float(
        valid.astype(bool).mean()
    )


_recomputed = {

    "evaluable_levels":
        int(len(dev)),

    "accepted":
        int(len(_frozen_dev)),

    "coverage":
        float(
            len(_frozen_dev)
            / len(dev)
        ),

    "series_accuracy":
        _frozen_bool_rate(
            _frozen_dev[
                "exact_level_assignment"
            ]
        ),

    "exact_slice_accuracy":
        _frozen_bool_rate(
            _frozen_dev[
                "strict_exact_slice"
            ]
        ),

    "within_1_accuracy":
        _frozen_bool_rate(
            _frozen_dev[
                "strict_within_1"
            ]
        ),

    "within_2_accuracy":
        _frozen_bool_rate(
            _frozen_dev[
                "strict_within_2"
            ]
        ),

    "p95_selected_plane_distance_mm":
        float(
            _frozen_dev[
                "selected_plane_distance_mm"
            ].quantile(0.95)
        ),
}


for _key, _expected in (
    FROZEN_67C_DEV_REFERENCE.items()
):

    _actual = _recomputed[_key]

    if isinstance(
        _expected,
        float,
    ):

        assert abs(
            _actual
            - _expected
        ) < 1e-12, (
            f"DEV freeze drift in {_key}: "
            f"{_actual} != {_expected}"
        )

    else:

        assert (
            _actual
            == _expected
        )


print(
    "Frozen DEV operating-point parity: PASS"
)


# ============================================================
# 4 — RUNTIME FROZEN POLICY
#
# Candidate generation remains EXACTLY the 67C-B geometry
# algorithm.
#
# This function ONLY performs the frozen selection/abstention.
# ============================================================

def apply_frozen_67c_pairing_policy(
    candidates_for_level,
):

    if not candidates_for_level:

        return {
            "emit": False,
            "reason": "NO_CANDIDATES",
            "candidate": None,
            "margin_mm": None,
        }


    ranked = sorted(
        candidates_for_level,
        key=lambda c: (
            not c[
                "inside_fov"
            ],
            c[
                "plane_distance_mm"
            ],
        ),
    )


    winner = ranked[0]


    if not bool(
        winner[
            "inside_fov"
        ]
    ):

        return {
            "emit": False,
            "reason": "OUTSIDE_FOV",
            "candidate": winner,
            "margin_mm": None,
        }


    if (
        winner[
            "plane_distance_mm"
        ]
        >
        FROZEN_67C_MAX_PLANE_DISTANCE_MM
    ):

        return {
            "emit": False,
            "reason": "PLANE_DISTANCE",
            "candidate": winner,
            "margin_mm": None,
        }


    if len(ranked) < 2:

        return {
            "emit": False,
            "reason": "NO_SECOND_CANDIDATE",
            "candidate": winner,
            "margin_mm": None,
        }


    distance_1 = float(
        ranked[0][
            "plane_distance_mm"
        ]
    )

    distance_2 = float(
        ranked[1][
            "plane_distance_mm"
        ]
    )

    margin_mm = (
        distance_2
        - distance_1
    )


    if (
        margin_mm
        <
        FROZEN_67C_MIN_MARGIN_MM
    ):

        return {
            "emit": False,
            "reason": "AMBIGUOUS_MARGIN",
            "candidate": winner,
            "margin_mm": margin_mm,
        }


    return {
        "emit": True,
        "reason": "PASS",
        "candidate": winner,
        "margin_mm": margin_mm,
    }


# ============================================================
# 5 — FROZEN 67C-D VALIDATION GATES
#
# These are declared NOW, before opening outer-validation
# axial GT.
#
# Exact-slice is diagnostic, not a hard gate.
# ±1 / ±2 are the primary strict pairing metrics.
# ============================================================

FROZEN_67C_D_VALIDATION_GATES = {

    # Among pairing-evaluable levels after upstream
    # localization reaches 67C.
    "pairing_conditional_coverage_min":
        0.80,

    # Strict slice tolerance metrics, accuracy GIVEN EMIT.
    "strict_within_1_accuracy_min":
        0.90,

    "strict_within_2_accuracy_min":
        0.97,

    # Secondary anatomical-level sanity gate.
    "series_level_accuracy_min":
        0.95,

    # Full upstream + pairing pipeline.
    #
    # Upstream 67B2 already has its own abstention, therefore
    # full E2E coverage is expected to be lower than the
    # geometry-only DEV coverage.
    "full_pipeline_gt_evaluable_coverage_min":
        0.60,
}


# ============================================================
# 6 — FORMAL FREEZE MANIFEST
# ============================================================

HEURISTIC_RULES_FROZEN = True


freeze_manifest = {

    "notebook":
        "67C_postE50_axial_level_pairing",

    "stage":
        "67C-C",

    "status":
        "FROZEN_BEFORE_LOCKED_VALIDATION",

    "generated_at":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "pairing_method":
        FROZEN_67C_PAIRING_METHOD,

    "selection_rule": {
        "require_inside_fov":
            FROZEN_67C_REQUIRE_INSIDE_FOV,

        "ranking":
            "inside_fov priority, then minimum absolute point-to-plane distance",

        "min_margin_mm":
            FROZEN_67C_MIN_MARGIN_MM,

        "max_plane_distance_mm":
            FROZEN_67C_MAX_PLANE_DISTANCE_MM,
    },

    "baseline_comparison": {

        "baseline_a_dev_accuracy":
            float(
                baseline_a_dev_accuracy
            ),

        "baseline_b_dev_accuracy":
            float(
                baseline_b_dev_accuracy
            ),

        "decision":
            "BASELINE_A_FROZEN_BY_SIMPLICITY_NO_BENEFIT_FROM_HUNGARIAN",
    },

    "dev_reference_metrics":
        FROZEN_67C_DEV_REFERENCE,

    "locked_validation_gates":
        FROZEN_67C_D_VALIDATION_GATES,

    "metric_contract": {

        "primary": [
            "strict_within_1_accuracy_given_emit",
            "strict_within_2_accuracy_given_emit",
            "pairing_conditional_coverage",
            "full_pipeline_gt_evaluable_coverage",
        ],

        "secondary": [
            "exact_slice_accuracy",
            "series_level_accuracy",
            "plane_distance_distribution",
        ],
    },

    "outer_validation_axial_gt_accessed":
        False,

    "internal_test_accessed":
        False,

    "official_test_accessed":
        False,

    "training_performed":
        False,

    "heuristic_rules_frozen":
        True,

    "AXIAL_PAIRING_VALIDATED_67C":
        False,

    "next_stage":
        "67C-D_LOCKED_E2E_VALIDATION",
}


FREEZE_MANIFEST_PATH = (
    METRICS_DIR
    / "67C_C_frozen_pairing_policy.json"
)


safe_write_text(
    FREEZE_MANIFEST_PATH,
    json.dumps(
        freeze_manifest,
        indent=2,
    ),
)


FREEZE_MANIFEST_SHA256 = (
    sha256_file(
        FREEZE_MANIFEST_PATH
    )
)


# ============================================================
# 7 — FINAL HARD ASSERTIONS
# ============================================================

assert HEURISTIC_RULES_FROZEN is True

assert ACCESS_OUTER_VALIDATION_AXIAL_GT is False
assert RUN_LOCKED_VALIDATION is False

assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False

assert AXIAL_PAIRING_VALIDATED_67C is False


# ============================================================
# 8 — CLOSING BLOCK
# ============================================================

print()
print("=" * 62)
print("67C-C PAIRING POLICY FREEZE COMPLETE")
print("=" * 62)

print()

print(
    "Method:",
    FROZEN_67C_PAIRING_METHOD,
)

print(
    "Require inside FOV:",
    FROZEN_67C_REQUIRE_INSIDE_FOV,
)

print(
    "Min margin mm:",
    FROZEN_67C_MIN_MARGIN_MM,
)

print(
    "Max plane distance mm:",
    FROZEN_67C_MAX_PLANE_DISTANCE_MM,
)

print()

print(
    "Frozen DEV coverage:",
    FROZEN_67C_DEV_REFERENCE[
        "coverage"
    ],
)

print(
    "Frozen DEV strict ±1:",
    FROZEN_67C_DEV_REFERENCE[
        "within_1_accuracy"
    ],
)

print(
    "Frozen DEV strict ±2:",
    FROZEN_67C_DEV_REFERENCE[
        "within_2_accuracy"
    ],
)

print(
    "Frozen DEV series accuracy:",
    FROZEN_67C_DEV_REFERENCE[
        "series_accuracy"
    ],
)

print()

print(
    "67C-D locked validation gates:"
)

for _gate, _value in (
    FROZEN_67C_D_VALIDATION_GATES.items()
):
    print(
        f"  {_gate}: {_value}"
    )

print()

print(
    "Freeze manifest:",
    FREEZE_MANIFEST_PATH,
)

print(
    "Freeze manifest SHA256:",
    FREEZE_MANIFEST_SHA256,
)

print()

print(
    "Outer validation axial GT accessed:",
    ACCESS_OUTER_VALIDATION_AXIAL_GT,
)

print(
    "Internal test accessed:",
    False,
)

print(
    "Official test accessed:",
    OFFICIAL_TEST_ACCESSED,
)

print()

print(
    "HEURISTIC_RULES_FROZEN:",
    HEURISTIC_RULES_FROZEN,
)

print(
    "AXIAL_PAIRING_VALIDATED_67C:",
    AXIAL_PAIRING_VALIDATED_67C,
)

print()

print(
    "NEXT: 67C-D LOCKED E2E VALIDATION"
)

Frozen DEV operating-point parity: PASS

67C-C PAIRING POLICY FREEZE COMPLETE

Method: BASELINE_A_NEAREST_PLANE_PLUS_FOV
Require inside FOV: True
Min margin mm: 0.26162020362301863
Max plane distance mm: 23.818094078633493

Frozen DEV coverage: 0.9100691775557264
Frozen DEV strict ±1: 0.9738175675675675
Frozen DEV strict ±2: 0.9907094594594594
Frozen DEV series accuracy: 0.9966216216216216

67C-D locked validation gates:
  pairing_conditional_coverage_min: 0.8
  strict_within_1_accuracy_min: 0.9
  strict_within_2_accuracy_min: 0.97
  series_level_accuracy_min: 0.95
  full_pipeline_gt_evaluable_coverage_min: 0.6

Freeze manifest: /content/drive/MyDrive/PFI_MVP/metrics/post_e50/67C/67C_C_frozen_pairing_policy.json
Freeze manifest SHA256: 4afbe5af1cfbed3f11a302ae9905ca6a4a6818824b60d54322341d4456505f34

Outer validation axial GT accessed: False
Internal test accessed: False
Official test accessed: False

HEURISTIC_RULES_FROZEN: True
AXIAL_PAIRING_VALIDATED_67C: False

NEXT: 67C-D LOCKED E

In [ ]:
# ============================================================
# 67C-D — LOCKED E2E VALIDATION PRE-FLIGHT
#
# STANDALONE CELL:
# safe to run after switching Colab runtime to L4.
#
# IMPORTANT:
# - DOES NOT open outer-validation axial GT.
# - DOES NOT access internal_test images/labels.
# - DOES NOT access official RSNA test.
# - DOES NOT change any frozen rule.
# - DOES NOT train anything.
# ============================================================

import hashlib
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch


# ============================================================
# 0 — ENVIRONMENT
# ============================================================

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False


PFI_DRIVE_ROOT = Path(
    os.environ.get(
        "PFI_DRIVE_ROOT",
        "/content/drive/MyDrive/PFI_MVP"
        if IN_COLAB
        else "./PFI_MVP",
    )
)

RSNA_ROOT = (
    PFI_DRIVE_ROOT
    / "data"
    / "RSNA_LUMBAR_DISC"
)

METRICS_67B2_DIR = (
    PFI_DRIVE_ROOT
    / "metrics"
    / "post_e50"
    / "67B2"
)

METRICS_67C_DIR = (
    PFI_DRIVE_ROOT
    / "metrics"
    / "post_e50"
    / "67C"
)

RESULTS_67C_DIR = (
    PFI_DRIVE_ROOT
    / "results"
    / "post_e50"
    / "67C"
)

MODEL_67B2_DIR = (
    PFI_DRIVE_ROOT
    / "models"
    / "research"
    / "67B2"
)

SAGITTAL_MODEL_PATH = (
    PFI_DRIVE_ROOT
    / "models"
    / "final"
    / "sagittal_spider_multiclass_final_best_cf11dcc0.pt"
)

VALIDATED_67B2_MODEL_PATH = (
    MODEL_67B2_DIR
    / "absolute_lumbar_window_scorer_validated.pt"
)

LOCKED_67B2_DIR = (
    METRICS_67B2_DIR
    / "locked_validation"
)

LOCKED_67B2_PREDICTIONS_PATH = (
    LOCKED_67B2_DIR
    / "locked_validation_predictions.csv"
)

LOCKED_67B2_SUMMARY_PATH = (
    LOCKED_67B2_DIR
    / "67B2_locked_validation_summary.json"
)

FREEZE_67C_PATH = (
    METRICS_67C_DIR
    / "67C_C_frozen_pairing_policy.json"
)


# ============================================================
# 1 — EXPECTED FROZEN IDENTITIES
# ============================================================

EXPECTED_67C_FREEZE_SHA256 = (
    "4afbe5af1cfbed3f11a302ae9905ca6a"
    "4a6818824b60d54322341d4456505f34"
)

EXPECTED_SAGITTAL_SHA256 = (
    "cf11dcc0ad77a7c787e64a796a2fd739"
    "8ef906add461cef4b3d61f1a5238e944"
)

EXPECTED_67B2_VALIDATED_SHA256 = (
    "bb9e9d812972305cbc3dc87b96b95f78"
    "b7c9de4afc8ac4e6f071b35ae6fffd2c"
)

EXPECTED_67B2_LOCKED_SUMMARY_SHA256 = (
    "2b74c8a5b0d8ee562f6a774e136abf95"
    "225074d32c32c7646a5ca3f0fd6e8dcb"
)

EXPECTED_67B2_LOCKED_PREDICTIONS_SHA256 = (
    "baae269fe8f0e7bc7725a44ebebbb143"
    "1600d7ce23d56b7dd82c05a0cbcc0405"
)

EXPECTED_SERIES_CSV_SHA256 = (
    "bf8cc1aa55e4b5536b2f0fa8a060fc19"
    "12faeb65f9ba422d9d91b92a947b1c33"
)


# ============================================================
# 2 — HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def opaque_id(raw_value) -> str:
    return hashlib.sha256(
        str(raw_value).encode("utf-8")
    ).hexdigest()[:12]


def build_reproducible_study_split(
    study_ids,
    seed=2026,
    train_frac=0.70,
    validation_frac=0.15,
):
    ordered = sorted(
        str(x)
        for x in study_ids
    )

    rng = np.random.default_rng(seed)

    shuffled = list(ordered)
    rng.shuffle(shuffled)

    n = len(shuffled)

    n_train = int(
        n * train_frac
    )

    n_validation = int(
        n * validation_frac
    )

    return {
        "train":
            sorted(
                shuffled[:n_train]
            ),

        "validation":
            sorted(
                shuffled[
                    n_train:
                    n_train + n_validation
                ]
            ),

        "internal_test":
            sorted(
                shuffled[
                    n_train + n_validation:
                ]
            ),
    }


# ============================================================
# 3 — GPU CHECK
# ============================================================

assert torch.cuda.is_available(), (
    "67C-D requires GPU runtime. "
    "Switch Colab to GPU L4 before continuing."
)

DEVICE = torch.device("cuda")

GPU_NAME = torch.cuda.get_device_name(0)

print("DEVICE:", DEVICE)
print("GPU:", GPU_NAME)

if "L4" not in GPU_NAME.upper():
    print(
        "WARNING: runtime is GPU but not NVIDIA L4. "
        "Method is unchanged, but L4 is the expected runtime."
    )


# ============================================================
# 4 — ARTIFACT EXISTENCE
# ============================================================

required_paths = [
    FREEZE_67C_PATH,
    SAGITTAL_MODEL_PATH,
    VALIDATED_67B2_MODEL_PATH,
    LOCKED_67B2_SUMMARY_PATH,
    LOCKED_67B2_PREDICTIONS_PATH,
    RSNA_ROOT / "train_series_descriptions.csv",
    RSNA_ROOT / "train_images",
]


for path in required_paths:
    assert path.exists(), (
        f"Required artifact missing: {path}"
    )


print()
print("Required artifact existence: PASS")


# ============================================================
# 5 — CRYPTOGRAPHIC IDENTITY CHECK
# ============================================================

actual_hashes = {
    "67C_freeze":
        sha256_file(
            FREEZE_67C_PATH
        ),

    "sagittal_checkpoint":
        sha256_file(
            SAGITTAL_MODEL_PATH
        ),

    "67B2_validated_model":
        sha256_file(
            VALIDATED_67B2_MODEL_PATH
        ),

    "67B2_locked_summary":
        sha256_file(
            LOCKED_67B2_SUMMARY_PATH
        ),

    "67B2_locked_predictions":
        sha256_file(
            LOCKED_67B2_PREDICTIONS_PATH
        ),

    "series_csv":
        sha256_file(
            RSNA_ROOT
            / "train_series_descriptions.csv"
        ),
}


expected_hashes = {
    "67C_freeze":
        EXPECTED_67C_FREEZE_SHA256,

    "sagittal_checkpoint":
        EXPECTED_SAGITTAL_SHA256,

    "67B2_validated_model":
        EXPECTED_67B2_VALIDATED_SHA256,

    "67B2_locked_summary":
        EXPECTED_67B2_LOCKED_SUMMARY_SHA256,

    "67B2_locked_predictions":
        EXPECTED_67B2_LOCKED_PREDICTIONS_SHA256,

    "series_csv":
        EXPECTED_SERIES_CSV_SHA256,
}


for name in expected_hashes:

    actual = actual_hashes[name]
    expected = expected_hashes[name]

    print(
        name,
        actual,
        "MATCH"
        if actual == expected
        else "MISMATCH",
    )

    assert actual == expected, (
        f"HARD STOP — SHA mismatch for {name}"
    )


print()
print("Frozen artifact identity: PASS")


# ============================================================
# 6 — LOAD 67C-C FROZEN CONTRACT
# ============================================================

freeze_67c = json.loads(
    FREEZE_67C_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    freeze_67c["status"]
    ==
    "FROZEN_BEFORE_LOCKED_VALIDATION"
)

assert (
    freeze_67c[
        "pairing_method"
    ]
    ==
    "BASELINE_A_NEAREST_PLANE_PLUS_FOV"
)

assert (
    freeze_67c[
        "heuristic_rules_frozen"
    ]
    is True
)

assert (
    freeze_67c[
        "outer_validation_axial_gt_accessed"
    ]
    is False
)

assert (
    freeze_67c[
        "internal_test_accessed"
    ]
    is False
)

assert (
    freeze_67c[
        "official_test_accessed"
    ]
    is False
)


FROZEN_67C_REQUIRE_INSIDE_FOV = bool(
    freeze_67c[
        "selection_rule"
    ][
        "require_inside_fov"
    ]
)

FROZEN_67C_MIN_MARGIN_MM = float(
    freeze_67c[
        "selection_rule"
    ][
        "min_margin_mm"
    ]
)

FROZEN_67C_MAX_PLANE_DISTANCE_MM = float(
    freeze_67c[
        "selection_rule"
    ][
        "max_plane_distance_mm"
    ]
)

FROZEN_67C_D_VALIDATION_GATES = dict(
    freeze_67c[
        "locked_validation_gates"
    ]
)


assert (
    FROZEN_67C_REQUIRE_INSIDE_FOV
    is True
)

assert abs(
    FROZEN_67C_MIN_MARGIN_MM
    - 0.26162020362301863
) < 1e-15

assert abs(
    FROZEN_67C_MAX_PLANE_DISTANCE_MM
    - 23.818094078633493
) < 1e-12


print()
print("67C-C frozen policy loaded: PASS")

print(
    "Min margin:",
    FROZEN_67C_MIN_MARGIN_MM,
)

print(
    "Max plane distance:",
    FROZEN_67C_MAX_PLANE_DISTANCE_MM,
)

print(
    "Gates:",
    FROZEN_67C_D_VALIDATION_GATES,
)


# ============================================================
# 7 — VERIFY CLOSED 67B2 VALIDATION
# ============================================================

summary_67b2 = json.loads(
    LOCKED_67B2_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    summary_67b2[
        "validation_total"
    ]
    == 296
)

assert (
    summary_67b2[
        "AUTOMATIC_DISC_LOCALIZATION_VALIDATED"
    ]
    is True
)

assert (
    summary_67b2[
        "internal_test_accessed"
    ]
    is False
)

assert (
    summary_67b2[
        "official_test_accessed"
    ]
    is False
)

assert all(
    summary_67b2[
        "gate_results"
    ].values()
)


print()
print("67B2 closed validation evidence: PASS")


# ============================================================
# 8 — RECONSTRUCT EXACT OUTER SPLIT
#
# IMPORTANT:
# uses ONLY series metadata.
#
# train_label_coordinates.csv is NOT loaded here.
# Therefore 67C-D axial GT remains unopened.
# ============================================================

series_df_67c_d = pd.read_csv(
    RSNA_ROOT
    / "train_series_descriptions.csv",

    dtype={
        "study_id": "string",
        "series_id": "string",
    },
)


outer_split_67c_d = (
    build_reproducible_study_split(
        series_df_67c_d[
            "study_id"
        ]
        .dropna()
        .unique(),
        seed=SEED,
    )
)


assert (
    len(
        outer_split_67c_d[
            "train"
        ]
    )
    == 1382
)

assert (
    len(
        outer_split_67c_d[
            "validation"
        ]
    )
    == 296
)

assert (
    len(
        outer_split_67c_d[
            "internal_test"
        ]
    )
    == 297
)


LOCKED_67C_VALIDATION_RAW = list(
    outer_split_67c_d[
        "validation"
    ]
)

LOCKED_67C_INTERNAL_TEST_RAW = list(
    outer_split_67c_d[
        "internal_test"
    ]
)


assert set(
    LOCKED_67C_VALIDATION_RAW
).isdisjoint(
    LOCKED_67C_INTERNAL_TEST_RAW
)


print()
print(
    "Outer split:",
    {
        k: len(v)
        for k, v
        in outer_split_67c_d.items()
    },
)

print(
    "Validation raw IDs reconstructed:",
    len(
        LOCKED_67C_VALIDATION_RAW
    ),
)


# ============================================================
# 9 — VERIFY EXACT 67B2 LOCKED PREDICTION COHORT
#
# We reuse only predictions that were already produced by the
# frozen 67B2 pipeline.
#
# No axial GT is involved here.
# ============================================================

locked_67b2_predictions = pd.read_csv(
    LOCKED_67B2_PREDICTIONS_PATH
)


assert (
    len(
        locked_67b2_predictions
    )
    == 296
)

assert (
    "study_id_opaque"
    in locked_67b2_predictions.columns
)

assert (
    locked_67b2_predictions[
        "study_id_opaque"
    ]
    .astype(str)
    .nunique()
    == 296
)


expected_validation_opaque = set(
    opaque_id(sid)
    for sid
    in LOCKED_67C_VALIDATION_RAW
)

actual_locked_opaque = set(
    locked_67b2_predictions[
        "study_id_opaque"
    ]
    .astype(str)
)


assert (
    actual_locked_opaque
    ==
    expected_validation_opaque
), (
    "HARD STOP — 67B2 locked predictions "
    "do not match the exact outer validation cohort."
)


internal_test_opaque = set(
    opaque_id(sid)
    for sid
    in LOCKED_67C_INTERNAL_TEST_RAW
)


assert actual_locked_opaque.isdisjoint(
    internal_test_opaque
)


print()
print(
    "67B2 locked predictions cohort parity: PASS"
)


# ============================================================
# 10 — UPSTREAM FROZEN OUTPUT COUNTS
# ============================================================

print()
print("67B2 decision_status:")
print(
    locked_67b2_predictions[
        "decision_status"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)

print()

print(
    "67B2 emitted absolute sequence:",
    int(
        locked_67b2_predictions[
            "emitted_absolute_sequence"
        ]
        .astype(bool)
        .sum()
    ),
    "/ 296",
)


# ============================================================
# 11 — 67C-D STATE
#
# Prediction stage has NOT started.
# Validation axial GT remains locked.
# ============================================================

HEURISTIC_RULES_FROZEN = True

ACCESS_OUTER_VALIDATION_AXIAL_GT = False
RUN_LOCKED_VALIDATION = False

RSNA_INTERNAL_TEST_LOCKED = True
OFFICIAL_TEST_ACCESSED = False

AXIAL_PAIRING_VALIDATED_67C = False

TRAINING_PERFORMED = False


assert HEURISTIC_RULES_FROZEN is True

assert (
    ACCESS_OUTER_VALIDATION_AXIAL_GT
    is False
)

assert RUN_LOCKED_VALIDATION is False

assert RSNA_INTERNAL_TEST_LOCKED is True

assert OFFICIAL_TEST_ACCESSED is False

assert AXIAL_PAIRING_VALIDATED_67C is False

assert TRAINING_PERFORMED is False


# ============================================================
# 12 — CLOSING BLOCK
# ============================================================

print()
print("=" * 62)
print("67C-D LOCKED VALIDATION PRE-FLIGHT COMPLETE")
print("=" * 62)

print()

print(
    "GPU:",
    GPU_NAME,
)

print(
    "67C-C freeze SHA:",
    actual_hashes[
        "67C_freeze"
    ],
)

print(
    "Sagittal checkpoint SHA:",
    actual_hashes[
        "sagittal_checkpoint"
    ],
)

print(
    "67B2 validated scorer SHA:",
    actual_hashes[
        "67B2_validated_model"
    ],
)

print(
    "67B2 locked predictions SHA:",
    actual_hashes[
        "67B2_locked_predictions"
    ],
)

print()

print(
    "Locked outer validation studies:",
    len(
        LOCKED_67C_VALIDATION_RAW
    ),
)

print(
    "67B2 locked cohort parity:",
    "PASS",
)

print()

print(
    "Frozen 67C method:",
    freeze_67c[
        "pairing_method"
    ],
)

print(
    "Frozen min margin mm:",
    FROZEN_67C_MIN_MARGIN_MM,
)

print(
    "Frozen max plane distance mm:",
    FROZEN_67C_MAX_PLANE_DISTANCE_MM,
)

print()

print(
    "Outer validation axial GT accessed:",
    ACCESS_OUTER_VALIDATION_AXIAL_GT,
)

print(
    "Internal test accessed:",
    False,
)

print(
    "Official test accessed:",
    OFFICIAL_TEST_ACCESSED,
)

print()

print(
    "AXIAL_PAIRING_VALIDATED_67C:",
    AXIAL_PAIRING_VALIDATED_67C,
)

print()

print(
    "READY FOR 67C-D PREDICTION PHASE — AXIAL GT STILL LOCKED"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DEVICE: cuda
GPU: NVIDIA L4

Required artifact existence: PASS
67C_freeze 4afbe5af1cfbed3f11a302ae9905ca6a4a6818824b60d54322341d4456505f34 MATCH
sagittal_checkpoint cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944 MATCH
67B2_validated_model bb9e9d812972305cbc3dc87b96b95f78b7c9de4afc8ac4e6f071b35ae6fffd2c MATCH
67B2_locked_summary 2b74c8a5b0d8ee562f6a774e136abf95225074d32c32c7646a5ca3f0fd6e8dcb MATCH
67B2_locked_predictions baae269fe8f0e7bc7725a44ebebbb1431600d7ce23d56b7dd82c05a0cbcc0405 MATCH
series_csv bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33 MATCH

Frozen artifact identity: PASS

67C-C frozen policy loaded: PASS
Min margin: 0.26162020362301863
Max plane distance: 23.818094078633493
Gates: {'pairing_conditional_coverage_min': 0.8, 'strict_within_1_accuracy_min': 0.9, 'strict_within_2_accuracy_min': 0.97, 'series_level_

In [ ]:
# ============================================================
# 67C-D — SAFE 67B2 RUNTIME BOOTSTRAP
#
# PURPOSE:
# recover ONLY Python definitions needed for frozen sagittal
# inference from the versioned 67B2 notebook.
#
# CRITICAL:
# - does NOT execute notebook top-level dataset loading
# - does NOT load train_label_coordinates.csv
# - does NOT train
# - does NOT run 67B2 validation
# - does NOT access axial GT
#
# After this cell passes, we run the 67C-D prediction phase.
# ============================================================

import ast
import json
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# 0 — Safety
# ------------------------------------------------------------

assert torch.cuda.is_available()
assert "L4" in torch.cuda.get_device_name(0).upper()

assert ACCESS_OUTER_VALIDATION_AXIAL_GT is False
assert RUN_LOCKED_VALIDATION is False
assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False
assert AXIAL_PAIRING_VALIDATED_67C is False


print("=" * 68)
print("67C-D SAFE 67B2 RUNTIME BOOTSTRAP")
print("=" * 68)


# ------------------------------------------------------------
# 1 — Obtain versioned 67B2 notebook SOURCE only
# ------------------------------------------------------------

REPO_RAW_67B2_URL = (
    "https://raw.githubusercontent.com/"
    "EnzoAA004/"
    "PFI_MVPTest_Enzo_AImodule/"
    "research/post-e50-axial-level-pairing/"
    "notebooks/post_e50/"
    "67B2_postE50_absolute_lumbar_window_scorer.ipynb"
)

BOOTSTRAP_SOURCE_PATH = (
    PFI_DRIVE_ROOT
    / "results"
    / "post_e50"
    / "67C"
    / "67C_D_67B2_SOURCE_ONLY.ipynb"
)


if not BOOTSTRAP_SOURCE_PATH.is_file():

    urllib.request.urlretrieve(
        REPO_RAW_67B2_URL,
        BOOTSTRAP_SOURCE_PATH,
    )


nb = json.loads(
    BOOTSTRAP_SOURCE_PATH.read_text(
        encoding="utf-8"
    )
)


print(
    "67B2 source notebook loaded as JSON only."
)


# ------------------------------------------------------------
# 2 — Extract ONLY def/class/import AST nodes.
#
# Top-level assignments, pd.read_csv(), training loops,
# torch.load(), feature extraction, validation, etc. are NOT
# executed here.
# ------------------------------------------------------------

definition_nodes = []

function_names_found = set()
class_names_found = set()


for cell in nb.get("cells", []):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get(
            "source",
            [],
        )
    )

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:

        if isinstance(
            node,
            (
                ast.Import,
                ast.ImportFrom,
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef,
            ),
        ):

            definition_nodes.append(
                node
            )

            if isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                ),
            ):
                function_names_found.add(
                    node.name
                )

            elif isinstance(
                node,
                ast.ClassDef,
            ):
                class_names_found.add(
                    node.name
                )


required_source_functions = {
    "_load_and_prepare_study_once",
    "_infer_loaded_studies_ultrafast",
    "orient_and_sort_instances",
    "enumerate_candidate_windows",
}


missing_in_source = (
    required_source_functions
    -
    function_names_found
)


assert not missing_in_source, (
    "Required functions were not found in the "
    "versioned 67B2 notebook source: "
    f"{sorted(missing_in_source)}"
)


print(
    "Required function definitions found in source: PASS"
)


# ------------------------------------------------------------
# 3 — Execute definitions ONLY
# ------------------------------------------------------------

definition_module = ast.Module(
    body=definition_nodes,
    type_ignores=[],
)

ast.fix_missing_locations(
    definition_module
)


exec(
    compile(
        definition_module,
        "<67B2_definitions_only>",
        "exec",
    ),
    globals(),
)


# ------------------------------------------------------------
# 4 — Verify exact functions now exist
# ------------------------------------------------------------

required_runtime_functions = [
    "_load_and_prepare_study_once",
    "_infer_loaded_studies_ultrafast",
    "orient_and_sort_instances",
    "enumerate_candidate_windows",
]


missing_runtime = [
    name
    for name
    in required_runtime_functions
    if name not in globals()
]


assert not missing_runtime, (
    "Definitions-only bootstrap failed. "
    f"Missing: {missing_runtime}"
)


print(
    "Definitions-only bootstrap: PASS"
)


# ------------------------------------------------------------
# 5 — Explicit proof that the GT coordinate CSV has NOT
# been loaded by this bootstrap.
#
# NOTE:
# The preflight intentionally loads only series metadata.
# ------------------------------------------------------------

for forbidden_global in [
    "coords_df",
    "reference_joined",
    "reference_joined_train",
    "axial_reference_rows_train",
]:

    assert forbidden_global not in globals(), (
        "HARD STOP — GT-related dataframe unexpectedly "
        f"exists in clean runtime: {forbidden_global}"
    )


print(
    "GT-related dataframe globals absent: PASS"
)


# ------------------------------------------------------------
# 6 — Frozen artifact paths / constants required by the
# recovered functions
# ------------------------------------------------------------

CHECKPOINT_PATH = (
    PFI_DRIVE_ROOT
    / "models"
    / "final"
    / "sagittal_spider_multiclass_final_best_cf11dcc0.pt"
)

assert CHECKPOINT_PATH.is_file()

assert (
    sha256_file(CHECKPOINT_PATH)
    ==
    EXPECTED_SAGITTAL_SHA256
)


DEVICE = torch.device("cuda")

SEED = 2026

CANONICAL_LEVELS = [
    "L1-L2",
    "L2-L3",
    "L3-L4",
    "L4-L5",
    "L5-S1",
]


print(
    "Frozen sagittal checkpoint identity: PASS"
)


# ------------------------------------------------------------
# 7 — Show unresolved GLOBAL dependencies before we execute
# any patient/study inference.
#
# This is diagnostic only.
# ------------------------------------------------------------

def unresolved_function_globals(fn):

    code_names = set(
        fn.__code__.co_names
    )

    builtin_names = set(
        dir(__builtins__)
    )

    return sorted(
        name
        for name
        in code_names
        if (
            name not in globals()
            and
            name not in builtin_names
        )
    )


for fn_name in [
    "_load_and_prepare_study_once",
    "_infer_loaded_studies_ultrafast",
]:

    unresolved = (
        unresolved_function_globals(
            globals()[fn_name]
        )
    )

    print()
    print(
        fn_name,
        "unresolved globals:",
        unresolved,
    )


# ------------------------------------------------------------
# 8 — Reassert methodological locks
# ------------------------------------------------------------

ACCESS_OUTER_VALIDATION_AXIAL_GT = False
RUN_LOCKED_VALIDATION = False
RSNA_INTERNAL_TEST_LOCKED = True
OFFICIAL_TEST_ACCESSED = False
AXIAL_PAIRING_VALIDATED_67C = False


print()
print("=" * 68)
print("67C-D SAFE BOOTSTRAP AUDIT COMPLETE")
print("=" * 68)

print(
    "GPU:",
    torch.cuda.get_device_name(0),
)

print(
    "train_label_coordinates.csv loaded:",
    False,
)

print(
    "Outer validation axial GT accessed:",
    False,
)

print(
    "Internal test accessed:",
    False,
)

print(
    "Official test accessed:",
    False,
)

print()
print(
    "DO NOT RUN THE PREDICTION CELL YET."
)

print(
    "SEND ME THE TWO 'unresolved globals' LINES."
)

67C-D SAFE 67B2 RUNTIME BOOTSTRAP
67B2 source notebook loaded as JSON only.
Required function definitions found in source: PASS
Definitions-only bootstrap: PASS


AssertionError: HARD STOP — GT-related dataframe unexpectedly exists in clean runtime: coords_df

In [ ]:
# ============================================================
# 67C-D — PREDICTION PHASE
#
# LOCKED / GT-BLIND PHASE
#
# INPUT:
#   - exact 296-study outer-validation cohort
#   - frozen 67B2 sagittal pipeline
#   - frozen 67B2 selected_window_index / emission decision
#   - frozen 67C-C geometry + abstention policy
#
# OUTPUT:
#   - one immutable GT-blind prediction table
#
# DOES NOT:
#   - read train_label_coordinates.csv
#   - resolve axial GT
#   - access internal_test
#   - access official test
#   - tune thresholds
#   - retrain anything
# ============================================================

import copy
import gc
import json
import math
import os
import subprocess
import sys
import time
import urllib.request

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed,
)

from pathlib import Path

import numpy as np
import pandas as pd
import torch


# ============================================================
# 0 — HARD PRE-FLIGHT CONTRACT
# ============================================================

_required_preflight = [
    "PFI_DRIVE_ROOT",
    "RSNA_ROOT",
    "LOCKED_67C_VALIDATION_RAW",
    "locked_67b2_predictions",
    "FROZEN_67C_MIN_MARGIN_MM",
    "FROZEN_67C_MAX_PLANE_DISTANCE_MM",
    "EXPECTED_67C_FREEZE_SHA256",
    "FREEZE_67C_PATH",
    "sha256_file",
    "opaque_id",
]

_missing = [
    x
    for x in _required_preflight
    if x not in globals()
]

assert not _missing, (
    "Run the 67C-D PRE-FLIGHT cell first. "
    f"Missing: {_missing}"
)

assert torch.cuda.is_available()
assert "L4" in torch.cuda.get_device_name(0).upper()

assert (
    sha256_file(FREEZE_67C_PATH)
    ==
    EXPECTED_67C_FREEZE_SHA256
)

assert ACCESS_OUTER_VALIDATION_AXIAL_GT is False
assert RUN_LOCKED_VALIDATION is False
assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False
assert AXIAL_PAIRING_VALIDATED_67C is False
assert TRAINING_PERFORMED is False
assert HEURISTIC_RULES_FROZEN is True


DEVICE = torch.device("cuda")

CANONICAL_LEVELS_67C_D = [
    "L1-L2",
    "L2-L3",
    "L3-L4",
    "L4-L5",
    "L5-S1",
]


print("=" * 68)
print("67C-D PREDICTION PHASE — GT BLIND")
print("=" * 68)

print(
    "GPU:",
    torch.cuda.get_device_name(0),
)

print(
    "Locked studies:",
    len(LOCKED_67C_VALIDATION_RAW),
)

print(
    "Axial GT opened:",
    ACCESS_OUTER_VALIDATION_AXIAL_GT,
)


# ============================================================
# 1 — LOAD pydicom / PIL IF NECESSARY
# ============================================================

try:
    import pydicom
except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "pydicom",
        ]
    )
    import pydicom

try:
    from PIL import Image
except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "Pillow",
        ]
    )
    from PIL import Image


# ============================================================
# 2 — BOOTSTRAP THE FROZEN 67B2 INFERENCE DEFINITIONS
#
# The new L4 runtime lost Python functions from the previous
# session. We recover the CODE DEFINITIONS from the exact
# versioned 67B2 notebook.
#
# IMPORTANT:
# We DO NOT execute its TRAIN extraction, scorer training,
# DEV selection or locked-validation loop.
# ============================================================

REPO_RAW_67B2_URL = (
    "https://raw.githubusercontent.com/"
    "EnzoAA004/"
    "PFI_MVPTest_Enzo_AImodule/"
    "research/post-e50-axial-level-pairing/"
    "notebooks/post_e50/"
    "67B2_postE50_absolute_lumbar_window_scorer.ipynb"
)


BOOTSTRAP_CACHE_PATH = (
    PFI_DRIVE_ROOT
    / "results"
    / "post_e50"
    / "67C"
    / "67C_D_67B2_source_bootstrap.ipynb"
)


if not BOOTSTRAP_CACHE_PATH.is_file():

    print()
    print(
        "Downloading versioned 67B2 notebook source..."
    )

    urllib.request.urlretrieve(
        REPO_RAW_67B2_URL,
        BOOTSTRAP_CACHE_PATH,
    )


nb67b2 = json.loads(
    BOOTSTRAP_CACHE_PATH.read_text(
        encoding="utf-8"
    )
)


# ------------------------------------------------------------
# We execute ONLY the early setup / scientific definition cells.
#
# HARD STOP before:
#   feature extraction / scorer split / training / freeze /
#   locked validation.
# ------------------------------------------------------------

STOP_MARKERS = [
    "ULTRA-FAST FEATURE EXTRACTION",
    "Build deterministic scorer TRAIN / DEV split",
    "67B2 TRAINING",
    "FREEZE 67B2 BEFORE LOCKED VALIDATION",
    "67B2 LOCKED VALIDATION",
]


def _cell_text(cell):
    return "".join(
        cell.get(
            "source",
            [],
        )
    )


bootstrap_cells = []


for cell in nb67b2["cells"]:

    if cell.get("cell_type") != "code":
        continue

    src = _cell_text(cell)

    if any(
        marker in src
        for marker
        in STOP_MARKERS
    ):
        break

    bootstrap_cells.append(src)


assert len(bootstrap_cells) > 0


# Prevent the bootstrapped notebook from thinking that its
# original outer validation should be executed.
RUN_LOCKED_VALIDATION = False


_bootstrap_namespace = globals()


for idx, src in enumerate(
    bootstrap_cells
):

    # --------------------------------------------------------
    # Skip top-level TRAIN execution blocks if encountered.
    #
    # We want definitions / checkpoint initialization only.
    # --------------------------------------------------------

    upper = src.upper()

    skip_markers = [
        "FEATURE EXTRACTION",
        "MAX_STUDIES",
        "ELIGIBLE_TRAIN_RAW",
        "FOR GROUP_START IN RANGE",
    ]

    if (
        "def " not in src
        and
        "class " not in src
        and
        any(
            marker in upper
            for marker
            in skip_markers
        )
    ):
        continue

    try:

        exec(
            compile(
                src,
                f"<67B2_bootstrap_cell_{idx}>",
                "exec",
            ),
            _bootstrap_namespace,
        )

    except AssertionError as exc:

        # Some notebook setup assertions concern its historical
        # TRAIN-stage flags. Do not silently ignore unknown
        # execution failures.
        text = str(exc)

        if text:
            raise

    except Exception as exc:

        # If a cell is purely an already-produced reporting /
        # artifact cell, it may expect TRAIN variables that are
        # intentionally absent. Ignore ONLY known missing-state
        # NameErrors here.
        if isinstance(
            exc,
            NameError,
        ):
            continue

        raise


# ============================================================
# 3 — VERIFY REQUIRED FROZEN INFERENCE FUNCTIONS EXIST
# ============================================================

_REQUIRED_67B2_RUNTIME = [
    "_load_and_prepare_study_once",
    "_infer_loaded_studies_ultrafast",
    "orient_and_sort_instances",
    "enumerate_candidate_windows",
    "project_points_onto_axis",
]


_missing_runtime = [
    x
    for x in _REQUIRED_67B2_RUNTIME
    if x not in globals()
]


assert not _missing_runtime, (
    "67B2 inference bootstrap incomplete. "
    f"Missing functions: {_missing_runtime}"
)


print()
print(
    "67B2 frozen inference runtime bootstrap: PASS"
)


# ============================================================
# 4 — REASSERT 67C-D CONTRACT AFTER BOOTSTRAP
#
# 67B2 code may define similarly named stage variables.
# Restore the authoritative 67C-D lock state.
# ============================================================

ACCESS_OUTER_VALIDATION_AXIAL_GT = False
RUN_LOCKED_VALIDATION = False

RSNA_INTERNAL_TEST_LOCKED = True
OFFICIAL_TEST_ACCESSED = False

AXIAL_PAIRING_VALIDATED_67C = False
TRAINING_PERFORMED = False
HEURISTIC_RULES_FROZEN = True


assert ACCESS_OUTER_VALIDATION_AXIAL_GT is False
assert RUN_LOCKED_VALIDATION is False
assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False
assert AXIAL_PAIRING_VALIDATED_67C is False


# ============================================================
# 5 — AXIAL SERIES METADATA
#
# ONLY train_series_descriptions.csv.
# No coordinate / GT file is loaded here.
# ============================================================

series_metadata_67c_d = pd.read_csv(
    RSNA_ROOT
    / "train_series_descriptions.csv",

    dtype={
        "study_id": "string",
        "series_id": "string",
    },
)


def _is_axial_description(value):
    return (
        "AXIAL"
        in str(value)
        .strip()
        .upper()
    )


axial_series_desc_67c_d = sorted(
    x
    for x
    in series_metadata_67c_d[
        "series_description"
    ]
    .dropna()
    .unique()
    if _is_axial_description(x)
)


assert len(
    axial_series_desc_67c_d
) >= 1


print(
    "Observed axial descriptions:",
    axial_series_desc_67c_d,
)


# ============================================================
# 6 — PURE DICOM GEOMETRY FUNCTIONS FOR AXIAL PAIRING
#
# Same 67C-B mathematics.
# No GT.
# ============================================================

def _check_orientation_valid(iop):

    if (
        iop is None
        or len(iop) != 6
    ):
        return False

    row = np.asarray(
        iop[:3],
        dtype=np.float64,
    )

    col = np.asarray(
        iop[3:],
        dtype=np.float64,
    )

    return bool(
        abs(
            np.linalg.norm(row)
            - 1.0
        ) < 1e-2
        and
        abs(
            np.linalg.norm(col)
            - 1.0
        ) < 1e-2
        and
        abs(
            float(
                np.dot(
                    row,
                    col,
                )
            )
        ) < 1e-2
    )


def _read_axial_geometry_67c_d(
    dcm_path,
):

    try:

        ds = pydicom.dcmread(
            str(dcm_path),
            stop_before_pixels=True,
        )

    except Exception:

        return None


    required = [
        "Rows",
        "Columns",
        "PixelSpacing",
        "ImagePositionPatient",
        "ImageOrientationPatient",
        "InstanceNumber",
    ]


    if not all(
        hasattr(
            ds,
            tag,
        )
        for tag
        in required
    ):
        return None


    try:

        rows = int(
            ds.Rows
        )

        columns = int(
            ds.Columns
        )

        spacing = tuple(
            float(x)
            for x
            in ds.PixelSpacing
        )

        ipp = np.asarray(
            [
                float(x)
                for x
                in ds.ImagePositionPatient
            ],
            dtype=np.float64,
        )

        iop = tuple(
            float(x)
            for x
            in ds.ImageOrientationPatient
        )

        instance = int(
            ds.InstanceNumber
        )

    except Exception:

        return None


    if (
        rows <= 0
        or columns <= 0
        or len(spacing) != 2
        or min(spacing) <= 0
        or not _check_orientation_valid(iop)
    ):
        return None


    row_cosines = np.asarray(
        iop[:3],
        dtype=np.float64,
    )

    column_cosines = np.asarray(
        iop[3:],
        dtype=np.float64,
    )

    normal = np.cross(
        row_cosines,
        column_cosines,
    )

    norm = float(
        np.linalg.norm(
            normal
        )
    )

    if (
        norm <= 1e-9
        or not np.all(
            np.isfinite(
                normal
            )
        )
    ):
        return None


    normal = normal / norm


    return {
        "instance_number":
            instance,

        "rows":
            rows,

        "columns":
            columns,

        "pixel_spacing":
            spacing,

        "ipp":
            ipp,

        "row_cosines":
            row_cosines,

        "column_cosines":
            column_cosines,

        "normal":
            normal,
    }


def _project_xyz_to_axial_slice_67c_d(
    point_xyz,
    geom,
):

    point_xyz = np.asarray(
        point_xyz,
        dtype=np.float64,
    )

    O = geom["ipp"]
    n = geom["normal"]

    signed = float(
        np.dot(
            point_xyz - O,
            n,
        )
    )

    distance = abs(
        signed
    )

    projected = (
        point_xyz
        -
        signed * n
    )

    diff = projected - O

    row_spacing, col_spacing = (
        geom[
            "pixel_spacing"
        ]
    )

    projected_col = float(
        np.dot(
            diff,
            geom[
                "row_cosines"
            ],
        )
        /
        col_spacing
    )

    projected_row = float(
        np.dot(
            diff,
            geom[
                "column_cosines"
            ],
        )
        /
        row_spacing
    )

    inside = bool(
        0
        <= projected_col
        < geom["columns"]
        and
        0
        <= projected_row
        < geom["rows"]
    )


    return {
        "plane_distance_mm":
            distance,

        "signed_plane_distance_mm":
            signed,

        "projected_row_px":
            projected_row,

        "projected_col_px":
            projected_col,

        "inside_fov":
            inside,
    }


# ============================================================
# 7 — PARALLEL AXIAL GEOMETRY CACHE
# ============================================================

AXIAL_IO_WORKERS_67C_D = 16

_axial_geom_cache_67c_d = {}


def _axial_dicom_paths_for_study_67c_d(
    sid,
):

    rows = series_metadata_67c_d[
        (
            series_metadata_67c_d[
                "study_id"
            ].astype(str)
            ==
            str(sid)
        )
        &
        (
            series_metadata_67c_d[
                "series_description"
            ]
            .isin(
                axial_series_desc_67c_d
            )
        )
    ]


    records = []


    for _, r in rows.iterrows():

        series_id = str(
            r[
                "series_id"
            ]
        )

        series_dir = (
            RSNA_ROOT
            / "train_images"
            / str(sid)
            / series_id
        )

        if not series_dir.is_dir():
            continue


        for path in sorted(
            series_dir.glob(
                "*.dcm"
            )
        ):

            records.append(
                (
                    series_id,
                    str(
                        r[
                            "series_description"
                        ]
                    ),
                    path,
                )
            )


    return records


def _preload_axial_geometry_67c_d(
    sid,
):

    records = (
        _axial_dicom_paths_for_study_67c_d(
            sid
        )
    )


    missing = [
        rec
        for rec
        in records
        if str(
            rec[2]
        )
        not in _axial_geom_cache_67c_d
    ]


    if missing:

        with ThreadPoolExecutor(
            max_workers=AXIAL_IO_WORKERS_67C_D
        ) as pool:

            futures = {
                pool.submit(
                    _read_axial_geometry_67c_d,
                    path,
                ):
                    (
                        series_id,
                        desc,
                        path,
                    )

                for (
                    series_id,
                    desc,
                    path,
                )
                in missing
            }


            for future in as_completed(
                futures
            ):

                (
                    series_id,
                    desc,
                    path,
                ) = futures[
                    future
                ]

                geom = future.result()

                _axial_geom_cache_67c_d[
                    str(path)
                ] = (
                    series_id,
                    desc,
                    geom,
                )


    result = []


    for (
        series_id,
        desc,
        path,
    ) in records:

        cached = (
            _axial_geom_cache_67c_d.get(
                str(path)
            )
        )

        if (
            cached is not None
            and
            cached[2] is not None
        ):

            result.append(
                (
                    series_id,
                    desc,
                    path,
                    cached[2],
                )
            )


    return result


# ============================================================
# 8 — FROZEN 67C POLICY
# ============================================================

def _apply_frozen_pairing_67c_d(
    candidates,
):

    if not candidates:

        return {
            "emit": False,
            "reason": "NO_CANDIDATES",
            "winner": None,
            "margin_mm": None,
        }


    ranked = sorted(
        candidates,
        key=lambda c: (
            not bool(
                c[
                    "inside_fov"
                ]
            ),
            float(
                c[
                    "plane_distance_mm"
                ]
            ),
        ),
    )


    winner = ranked[0]


    if not bool(
        winner[
            "inside_fov"
        ]
    ):

        return {
            "emit": False,
            "reason": "OUTSIDE_FOV",
            "winner": winner,
            "margin_mm": None,
        }


    if (
        float(
            winner[
                "plane_distance_mm"
            ]
        )
        >
        FROZEN_67C_MAX_PLANE_DISTANCE_MM
    ):

        return {
            "emit": False,
            "reason": "PLANE_DISTANCE",
            "winner": winner,
            "margin_mm": None,
        }


    if len(
        ranked
    ) < 2:

        return {
            "emit": False,
            "reason": "NO_SECOND_CANDIDATE",
            "winner": winner,
            "margin_mm": None,
        }


    margin = (
        float(
            ranked[1][
                "plane_distance_mm"
            ]
        )
        -
        float(
            ranked[0][
                "plane_distance_mm"
            ]
        )
    )


    if (
        margin
        <
        FROZEN_67C_MIN_MARGIN_MM
    ):

        return {
            "emit": False,
            "reason": "AMBIGUOUS_MARGIN",
            "winner": winner,
            "margin_mm": margin,
        }


    return {
        "emit": True,
        "reason": "PASS",
        "winner": winner,
        "margin_mm": margin,
    }


# ============================================================
# 9 — LOCKED 67B2 LOOKUP
# ============================================================

locked_67b2_by_opaque = {
    str(r["study_id_opaque"]):
        r

    for _, r
    in locked_67b2_predictions.iterrows()
}


assert len(
    locked_67b2_by_opaque
) == 296


# ============================================================
# 10 — OUTPUT / RESUME CACHE
#
# This file contains NO GT columns.
# ============================================================

PREDICTIONS_DIR_67C_D = (
    RESULTS_67C_DIR
    / "locked_validation"
)

PREDICTIONS_DIR_67C_D.mkdir(
    parents=True,
    exist_ok=True,
)


PREDICTIONS_PATH_67C_D = (
    PREDICTIONS_DIR_67C_D
    / "67C_D_predictions_GT_BLIND.csv"
)


if PREDICTIONS_PATH_67C_D.is_file():

    prediction_df_67c_d = (
        pd.read_csv(
            PREDICTIONS_PATH_67C_D
        )
    )

else:

    prediction_df_67c_d = (
        pd.DataFrame()
    )


if (
    len(
        prediction_df_67c_d
    )
    and
    "study_id_opaque"
    in prediction_df_67c_d.columns
):

    processed_opaque_67c_d = set(
        prediction_df_67c_d[
            "study_id_opaque"
        ]
        .astype(str)
        .unique()
    )

else:

    processed_opaque_67c_d = set()


remaining_67c_d = [
    sid
    for sid
    in LOCKED_67C_VALIDATION_RAW
    if opaque_id(
        sid
    )
    not in processed_opaque_67c_d
]


print()
print(
    "Already processed:",
    len(
        processed_opaque_67c_d
    ),
)

print(
    "Remaining:",
    len(
        remaining_67c_d
    ),
)


# ============================================================
# 11 — FROZEN UPSTREAM -> XYZ -> AXIAL PAIRING
# ============================================================

GROUP_SIZE_67C_D = 16
LOAD_WORKERS_67C_D = 8


t0_67c_d = time.time()


for group_start in range(
    0,
    len(
        remaining_67c_d
    ),
    GROUP_SIZE_67C_D,
):

    group_ids = (
        remaining_67c_d[
            group_start:
            group_start
            + GROUP_SIZE_67C_D
        ]
    )


    # --------------------------------------------------------
    # 11A — Load sagittal study once
    # --------------------------------------------------------

    with ThreadPoolExecutor(
        max_workers=LOAD_WORKERS_67C_D
    ) as pool:

        futures = {
            pool.submit(
                _load_and_prepare_study_once,
                sid,
            ):
                sid

            for sid
            in group_ids
        }


        loaded_group = []


        for future in as_completed(
            futures
        ):

            sid = futures[
                future
            ]

            try:

                loaded_group.append(
                    future.result()
                )

            except Exception as exc:

                loaded_group.append(
                    {
                        "study_id_raw":
                            str(sid),

                        "status":
                            (
                                "LOAD_ERROR:"
                                f"{type(exc).__name__}"
                            ),
                    }
                )


    loaded_map = {
        str(
            item[
                "study_id_raw"
            ]
        ):
            item

        for item
        in loaded_group
    }


    loaded_group = [
        loaded_map[
            str(sid)
        ]
        for sid
        in group_ids
    ]


    # --------------------------------------------------------
    # 11B — Frozen sagittal inference
    # --------------------------------------------------------

    group_inference = (
        _infer_loaded_studies_ultrafast(
            loaded_group
        )
    )


    group_output_rows = []


    for sid_raw in group_ids:

        sid = str(
            sid_raw
        )

        opaque = opaque_id(
            sid
        )

        upstream_locked = (
            locked_67b2_by_opaque[
                opaque
            ]
        )

        upstream_emit = bool(
            upstream_locked[
                "emitted_absolute_sequence"
            ]
        )


        base_record = {
            "study_id_opaque":
                opaque,

            "upstream_67b2_emitted":
                upstream_emit,

            "upstream_67b2_decision_status":
                upstream_locked.get(
                    "decision_status"
                ),

            "upstream_67b2_selected_window_index":
                upstream_locked.get(
                    "selected_window_index"
                ),
        }


        # ----------------------------------------------------
        # Upstream abstention:
        # no 67C pairing should occur.
        # ----------------------------------------------------

        if not upstream_emit:

            for level in (
                CANONICAL_LEVELS_67C_D
            ):

                group_output_rows.append(
                    {
                        **base_record,

                        "level":
                            level,

                        "upstream_xyz_available":
                            False,

                        "pairing_emit":
                            False,

                        "pairing_reason":
                            "UPSTREAM_67B2_ABSTAIN",

                        "selected_series_id_opaque":
                            None,

                        "selected_instance_number":
                            None,

                        "selected_plane_distance_mm":
                            None,

                        "selected_inside_fov":
                            None,

                        "distance_margin_mm":
                            None,

                        "projected_row_px":
                            None,

                        "projected_col_px":
                            None,

                        "predicted_disc_x_mm":
                            None,

                        "predicted_disc_y_mm":
                            None,

                        "predicted_disc_z_mm":
                            None,
                    }
                )

            continue


        run = group_inference.get(
            sid,
            {}
        )


        if (
            run.get(
                "status"
            )
            != "EXECUTED"
        ):

            for level in (
                CANONICAL_LEVELS_67C_D
            ):

                group_output_rows.append(
                    {
                        **base_record,

                        "level":
                            level,

                        "upstream_xyz_available":
                            False,

                        "pairing_emit":
                            False,

                        "pairing_reason":
                            "SAGITTAL_REINFERENCE_FAILED",

                        "selected_series_id_opaque":
                            None,

                        "selected_instance_number":
                            None,

                        "selected_plane_distance_mm":
                            None,

                        "selected_inside_fov":
                            None,

                        "distance_margin_mm":
                            None,

                        "projected_row_px":
                            None,

                        "projected_col_px":
                            None,

                        "predicted_disc_x_mm":
                            None,

                        "predicted_disc_y_mm":
                            None,

                        "predicted_disc_z_mm":
                            None,
                    }
                )

            continue


        # ----------------------------------------------------
        # Orient predicted disc instances exactly as 67B2.
        # ----------------------------------------------------

        (
            direction,
            ordered_instances,
        ) = orient_and_sort_instances(
            run
        )


        if (
            direction.get(
                "status"
            )
            != "RESOLVED"
        ):

            for level in (
                CANONICAL_LEVELS_67C_D
            ):

                group_output_rows.append(
                    {
                        **base_record,

                        "level":
                            level,

                        "upstream_xyz_available":
                            False,

                        "pairing_emit":
                            False,

                        "pairing_reason":
                            "SAGITTAL_DIRECTION_UNRESOLVED",

                        "selected_series_id_opaque":
                            None,

                        "selected_instance_number":
                            None,

                        "selected_plane_distance_mm":
                            None,

                        "selected_inside_fov":
                            None,

                        "distance_margin_mm":
                            None,

                        "projected_row_px":
                            None,

                        "projected_col_px":
                            None,

                        "predicted_disc_x_mm":
                            None,

                        "predicted_disc_y_mm":
                            None,

                        "predicted_disc_z_mm":
                            None,
                    }
                )

            continue


        # ----------------------------------------------------
        # Frozen 67B2 selected window from the ORIGINAL locked
        # validation output. We DO NOT rescore anything here.
        # ----------------------------------------------------

        selected_window_raw = (
            upstream_locked.get(
                "selected_window_index"
            )
        )


        if pd.isna(
            selected_window_raw
        ):

            selected_window_index = None

        else:

            selected_window_index = int(
                selected_window_raw
            )


        windows = enumerate_candidate_windows(
            len(
                ordered_instances
            )
        )


        selected_window = next(
            (
                w
                for w
                in windows
                if int(
                    w[
                        "window_index"
                    ]
                )
                ==
                selected_window_index
            ),
            None,
        )


        assert selected_window is not None, (
            "HARD STOP — frozen 67B2 selected window "
            f"not reproducible for study {opaque}. "
            f"selected={selected_window_index}, "
            f"n_instances={len(ordered_instances)}"
        )


        # ----------------------------------------------------
        # 67B2 candidate-window ranks are 1-based.
        # Selected window contains exactly five ordered
        # sagittal predicted instances.
        # ----------------------------------------------------

        selected_ranks = list(
            selected_window[
                "ranks"
            ]
        )


        assert (
            len(
                selected_ranks
            )
            == 5
        )


        selected_instances = [
            ordered_instances[
                int(rank) - 1
            ]
            for rank
            in selected_ranks
        ]


        predicted_xyz_by_level = {}


        for level, instance in zip(
            CANONICAL_LEVELS_67C_D,
            selected_instances,
        ):

            xyz = np.asarray(
                instance[
                    "centroid_xyz"
                ],
                dtype=np.float64,
            )

            assert (
                xyz.shape
                ==
                (3,)
            )

            assert np.all(
                np.isfinite(
                    xyz
                )
            )

            predicted_xyz_by_level[
                level
            ] = xyz


        # ----------------------------------------------------
        # Load axial geometry ONCE for this study.
        # ----------------------------------------------------

        axial_geometries = (
            _preload_axial_geometry_67c_d(
                sid
            )
        )


        # ----------------------------------------------------
        # Frozen 67C-C pairing, independently by level.
        #
        # NO axial GT.
        # ----------------------------------------------------

        for level in (
            CANONICAL_LEVELS_67C_D
        ):

            P = predicted_xyz_by_level[
                level
            ]


            candidates = []


            for (
                series_id_raw,
                series_desc,
                dcm_path,
                geom,
            ) in axial_geometries:

                proj = (
                    _project_xyz_to_axial_slice_67c_d(
                        P,
                        geom,
                    )
                )


                candidates.append(
                    {
                        "series_id_opaque":
                            opaque_id(
                                series_id_raw
                            ),

                        "series_description":
                            series_desc,

                        "instance_number":
                            int(
                                geom[
                                    "instance_number"
                                ]
                            ),

                        "plane_distance_mm":
                            float(
                                proj[
                                    "plane_distance_mm"
                                ]
                            ),

                        "inside_fov":
                            bool(
                                proj[
                                    "inside_fov"
                                ]
                            ),

                        "projected_row_px":
                            float(
                                proj[
                                    "projected_row_px"
                                ]
                            ),

                        "projected_col_px":
                            float(
                                proj[
                                    "projected_col_px"
                                ]
                            ),
                    }
                )


            decision = (
                _apply_frozen_pairing_67c_d(
                    candidates
                )
            )


            winner = decision[
                "winner"
            ]


            group_output_rows.append(
                {
                    **base_record,

                    "level":
                        level,

                    "upstream_xyz_available":
                        True,

                    "pairing_emit":
                        bool(
                            decision[
                                "emit"
                            ]
                        ),

                    "pairing_reason":
                        decision[
                            "reason"
                        ],

                    "selected_series_id_opaque":
                        (
                            winner[
                                "series_id_opaque"
                            ]
                            if winner is not None
                            else None
                        ),

                    "selected_instance_number":
                        (
                            winner[
                                "instance_number"
                            ]
                            if winner is not None
                            else None
                        ),

                    "selected_plane_distance_mm":
                        (
                            winner[
                                "plane_distance_mm"
                            ]
                            if winner is not None
                            else None
                        ),

                    "selected_inside_fov":
                        (
                            winner[
                                "inside_fov"
                            ]
                            if winner is not None
                            else None
                        ),

                    "distance_margin_mm":
                        decision[
                            "margin_mm"
                        ],

                    "projected_row_px":
                        (
                            winner[
                                "projected_row_px"
                            ]
                            if winner is not None
                            else None
                        ),

                    "projected_col_px":
                        (
                            winner[
                                "projected_col_px"
                            ]
                            if winner is not None
                            else None
                        ),

                    "predicted_disc_x_mm":
                        float(
                            P[0]
                        ),

                    "predicted_disc_y_mm":
                        float(
                            P[1]
                        ),

                    "predicted_disc_z_mm":
                        float(
                            P[2]
                        ),
                }
            )


    # --------------------------------------------------------
    # 11C — Incremental immutable-style cache
    # --------------------------------------------------------

    new_df = pd.DataFrame(
        group_output_rows
    )


    prediction_df_67c_d = pd.concat(
        [
            prediction_df_67c_d,
            new_df,
        ],
        ignore_index=True,
    )


    prediction_df_67c_d = (
        prediction_df_67c_d
        .drop_duplicates(
            subset=[
                "study_id_opaque",
                "level",
            ],
            keep="last",
        )
        .sort_values(
            [
                "study_id_opaque",
                "level",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    prediction_df_67c_d.to_csv(
        PREDICTIONS_PATH_67C_D,
        index=False,
    )


    del loaded_group
    del loaded_map
    del group_inference

    gc.collect()

    torch.cuda.empty_cache()


    elapsed = (
        time.time()
        -
        t0_67c_d
    )


    studies_done = (
        prediction_df_67c_d[
            "study_id_opaque"
        ]
        .astype(str)
        .nunique()
    )


    print(
        f"67C-D prediction "
        f"{studies_done}/296 studies"
        f" | rows={len(prediction_df_67c_d)}"
        f" | elapsed={elapsed/60:.1f} min"
    )


# ============================================================
# 12 — FINAL GT-BLIND INTEGRITY AUDIT
# ============================================================

assert (
    prediction_df_67c_d[
        "study_id_opaque"
    ]
    .astype(str)
    .nunique()
    == 296
)

assert (
    len(
        prediction_df_67c_d
    )
    ==
    296
    *
    len(
        CANONICAL_LEVELS_67C_D
    )
)

assert set(
    prediction_df_67c_d[
        "study_id_opaque"
    ]
    .astype(str)
) == set(
    opaque_id(
        sid
    )
    for sid
    in LOCKED_67C_VALIDATION_RAW
)


# No GT columns are permitted in frozen predictions.
for forbidden in [
    "gt_",
    "ground_truth",
    "exact_slice",
    "within_1",
    "within_2",
    "correct",
]:

    assert not any(
        forbidden
        in str(col).lower()
        for col
        in prediction_df_67c_d.columns
    ), (
        f"GT-like column leaked into prediction artifact: "
        f"{forbidden}"
    )


# ============================================================
# 13 — FREEZE PREDICTION ARTIFACT
# ============================================================

PREDICTIONS_SHA256_67C_D = (
    sha256_file(
        PREDICTIONS_PATH_67C_D
    )
)


prediction_manifest_67c_d = {

    "stage":
        "67C-D_GT_BLIND_PREDICTION",

    "validation_studies":
        296,

    "prediction_rows":
        int(
            len(
                prediction_df_67c_d
            )
        ),

    "levels_per_study":
        5,

    "source_67b2_locked_predictions_sha256":
        EXPECTED_67B2_LOCKED_PREDICTIONS_SHA256,

    "source_67c_c_freeze_sha256":
        EXPECTED_67C_FREEZE_SHA256,

    "frozen_min_margin_mm":
        FROZEN_67C_MIN_MARGIN_MM,

    "frozen_max_plane_distance_mm":
        FROZEN_67C_MAX_PLANE_DISTANCE_MM,

    "predictions_path":
        str(
            PREDICTIONS_PATH_67C_D
        ),

    "predictions_sha256":
        PREDICTIONS_SHA256_67C_D,

    "outer_validation_axial_gt_accessed":
        False,

    "internal_test_accessed":
        False,

    "official_test_accessed":
        False,

    "training_performed":
        False,

    "post_prediction_tuning_allowed":
        False,
}


PREDICTION_MANIFEST_PATH_67C_D = (
    METRICS_67C_DIR
    / "67C_D_GT_blind_prediction_manifest.json"
)


PREDICTION_MANIFEST_PATH_67C_D.write_text(
    json.dumps(
        prediction_manifest_67c_d,
        indent=2,
    ),
    encoding="utf-8",
)


PREDICTION_MANIFEST_SHA256_67C_D = (
    sha256_file(
        PREDICTION_MANIFEST_PATH_67C_D
    )
)


# ============================================================
# 14 — SUMMARY — STILL NO GT
# ============================================================

upstream_emitted_studies = (
    prediction_df_67c_d[
        prediction_df_67c_d[
            "upstream_67b2_emitted"
        ].astype(bool)
    ][
        "study_id_opaque"
    ]
    .astype(str)
    .nunique()
)


pairing_evaluable_rows = (
    prediction_df_67c_d[
        prediction_df_67c_d[
            "upstream_xyz_available"
        ].astype(bool)
    ]
)


pairing_emitted_rows = (
    pairing_evaluable_rows[
        pairing_evaluable_rows[
            "pairing_emit"
        ].astype(bool)
    ]
)


pairing_conditional_coverage_pre_gt = (
    len(
        pairing_emitted_rows
    )
    /
    max(
        len(
            pairing_evaluable_rows
        ),
        1,
    )
)


full_pipeline_emitted_rows = (
    prediction_df_67c_d[
        prediction_df_67c_d[
            "pairing_emit"
        ].astype(bool)
    ]
)


full_pipeline_raw_level_coverage = (
    len(
        full_pipeline_emitted_rows
    )
    /
    (
        296
        *
        5
    )
)


print()
print("=" * 68)
print("67C-D GT-BLIND PREDICTION PHASE COMPLETE")
print("=" * 68)

print()

print(
    "Validation studies:",
    296,
)

print(
    "Prediction rows:",
    len(
        prediction_df_67c_d
    ),
)

print(
    "Upstream 67B2 emitted studies:",
    upstream_emitted_studies,
    "/ 296",
)

print()

print(
    "Pairing-evaluable levels:",
    len(
        pairing_evaluable_rows
    ),
)

print(
    "Pairing-emitted levels:",
    len(
        pairing_emitted_rows
    ),
)

print(
    "GT-blind pairing conditional coverage:",
    pairing_conditional_coverage_pre_gt,
)

print(
    "GT-blind full-pipeline raw level coverage:",
    full_pipeline_raw_level_coverage,
)

print()

print(
    "Pairing reasons:"
)

print(
    prediction_df_67c_d[
        "pairing_reason"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)

print()

print(
    "Predictions:",
    PREDICTIONS_PATH_67C_D,
)

print(
    "Predictions SHA256:",
    PREDICTIONS_SHA256_67C_D,
)

print()

print(
    "Manifest:",
    PREDICTION_MANIFEST_PATH_67C_D,
)

print(
    "Manifest SHA256:",
    PREDICTION_MANIFEST_SHA256_67C_D,
)

print()

print(
    "Outer validation axial GT accessed:",
    ACCESS_OUTER_VALIDATION_AXIAL_GT,
)

print(
    "Internal test accessed:",
    False,
)

print(
    "Official test accessed:",
    OFFICIAL_TEST_ACCESSED,
)

print()

print(
    "AXIAL_PAIRING_VALIDATED_67C:",
    AXIAL_PAIRING_VALIDATED_67C,
)

print()

print(
    "PREDICTIONS ARE NOW FROZEN."
)

print(
    "NEXT: OPEN AXIAL GT POST-HOC AND EVALUATE — NO TUNING."
)

67C-D PREDICTION PHASE — GT BLIND
GPU: NVIDIA L4
Locked studies: 296
Axial GT opened: False

{'torch': '2.11.0+cu128', 'SimpleITK': '2.5.6', 'scipy': '1.16.3', 'pydicom': '3.0.2', 'PIL': '11.3.0'}
torch: 2.11.0+cu128
SimpleITK: 2.5.6
scipy: 1.16.3
pydicom: 3.0.2
DEVICE: cuda
GPU: NVIDIA L4
GPU memory GB: 22.03
train.csv f0c9e06486bcddcd83b1ee1d95ab9db8e028d7c3c9fcbada950f0b8e4d828528 MATCH
train_label_coordinates.csv 416fb434b5bdbf69814210d03dd791dff5ef9a010bac004f5f68e8b79e852635 MATCH
train_series_descriptions.csv bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33 MATCH
shapes: (1975, 26) (48692, 7) (6294, 3)
{'train': 1382, 'validation': 296, 'internal_test': 297}
Outer split leakage audit: PASS
Geometry / preprocessing helpers: READY
67B1 orientation rule: READY
Frozen study inference: READY
Target + feature pipeline: READY
DEVICE: cuda
GPU: NVIDIA L4
INFERENCE_BATCH_SIZE: 32
STAGE_SERIES_LOCALLY: True
Legacy inference preserved.
Optimized batched inference defined.


AssertionError: 67B2 inference bootstrap incomplete. Missing functions: ['_load_and_prepare_study_once', '_infer_loaded_studies_ultrafast']

In [ ]:
# ============================================================
# 67C-D — LOCKED E2E PREDICTION PHASE
#
# INPUT:
#   - exact 296-study validation cohort from PRE-FLIGHT
#   - frozen 67B2 decisions already stored on disk
#   - frozen sagittal checkpoint
#
# OUTPUT:
#   - immutable GT-FREE axial pairing predictions
#
# IMPORTANT:
#   - DOES NOT read train_label_coordinates.csv
#   - DOES NOT open axial validation GT
#   - DOES NOT access internal_test
#   - DOES NOT access official test
#   - DOES NOT retrain / rerank / retune anything
#
# 67B2 selected_window_index is REUSED AS-IS.
# 67C-C method/thresholds are REUSED AS-IS.
# ============================================================

import gc
import hashlib
import json
import math
import os
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Mapping

import numpy as np
import pandas as pd
import torch
import torch.nn as nn


# ============================================================
# 0 — HARD PRE-FLIGHT CONTRACT
# ============================================================

_required_preflight = [
    "PFI_DRIVE_ROOT",
    "RSNA_ROOT",
    "SAGITTAL_MODEL_PATH",
    "LOCKED_67B2_PREDICTIONS_PATH",
    "FREEZE_67C_PATH",
    "LOCKED_67C_VALIDATION_RAW",
    "locked_67b2_predictions",
    "series_df_67c_d",
    "FROZEN_67C_MIN_MARGIN_MM",
    "FROZEN_67C_MAX_PLANE_DISTANCE_MM",
    "EXPECTED_67C_FREEZE_SHA256",
    "EXPECTED_SAGITTAL_SHA256",
    "EXPECTED_67B2_LOCKED_PREDICTIONS_SHA256",
    "sha256_file",
    "opaque_id",
]

_missing = [
    x for x in _required_preflight
    if x not in globals()
]

assert not _missing, (
    "Run 67C-D PRE-FLIGHT first. "
    f"Missing globals: {_missing}"
)

assert torch.cuda.is_available()
assert "L4" in torch.cuda.get_device_name(0).upper()

assert HEURISTIC_RULES_FROZEN is True
assert ACCESS_OUTER_VALIDATION_AXIAL_GT is False
assert RUN_LOCKED_VALIDATION is False
assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False
assert AXIAL_PAIRING_VALIDATED_67C is False
assert TRAINING_PERFORMED is False

assert (
    sha256_file(FREEZE_67C_PATH)
    == EXPECTED_67C_FREEZE_SHA256
)

assert (
    sha256_file(SAGITTAL_MODEL_PATH)
    == EXPECTED_SAGITTAL_SHA256
)

assert (
    sha256_file(
        LOCKED_67B2_PREDICTIONS_PATH
    )
    ==
    EXPECTED_67B2_LOCKED_PREDICTIONS_SHA256
)

# Critical methodological guard:
# THIS FILE MUST NOT BE OPENED IN THIS PHASE.
AXIAL_GT_FILE_67C_D = (
    RSNA_ROOT
    / "train_label_coordinates.csv"
)

DEVICE_67C_D = torch.device("cuda")

CANONICAL_LEVELS_67C_D = [
    "L1-L2",
    "L2-L3",
    "L3-L4",
    "L4-L5",
    "L5-S1",
]

TRAIN_IMAGES_ROOT_67C_D = (
    RSNA_ROOT
    / "train_images"
)

print("=" * 62)
print("67C-D PREDICTION PHASE — HARD CONTRACT")
print("=" * 62)
print("GPU:", torch.cuda.get_device_name(0))
print("Validation studies:", len(LOCKED_67C_VALIDATION_RAW))
print("Axial GT file will NOT be read:", AXIAL_GT_FILE_67C_D)
print("Frozen 67B2 decisions: READ ONLY")
print("Frozen 67C-C rule: READ ONLY")


# ============================================================
# 1 — INSTALL MISSING SAFE DEPENDENCIES
# ============================================================

def _ensure_import(
    module_name,
    package_name,
):
    try:
        return __import__(module_name)
    except ImportError:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            package_name,
        ])
        return __import__(module_name)


sitk = _ensure_import(
    "SimpleITK",
    "SimpleITK",
)

pydicom = _ensure_import(
    "pydicom",
    "pydicom",
)

from PIL import Image


# ============================================================
# 2 — EXACT FROZEN SAGITTAL ARCHITECTURE
# ============================================================

def _67cd_normalize_state_dict(
    state_dict,
):
    out = {}

    for key, value in state_dict.items():

        clean = str(key)

        for prefix in (
            "module.",
            "model.",
        ):
            if clean.startswith(prefix):
                clean = clean[
                    len(prefix):
                ]

        out[clean] = value

    return out


def _67cd_checkpoint_state_dict(
    checkpoint,
):

    if isinstance(
        checkpoint,
        Mapping,
    ):

        for key in (
            "model_state_dict",
            "state_dict",
            "model",
        ):

            value = checkpoint.get(key)

            if isinstance(
                value,
                Mapping,
            ):
                return (
                    _67cd_normalize_state_dict(
                        value
                    )
                )

        if (
            checkpoint
            and
            all(
                torch.is_tensor(v)
                for v
                in checkpoint.values()
            )
        ):
            return (
                _67cd_normalize_state_dict(
                    checkpoint
                )
            )

    raise ValueError(
        "Frozen sagittal checkpoint "
        "has no usable state_dict"
    )


class _67CD_SagittalDoubleConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                3,
                padding=1,
            ),
            nn.BatchNorm2d(
                out_channels
            ),
            nn.ReLU(
                inplace=True
            ),
            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=1,
            ),
            nn.BatchNorm2d(
                out_channels
            ),
            nn.ReLU(
                inplace=True
            ),
        )

    def forward(
        self,
        x,
    ):
        return self.block(x)


class _67CD_SagittalUNet2D(nn.Module):

    def __init__(
        self,
        in_channels=1,
        num_classes=4,
        base_channels=16,
    ):
        super().__init__()

        b = base_channels

        self.enc1 = _67CD_SagittalDoubleConv(
            in_channels,
            b,
        )

        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = _67CD_SagittalDoubleConv(
            b,
            b * 2,
        )

        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = _67CD_SagittalDoubleConv(
            b * 2,
            b * 4,
        )

        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = (
            _67CD_SagittalDoubleConv(
                b * 4,
                b * 8,
            )
        )

        self.up3 = nn.ConvTranspose2d(
            b * 8,
            b * 4,
            2,
            2,
        )

        self.dec3 = _67CD_SagittalDoubleConv(
            b * 8,
            b * 4,
        )

        self.up2 = nn.ConvTranspose2d(
            b * 4,
            b * 2,
            2,
            2,
        )

        self.dec2 = _67CD_SagittalDoubleConv(
            b * 4,
            b * 2,
        )

        self.up1 = nn.ConvTranspose2d(
            b * 2,
            b,
            2,
            2,
        )

        self.dec1 = _67CD_SagittalDoubleConv(
            b * 2,
            b,
        )

        self.out_conv = nn.Conv2d(
            b,
            num_classes,
            1,
        )

    def forward(
        self,
        x,
    ):

        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool1(e1)
        )

        e3 = self.enc3(
            self.pool2(e2)
        )

        b = self.bottleneck(
            self.pool3(e3)
        )

        d3 = self.dec3(
            torch.cat([
                self.up3(b),
                e3,
            ], dim=1)
        )

        d2 = self.dec2(
            torch.cat([
                self.up2(d3),
                e2,
            ], dim=1)
        )

        d1 = self.dec1(
            torch.cat([
                self.up1(d2),
                e1,
            ], dim=1)
        )

        return self.out_conv(d1)


_frozen_sag_checkpoint = torch.load(
    SAGITTAL_MODEL_PATH,
    map_location=DEVICE_67C_D,
    weights_only=False,
)

_frozen_sag_state = (
    _67cd_checkpoint_state_dict(
        _frozen_sag_checkpoint
    )
)

_frozen_sag_base_channels = (
    int(
        _frozen_sag_checkpoint[
            "base_channels"
        ]
    )
    if (
        isinstance(
            _frozen_sag_checkpoint,
            Mapping,
        )
        and
        _frozen_sag_checkpoint.get(
            "base_channels"
        )
        is not None
    )
    else
    int(
        _frozen_sag_state[
            "enc1.block.0.weight"
        ].shape[0]
    )
)

_frozen_sag_num_classes = (
    int(
        _frozen_sag_checkpoint[
            "num_classes"
        ]
    )
    if (
        isinstance(
            _frozen_sag_checkpoint,
            Mapping,
        )
        and
        _frozen_sag_checkpoint.get(
            "num_classes"
        )
        is not None
    )
    else
    int(
        _frozen_sag_state[
            "out_conv.weight"
        ].shape[0]
    )
)

FROZEN_SAG_TARGET_SIZE_67C_D = (
    tuple(
        int(x)
        for x
        in _frozen_sag_checkpoint.get(
            "target_size",
            (256, 256),
        )
    )
    if isinstance(
        _frozen_sag_checkpoint,
        Mapping,
    )
    else
    (256, 256)
)

assert (
    _frozen_sag_num_classes
    == 4
)

assert (
    FROZEN_SAG_TARGET_SIZE_67C_D
    == (256, 256)
)

# Frozen class contract:
# 0 background
# 1 vertebra_group
# 2 canal
# 3 disc_group
FROZEN_DISC_CLASS_ID_67C_D = 3


sagittal_model_67c_d = (
    _67CD_SagittalUNet2D(
        num_classes=
            _frozen_sag_num_classes,

        base_channels=
            _frozen_sag_base_channels,
    )
)

sagittal_model_67c_d.load_state_dict(
    _frozen_sag_state,
    strict=True,
)

sagittal_model_67c_d.to(
    DEVICE_67C_D
)

sagittal_model_67c_d.eval()

assert (
    sagittal_model_67c_d.training
    is False
)

print()
print(
    "Frozen sagittal model loaded: PASS"
)

print(
    "  base channels:",
    _frozen_sag_base_channels,
)

print(
    "  classes:",
    _frozen_sag_num_classes,
)

print(
    "  target:",
    FROZEN_SAG_TARGET_SIZE_67C_D,
)

print(
    "  disc class:",
    FROZEN_DISC_CLASS_ID_67C_D,
)


# ============================================================
# 3 — FROZEN SAGITTAL PREPROCESSING / GEOMETRY
# ============================================================

def _67cd_robust_percentile_normalize(
    array,
    p_low=1.0,
    p_high=99.0,
):

    value = np.asarray(
        array,
        dtype=np.float32,
    )

    finite = np.isfinite(
        value
    )

    if not finite.any():
        return np.zeros_like(
            value,
            dtype=np.float32,
        )

    low, high = np.percentile(
        value[finite],
        [
            p_low,
            p_high,
        ],
    )

    if float(high) <= float(low):
        return np.zeros_like(
            value,
            dtype=np.float32,
        )

    clipped = np.clip(
        value,
        low,
        high,
    )

    return (
        (
            clipped
            - low
        )
        /
        (
            float(high)
            - float(low)
            + 1e-8
        )
    ).astype(np.float32)


def _67cd_resize_image(
    array,
    target_size,
):

    # Exactly the 67B2 helper semantics:
    # normalize inside resize_image.
    normalized = (
        _67cd_robust_percentile_normalize(
            array
        )
    )

    image = Image.fromarray(
        np.clip(
            normalized * 255.0,
            0,
            255,
        ).astype(np.uint8)
    )

    resized = image.resize(
        (
            target_size[1],
            target_size[0],
        ),
        resample=
            Image.Resampling.BILINEAR,
    )

    return (
        np.asarray(
            resized,
            dtype=np.float32,
        )
        / 255.0
    )


def _67cd_connected_instances(
    binary,
    min_pixels=20,
):

    labelled = (
        sitk.GetArrayFromImage(
            sitk.ConnectedComponent(
                sitk.GetImageFromArray(
                    binary.astype(
                        np.uint8
                    )
                )
            )
        )
    )

    instances = []

    for value in sorted(
        int(x)
        for x
        in np.unique(labelled)
        if int(x) != 0
    ):

        component = (
            labelled
            == value
        )

        if (
            int(
                component.sum()
            )
            >= min_pixels
        ):
            instances.append(
                component
            )

    instances.sort(
        key=lambda mask:
            float(
                np.where(mask)[0].mean()
            )
    )

    return instances


def _67cd_component_geometry(
    mask,
):

    idx = np.where(mask)

    return {
        "centroid_row":
            float(
                np.mean(idx[0])
            ),

        "centroid_col":
            float(
                np.mean(idx[1])
            ),

        "area_px":
            int(mask.sum()),

        "border_touch":
            bool(
                idx[0].min() == 0
                or idx[1].min() == 0
                or idx[0].max()
                    == mask.shape[0] - 1
                or idx[1].max()
                    == mask.shape[1] - 1
            ),
    }


def _67cd_check_orientation_valid(
    iop,
):

    if len(iop) != 6:
        return False

    row = np.asarray(
        iop[:3],
        dtype=np.float64,
    )

    col = np.asarray(
        iop[3:],
        dtype=np.float64,
    )

    return bool(
        abs(
            np.linalg.norm(row)
            - 1.0
        ) < 1e-2
        and
        abs(
            np.linalg.norm(col)
            - 1.0
        ) < 1e-2
        and
        abs(
            float(
                np.dot(
                    row,
                    col,
                )
            )
        ) < 1e-2
    )


def _67cd_check_xy_inside_frame(
    x,
    y,
    rows,
    columns,
):
    return bool(
        0.0 <= float(x) < float(columns)
        and
        0.0 <= float(y) < float(rows)
    )


def _67cd_dicom_pixel_to_patient_xyz(
    column_index,
    row_index,
    pixel_spacing,
    image_position_patient,
    image_orientation_patient,
):

    row_cosines = np.asarray(
        image_orientation_patient[:3],
        dtype=np.float64,
    )

    column_cosines = np.asarray(
        image_orientation_patient[3:],
        dtype=np.float64,
    )

    origin = np.asarray(
        image_position_patient,
        dtype=np.float64,
    )

    row_spacing, column_spacing = (
        pixel_spacing
    )

    return (
        origin
        +
        float(column_index)
        * column_spacing
        * row_cosines
        +
        float(row_index)
        * row_spacing
        * column_cosines
    )


def _67cd_read_sag_metadata(
    path,
):

    tags = [
        "Rows",
        "Columns",
        "PixelSpacing",
        "ImagePositionPatient",
        "ImageOrientationPatient",
        "InstanceNumber",
        "AnatomicalOrientationType",
    ]

    ds = pydicom.dcmread(
        str(path),
        stop_before_pixels=True,
        specific_tags=tags,
    )

    required = [
        "Rows",
        "Columns",
        "PixelSpacing",
        "ImagePositionPatient",
        "ImageOrientationPatient",
        "InstanceNumber",
    ]

    if not all(
        hasattr(ds, tag)
        for tag
        in required
    ):
        return None

    iop = tuple(
        float(v)
        for v
        in ds.ImageOrientationPatient
    )

    ipp = tuple(
        float(v)
        for v
        in ds.ImagePositionPatient
    )

    row_cosines = np.asarray(
        iop[:3],
        dtype=np.float64,
    )

    column_cosines = np.asarray(
        iop[3:],
        dtype=np.float64,
    )

    normal = np.cross(
        row_cosines,
        column_cosines,
    )

    norm = float(
        np.linalg.norm(normal)
    )

    if norm <= 0:
        return None

    normal = normal / norm

    return {
        "path":
            Path(path),

        "position_mm":
            float(
                np.dot(
                    np.asarray(
                        ipp,
                        dtype=np.float64,
                    ),
                    normal,
                )
            ),

        "instance_number":
            int(ds.InstanceNumber),

        "ipp":
            ipp,

        "iop":
            iop,

        "pixel_spacing":
            tuple(
                float(v)
                for v
                in ds.PixelSpacing
            ),

        "rows":
            int(ds.Rows),

        "columns":
            int(ds.Columns),

        "anatomical_orientation_type":
            getattr(
                ds,
                "AnatomicalOrientationType",
                None,
            ),
    }


def _67cd_order_sag_series(
    dicom_paths,
):

    records = []

    with ThreadPoolExecutor(
        max_workers=12
    ) as executor:

        futures = {
            executor.submit(
                _67cd_read_sag_metadata,
                p,
            ): p
            for p
            in dicom_paths
        }

        for future in as_completed(
            futures
        ):
            rec = future.result()

            if rec is not None:
                records.append(rec)

    records.sort(
        key=lambda r:
            r["position_mm"]
    )

    return records


def _67cd_consensus_union_find(
    centroids_xyz,
    distance_threshold_mm=15.0,
):

    n = len(
        centroids_xyz
    )

    parent = list(
        range(n)
    )

    def find(x):

        while parent[x] != x:

            parent[x] = parent[
                parent[x]
            ]

            x = parent[x]

        return x

    def union(a, b):

        ra = find(a)
        rb = find(b)

        if ra != rb:
            parent[rb] = ra

    for i in range(n):

        for j in range(
            i + 1,
            n,
        ):

            distance = (
                np.linalg.norm(
                    np.asarray(
                        centroids_xyz[i]
                    )
                    -
                    np.asarray(
                        centroids_xyz[j]
                    )
                )
            )

            if (
                distance
                <= distance_threshold_mm
            ):
                union(
                    i,
                    j,
                )

    groups = {}

    for i in range(n):
        groups.setdefault(
            find(i),
            [],
        ).append(i)

    return [
        value
        for _,
        value
        in sorted(
            groups.items()
        )
    ]


def _67cd_spine_axis_from_points(
    points_xyz,
):

    centered = (
        points_xyz
        -
        points_xyz.mean(
            axis=0
        )
    )

    _, _, vt = np.linalg.svd(
        centered,
        full_matrices=False,
    )

    axis = vt[0]

    return (
        axis
        /
        np.linalg.norm(axis)
    )


def _67cd_compute_instance_confidence(
    supporting_slice_count,
    centroid_spread_mm,
    mean_segmentation_confidence,
):

    multi_slice_score = min(
        supporting_slice_count
        / 3.0,
        1.0,
    )

    stability_score = float(
        np.clip(
            1.0
            -
            centroid_spread_mm
            / 20.0,
            0.0,
            1.0,
        )
    )

    return round(
        0.40
        * multi_slice_score
        +
        0.30
        * stability_score
        +
        0.30
        * mean_segmentation_confidence,
        4,
    )


# ============================================================
# 4 — FROZEN 67B1 DIRECTION RULE
# ============================================================

REFERENCE_SUPERIOR_DIRECTION_67C_D = (
    np.array(
        [0.0, 0.0, 1.0],
        dtype=np.float64,
    )
)

NUMERICAL_TOLERANCE_67C_D = 1e-6


def _67cd_resolve_axis_direction(
    predicted_axis,
    anatomical_orientation_type=None,
):

    if predicted_axis is None:

        return {
            "status":
                "ABSTAIN",

            "oriented_axis":
                None,

            "reason":
                "NO_PREDICTED_AXIS",
        }

    raw_orientation = (
        None
        if anatomical_orientation_type is None
        else
        str(
            anatomical_orientation_type
        ).strip().upper()
    )

    if (
        raw_orientation
        not in (
            None,
            "",
            "BIPED",
        )
    ):

        return {
            "status":
                "ABSTAIN",

            "oriented_axis":
                None,

            "reason":
                (
                    "UNSUPPORTED_"
                    "ANATOMICAL_ORIENTATION"
                ),
        }

    raw_axis = np.asarray(
        predicted_axis,
        dtype=np.float64,
    )

    alignment = float(
        np.dot(
            raw_axis,
            REFERENCE_SUPERIOR_DIRECTION_67C_D,
        )
    )

    if (
        abs(alignment)
        <= NUMERICAL_TOLERANCE_67C_D
    ):

        return {
            "status":
                "ABSTAIN",

            "oriented_axis":
                None,

            "reason":
                "AXIS_ORTHOGONAL_TO_PATIENT_Z",
        }

    oriented = (
        -raw_axis
        if alignment > 0
        else raw_axis
    )

    return {
        "status":
            "RESOLVED",

        "oriented_axis":
            oriented,

        "reason":
            "SUPERIOR_TO_INFERIOR",
    }


# ============================================================
# 5 — SAGITTAL INFERENCE FOR ONE STUDY
#
# This reproduces frozen prediction geometry.
# NO GT.
# ============================================================

def _67cd_run_sagittal_study(
    study_id_raw,
):

    study_rows = (
        series_df_67c_d[
            series_df_67c_d[
                "study_id"
            ].astype(str)
            ==
            str(study_id_raw)
        ]
    )

    sag_rows = study_rows[
        study_rows[
            "series_description"
        ]
        .astype(str)
        .str.strip()
        ==
        "Sagittal T2/STIR"
    ]

    if sag_rows.empty:

        return {
            "status":
                "NO_SAGITTAL_T2",
        }

    sag_series_id = str(
        sag_rows.iloc[0][
            "series_id"
        ]
    )

    series_dir = (
        TRAIN_IMAGES_ROOT_67C_D
        / str(study_id_raw)
        / sag_series_id
    )

    paths = sorted(
        series_dir.glob(
            "*.dcm"
        )
    )

    if not paths:

        return {
            "status":
                "NO_SAGITTAL_DICOM",
        }

    slices = (
        _67cd_order_sag_series(
            paths
        )
    )

    if len(slices) < 2:

        return {
            "status":
                "INSUFFICIENT_SAGITTAL_SLICES",
        }

    # ----------------------------
    # Read/preprocess all slices
    # ----------------------------

    native_arrays = []
    prepared_arrays = []

    valid_slices = []

    for rec in slices:

        ds = pydicom.dcmread(
            str(
                rec["path"]
            )
        )

        native = (
            ds.pixel_array
            .astype(np.float32)
        )

        # Same semantic sequence used in frozen Stage C:
        normalized = (
            _67cd_robust_percentile_normalize(
                native
            )
        )

        prepared = (
            _67cd_resize_image(
                normalized,
                FROZEN_SAG_TARGET_SIZE_67C_D,
            )
        )

        native_arrays.append(
            native
        )

        prepared_arrays.append(
            prepared
        )

        valid_slices.append(
            rec
        )

    if not prepared_arrays:

        return {
            "status":
                "NO_PREPARED_SLICES",
        }

    # ----------------------------
    # Batched L4 inference
    # ----------------------------

    batch = torch.from_numpy(
        np.stack(
            prepared_arrays,
            axis=0,
        )[:, None, :, :]
        .astype(np.float32)
    )

    predicted_components = []

    BATCH_SIZE = 32

    for start in range(
        0,
        len(batch),
        BATCH_SIZE,
    ):

        x = batch[
            start:
            start + BATCH_SIZE
        ].to(
            DEVICE_67C_D,
            non_blocking=True,
        )

        with torch.inference_mode():

            logits = (
                sagittal_model_67c_d(
                    x
                )
            )

            probs = torch.softmax(
                logits,
                dim=1,
            )

            preds = torch.argmax(
                probs,
                dim=1,
            ).cpu().numpy()

            confs = torch.max(
                probs,
                dim=1,
            ).values.cpu().numpy()

        for local_idx in range(
            len(preds)
        ):

            global_idx = (
                start
                + local_idx
            )

            prediction = (
                preds[
                    local_idx
                ].astype(np.uint8)
            )

            confidence = (
                confs[
                    local_idx
                ].astype(np.float32)
            )

            disc_mask = (
                prediction
                ==
                FROZEN_DISC_CLASS_ID_67C_D
            )

            native = (
                native_arrays[
                    global_idx
                ]
            )

            rec = (
                valid_slices[
                    global_idx
                ]
            )

            for component in (
                _67cd_connected_instances(
                    disc_mask
                )
            ):

                geom = (
                    _67cd_component_geometry(
                        component
                    )
                )

                row_native = (
                    geom[
                        "centroid_row"
                    ]
                    *
                    (
                        native.shape[0]
                        /
                        FROZEN_SAG_TARGET_SIZE_67C_D[0]
                    )
                )

                col_native = (
                    geom[
                        "centroid_col"
                    ]
                    *
                    (
                        native.shape[1]
                        /
                        FROZEN_SAG_TARGET_SIZE_67C_D[1]
                    )
                )

                if not (
                    _67cd_check_xy_inside_frame(
                        col_native,
                        row_native,
                        rec["rows"],
                        rec["columns"],
                    )
                ):
                    continue

                if not (
                    _67cd_check_orientation_valid(
                        rec["iop"]
                    )
                ):
                    continue

                xyz = (
                    _67cd_dicom_pixel_to_patient_xyz(
                        col_native,
                        row_native,
                        rec[
                            "pixel_spacing"
                        ],
                        rec["ipp"],
                        rec["iop"],
                    )
                )

                if not np.all(
                    np.isfinite(xyz)
                ):
                    continue

                predicted_components.append({

                    "slice_index":
                        global_idx,

                    "centroid_xyz":
                        xyz,

                    "area_px":
                        geom[
                            "area_px"
                        ],

                    "mean_confidence":
                        float(
                            confidence.mean()
                        ),
                })

    # ----------------------------
    # 15 mm multi-slice consensus
    # ----------------------------

    consensus_instances = []

    if predicted_components:

        groups = (
            _67cd_consensus_union_find(
                [
                    c["centroid_xyz"]
                    for c
                    in predicted_components
                ],
                distance_threshold_mm=
                    15.0,
            )
        )

        for indices in groups:

            members = [
                predicted_components[i]
                for i
                in indices
            ]

            centroids = np.stack([
                m["centroid_xyz"]
                for m
                in members
            ])

            mean_centroid = (
                centroids.mean(
                    axis=0
                )
            )

            spread = (
                float(
                    np.max(
                        np.linalg.norm(
                            centroids
                            -
                            mean_centroid,
                            axis=1,
                        )
                    )
                )
                if len(members) > 1
                else 0.0
            )

            instance_confidence = (
                _67cd_compute_instance_confidence(
                    len(members),
                    spread,
                    float(
                        np.mean([
                            m[
                                "mean_confidence"
                            ]
                            for m
                            in members
                        ])
                    ),
                )
            )

            consensus_instances.append({

                "centroid_xyz":
                    mean_centroid,

                "supporting_slice_count":
                    len(members),

                "instance_confidence":
                    instance_confidence,
            })

    if (
        len(
            consensus_instances
        )
        < 2
    ):

        return {
            "status":
                "INSUFFICIENT_CONSENSUS_INSTANCES",

            "predicted_instance_count":
                len(
                    consensus_instances
                ),
        }

    # Raw SVD axis.
    pred_points = np.stack([
        c["centroid_xyz"]
        for c
        in consensus_instances
    ])

    raw_axis = (
        _67cd_spine_axis_from_points(
            pred_points
        )
    )

    anatomical_orientation = (
        valid_slices[0].get(
            "anatomical_orientation_type"
        )
    )

    direction = (
        _67cd_resolve_axis_direction(
            raw_axis,
            anatomical_orientation,
        )
    )

    if (
        direction[
            "status"
        ]
        != "RESOLVED"
    ):

        return {
            "status":
                "DIRECTION_ABSTAIN",

            "reason":
                direction[
                    "reason"
                ],

            "predicted_instance_count":
                len(
                    consensus_instances
                ),
        }

    oriented_axis = (
        direction[
            "oriented_axis"
        ]
    )

    reference_point = (
        pred_points[0].copy()
    )

    for c in consensus_instances:

        c[
            "position_along_axis_mm"
        ] = float(
            np.dot(
                np.asarray(
                    c[
                        "centroid_xyz"
                    ]
                )
                -
                reference_point,
                oriented_axis,
            )
        )

    ordered_instances = sorted(
        consensus_instances,
        key=lambda c:
            c[
                "position_along_axis_mm"
            ],
    )

    return {
        "status":
            "EXECUTED",

        "predicted_instance_count":
            len(
                ordered_instances
            ),

        "ordered_instances":
            ordered_instances,
    }


# ============================================================
# 6 — AXIAL DICOM GEOMETRY
#
# NO LABEL FILE.
# ============================================================

def _67cd_read_axial_geometry(
    path,
):

    tags = [
        "Rows",
        "Columns",
        "PixelSpacing",
        "ImagePositionPatient",
        "ImageOrientationPatient",
        "InstanceNumber",
    ]

    try:

        ds = pydicom.dcmread(
            str(path),
            stop_before_pixels=True,
            specific_tags=tags,
        )

        required = [
            "Rows",
            "Columns",
            "PixelSpacing",
            "ImagePositionPatient",
            "ImageOrientationPatient",
            "InstanceNumber",
        ]

        if not all(
            hasattr(ds, tag)
            for tag
            in required
        ):
            return None

        iop = tuple(
            float(v)
            for v
            in ds.ImageOrientationPatient
        )

        ipp = np.asarray(
            [
                float(v)
                for v
                in ds.ImagePositionPatient
            ],
            dtype=np.float64,
        )

        row_cosines = np.asarray(
            iop[:3],
            dtype=np.float64,
        )

        column_cosines = np.asarray(
            iop[3:],
            dtype=np.float64,
        )

        if not (
            _67cd_check_orientation_valid(
                iop
            )
        ):
            return None

        normal = np.cross(
            row_cosines,
            column_cosines,
        )

        normal_norm = float(
            np.linalg.norm(
                normal
            )
        )

        if (
            normal_norm
            <= 1e-12
        ):
            return None

        normal = (
            normal
            /
            normal_norm
        )

        pixel_spacing = tuple(
            float(v)
            for v
            in ds.PixelSpacing
        )

        if (
            pixel_spacing[0] <= 0
            or
            pixel_spacing[1] <= 0
        ):
            return None

        return {

            "path":
                Path(path),

            "instance_number":
                int(
                    ds.InstanceNumber
                ),

            "ipp":
                ipp,

            "iop":
                iop,

            "row_cosines":
                row_cosines,

            "column_cosines":
                column_cosines,

            "normal":
                normal,

            "pixel_spacing":
                pixel_spacing,

            "rows":
                int(
                    ds.Rows
                ),

            "columns":
                int(
                    ds.Columns
                ),
        }

    except Exception:
        return None


def _67cd_project_point_to_axial_slice(
    P,
    geom,
):

    P = np.asarray(
        P,
        dtype=np.float64,
    )

    O = geom["ipp"]
    n = geom["normal"]

    row_spacing, column_spacing = (
        geom["pixel_spacing"]
    )

    row_cosines = (
        geom["row_cosines"]
    )

    column_cosines = (
        geom["column_cosines"]
    )

    signed_distance = float(
        np.dot(
            P - O,
            n,
        )
    )

    plane_distance = abs(
        signed_distance
    )

    projected = (
        P
        -
        signed_distance
        * n
    )

    diff = (
        projected
        -
        O
    )

    projected_col = (
        float(
            np.dot(
                diff,
                row_cosines,
            )
        )
        /
        column_spacing
    )

    projected_row = (
        float(
            np.dot(
                diff,
                column_cosines,
            )
        )
        /
        row_spacing
    )

    inside_fov = (
        _67cd_check_xy_inside_frame(
            projected_col,
            projected_row,
            geom["rows"],
            geom["columns"],
        )
    )

    return {
        "signed_distance_mm":
            signed_distance,

        "plane_distance_mm":
            plane_distance,

        "projected_row_px":
            projected_row,

        "projected_col_px":
            projected_col,

        "inside_fov":
            inside_fov,
    }


def _67cd_get_axial_geometry(
    study_id_raw,
):

    study_rows = (
        series_df_67c_d[
            series_df_67c_d[
                "study_id"
            ].astype(str)
            ==
            str(study_id_raw)
        ]
    )

    axial_rows = study_rows[
        study_rows[
            "series_description"
        ]
        .astype(str)
        .str.lower()
        .str.contains(
            "ax",
            na=False,
        )
    ]

    jobs = []

    for _, series_row in (
        axial_rows.iterrows()
    ):

        series_id = str(
            series_row[
                "series_id"
            ]
        )

        series_description = str(
            series_row[
                "series_description"
            ]
        )

        series_dir = (
            TRAIN_IMAGES_ROOT_67C_D
            /
            str(
                study_id_raw
            )
            /
            series_id
        )

        for path in (
            series_dir.glob(
                "*.dcm"
            )
            if series_dir.is_dir()
            else []
        ):

            jobs.append((
                series_id,
                series_description,
                path,
            ))

    rows = []

    with ThreadPoolExecutor(
        max_workers=16
    ) as executor:

        future_map = {
            executor.submit(
                _67cd_read_axial_geometry,
                path,
            ):
            (
                series_id,
                series_description,
            )
            for (
                series_id,
                series_description,
                path,
            )
            in jobs
        }

        for future in as_completed(
            future_map
        ):

            geom = future.result()

            if geom is None:
                continue

            series_id, description = (
                future_map[
                    future
                ]
            )

            geom[
                "series_id_raw"
            ] = series_id

            geom[
                "series_id_opaque"
            ] = opaque_id(
                series_id
            )

            geom[
                "series_description"
            ] = description

            rows.append(
                geom
            )

    return rows


# ============================================================
# 7 — EXACT FROZEN 67C-C SELECTION POLICY
# ============================================================

def _67cd_apply_frozen_pairing(
    candidates,
):

    if not candidates:

        return {
            "emit":
                False,

            "reason":
                "NO_CANDIDATES",

            "candidate":
                None,

            "margin_mm":
                None,
        }

    ranked = sorted(
        candidates,
        key=lambda c: (
            not c[
                "inside_fov"
            ],
            c[
                "plane_distance_mm"
            ],
        ),
    )

    winner = ranked[0]

    if not bool(
        winner[
            "inside_fov"
        ]
    ):

        return {
            "emit":
                False,

            "reason":
                "OUTSIDE_FOV",

            "candidate":
                winner,

            "margin_mm":
                None,
        }

    if (
        float(
            winner[
                "plane_distance_mm"
            ]
        )
        >
        FROZEN_67C_MAX_PLANE_DISTANCE_MM
    ):

        return {
            "emit":
                False,

            "reason":
                "PLANE_DISTANCE",

            "candidate":
                winner,

            "margin_mm":
                None,
        }

    if len(ranked) < 2:

        return {
            "emit":
                False,

            "reason":
                "NO_SECOND_CANDIDATE",

            "candidate":
                winner,

            "margin_mm":
                None,
        }

    margin = (
        float(
            ranked[1][
                "plane_distance_mm"
            ]
        )
        -
        float(
            ranked[0][
                "plane_distance_mm"
            ]
        )
    )

    if (
        margin
        <
        FROZEN_67C_MIN_MARGIN_MM
    ):

        return {
            "emit":
                False,

            "reason":
                "AMBIGUOUS_MARGIN",

            "candidate":
                winner,

            "margin_mm":
                margin,
        }

    return {
        "emit":
            True,

        "reason":
            "PASS",

        "candidate":
            winner,

        "margin_mm":
            margin,
    }


# ============================================================
# 8 — OUTPUT CONTRACT
# ============================================================

LOCKED_67C_D_DIR = (
    METRICS_67C_DIR
    / "locked_validation"
)

LOCKED_67C_D_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PREDICTIONS_67C_D_PATH = (
    LOCKED_67C_D_DIR
    / "67C_D_gt_free_predictions.csv"
)

STUDY_SUMMARY_67C_D_PATH = (
    LOCKED_67C_D_DIR
    / "67C_D_gt_free_study_summary.csv"
)

PREDICTION_CONTRACT_67C_D_PATH = (
    LOCKED_67C_D_DIR
    / "67C_D_prediction_contract.json"
)


validation_split_sha_67c_d = (
    hashlib.sha256(
        "\n".join(
            sorted(
                str(x)
                for x
                in LOCKED_67C_VALIDATION_RAW
            )
        ).encode(
            "utf-8"
        )
    ).hexdigest()
)


prediction_contract_67c_d = {

    "stage":
        "67C-D_GT_FREE_PREDICTION",

    "validation_studies":
        296,

    "validation_split_sha256":
        validation_split_sha_67c_d,

    "67C_C_freeze_sha256":
        EXPECTED_67C_FREEZE_SHA256,

    "sagittal_checkpoint_sha256":
        EXPECTED_SAGITTAL_SHA256,

    "67B2_locked_predictions_sha256":
        EXPECTED_67B2_LOCKED_PREDICTIONS_SHA256,

    "pairing_method":
        "BASELINE_A_NEAREST_PLANE_PLUS_FOV",

    "min_margin_mm":
        FROZEN_67C_MIN_MARGIN_MM,

    "max_plane_distance_mm":
        FROZEN_67C_MAX_PLANE_DISTANCE_MM,

    "upstream_decision_source":
        "67B2_locked_validation_predictions.csv",

    "selected_window_reranked":
        False,

    "axial_validation_gt_accessed":
        False,

    "internal_test_accessed":
        False,

    "official_test_accessed":
        False,

    "post_prediction_tuning_allowed":
        False,
}


if (
    PREDICTION_CONTRACT_67C_D_PATH
    .is_file()
):

    existing_contract = json.loads(
        PREDICTION_CONTRACT_67C_D_PATH
        .read_text(
            encoding="utf-8"
        )
    )

    assert (
        existing_contract
        ==
        prediction_contract_67c_d
    ), (
        "HARD STOP — existing 67C-D "
        "prediction contract differs."
    )

    print()
    print(
        "Existing identical "
        "67C-D prediction contract: PASS"
    )

else:

    PREDICTION_CONTRACT_67C_D_PATH.write_text(
        json.dumps(
            prediction_contract_67c_d,
            indent=2,
        ),
        encoding="utf-8",
    )

    print()
    print(
        "67C-D prediction contract created."
    )


# ============================================================
# 9 — MAP LOCKED 67B2 DECISIONS
# ============================================================

locked_by_opaque = (
    locked_67b2_predictions
    .set_index(
        "study_id_opaque",
        drop=False,
    )
)


raw_by_opaque = {
    opaque_id(sid):
        str(sid)

    for sid
    in LOCKED_67C_VALIDATION_RAW
}


assert (
    set(
        locked_by_opaque.index.astype(str)
    )
    ==
    set(
        raw_by_opaque.keys()
    )
)


# ============================================================
# 10 — RUN GT-FREE PREDICTION
#
# We process ALL 296 studies.
#
# For upstream abstentions:
#   no sagittal rerun is needed.
#
# For upstream emitted studies:
#   rerun frozen segmentation ONLY to reconstruct the five
#   predicted physical XYZ coordinates corresponding to the
#   already-frozen selected_window_index.
# ============================================================

all_prediction_rows = []
study_summary_rows = []

t0 = time.time()

emitted_expected = int(
    locked_67b2_predictions[
        "emitted_absolute_sequence"
    ].astype(bool).sum()
)

assert emitted_expected == 226


for study_idx, (
    opaque,
    locked_row,
) in enumerate(
    locked_by_opaque.iterrows(),
    start=1,
):

    opaque = str(
        opaque
    )

    raw_sid = (
        raw_by_opaque[
            opaque
        ]
    )

    upstream_emit = bool(
        locked_row[
            "emitted_absolute_sequence"
        ]
    )

    upstream_decision = str(
        locked_row[
            "decision_status"
        ]
    )

    study_summary = {

        "study_id_opaque":
            opaque,

        "upstream_67B2_emit":
            upstream_emit,

        "upstream_67B2_decision_status":
            upstream_decision,

        "upstream_selected_window_index":
            (
                None
                if pd.isna(
                    locked_row[
                        "selected_window_index"
                    ]
                )
                else
                int(
                    locked_row[
                        "selected_window_index"
                    ]
                )
            ),

        "sagittal_runtime_status":
            "NOT_RUN_UPSTREAM_ABSTAIN",

        "predicted_instance_count":
            None,

        "pairing_levels_attempted":
            0,

        "pairing_levels_emitted":
            0,

        "pairing_study_status":
            "UPSTREAM_ABSTAIN",
    }


    # ----------------------------------------
    # Upstream abstention is final.
    # ----------------------------------------

    if not upstream_emit:

        for level in (
            CANONICAL_LEVELS_67C_D
        ):

            all_prediction_rows.append({

                "study_id_opaque":
                    opaque,

                "level":
                    level,

                "upstream_67B2_emit":
                    False,

                "upstream_67B2_decision_status":
                    upstream_decision,

                "upstream_selected_window_index":
                    study_summary[
                        "upstream_selected_window_index"
                    ],

                "predicted_disc_x_mm":
                    None,

                "predicted_disc_y_mm":
                    None,

                "predicted_disc_z_mm":
                    None,

                "pairing_emit":
                    False,

                "pairing_reason":
                    "UPSTREAM_67B2_ABSTAIN",

                "selected_series_id_opaque":
                    None,

                "selected_instance_number":
                    None,

                "selected_plane_distance_mm":
                    None,

                "distance_margin_mm":
                    None,

                "selected_inside_fov":
                    None,

                "projected_row_px":
                    None,

                "projected_col_px":
                    None,
            })

        study_summary_rows.append(
            study_summary
        )

        continue


    # ----------------------------------------
    # Frozen sagittal inference
    # ----------------------------------------

    try:

        sag_run = (
            _67cd_run_sagittal_study(
                raw_sid
            )
        )

    except Exception as exc:

        sag_run = {
            "status":
                (
                    "RUNTIME_EXCEPTION:"
                    f"{type(exc).__name__}"
                )
        }


    study_summary[
        "sagittal_runtime_status"
    ] = sag_run.get(
        "status"
    )


    if (
        sag_run.get(
            "status"
        )
        != "EXECUTED"
    ):

        study_summary[
            "pairing_study_status"
        ] = "SAGITTAL_RUNTIME_FAILURE"

        for level in (
            CANONICAL_LEVELS_67C_D
        ):

            all_prediction_rows.append({

                "study_id_opaque":
                    opaque,

                "level":
                    level,

                "upstream_67B2_emit":
                    True,

                "upstream_67B2_decision_status":
                    upstream_decision,

                "upstream_selected_window_index":
                    study_summary[
                        "upstream_selected_window_index"
                    ],

                "predicted_disc_x_mm":
                    None,

                "predicted_disc_y_mm":
                    None,

                "predicted_disc_z_mm":
                    None,

                "pairing_emit":
                    False,

                "pairing_reason":
                    "SAGITTAL_RUNTIME_FAILURE",

                "selected_series_id_opaque":
                    None,

                "selected_instance_number":
                    None,

                "selected_plane_distance_mm":
                    None,

                "distance_margin_mm":
                    None,

                "selected_inside_fov":
                    None,

                "projected_row_px":
                    None,

                "projected_col_px":
                    None,
            })

        study_summary_rows.append(
            study_summary
        )

        continue


    ordered_instances = (
        sag_run[
            "ordered_instances"
        ]
    )

    n_instances = len(
        ordered_instances
    )

    study_summary[
        "predicted_instance_count"
    ] = n_instances


    # ----------------------------------------
    # Critical parity against 67B2 locked run
    # ----------------------------------------

    locked_pred_count = (
        None
        if pd.isna(
            locked_row[
                "predicted_instance_count"
            ]
        )
        else
        int(
            locked_row[
                "predicted_instance_count"
            ]
        )
    )

    assert (
        locked_pred_count
        ==
        n_instances
    ), (
        "HARD STOP — frozen sagittal parity "
        f"failed for {opaque}: "
        f"locked={locked_pred_count}, "
        f"recomputed={n_instances}"
    )


    selected_window_index = int(
        locked_row[
            "selected_window_index"
        ]
    )


    # Candidate window definition from 67B1/67B2:
    # window index w => ranks w+1 ... w+5.
    start_idx = (
        selected_window_index
    )

    end_idx = (
        start_idx
        + 5
    )


    assert (
        0
        <= start_idx
        <
        n_instances
    )

    assert (
        end_idx
        <= n_instances
    )


    selected_instances = (
        ordered_instances[
            start_idx:
            end_idx
        ]
    )


    assert (
        len(
            selected_instances
        )
        == 5
    )


    predicted_xyz_by_level = {

        level:
            np.asarray(
                selected_instances[i][
                    "centroid_xyz"
                ],
                dtype=np.float64,
            )

        for i, level
        in enumerate(
            CANONICAL_LEVELS_67C_D
        )
    }


    # ----------------------------------------
    # Axial geometry — image metadata ONLY
    # ----------------------------------------

    axial_geometry = (
        _67cd_get_axial_geometry(
            raw_sid
        )
    )


    study_summary[
        "pairing_levels_attempted"
    ] = 5


    emitted_levels = 0


    for level in (
        CANONICAL_LEVELS_67C_D
    ):

        P = (
            predicted_xyz_by_level[
                level
            ]
        )

        candidates = []

        for geom in (
            axial_geometry
        ):

            proj = (
                _67cd_project_point_to_axial_slice(
                    P,
                    geom,
                )
            )

            candidates.append({

                "series_id_opaque":
                    geom[
                        "series_id_opaque"
                    ],

                "instance_number":
                    geom[
                        "instance_number"
                    ],

                "plane_distance_mm":
                    proj[
                        "plane_distance_mm"
                    ],

                "signed_plane_distance_mm":
                    proj[
                        "signed_distance_mm"
                    ],

                "inside_fov":
                    proj[
                        "inside_fov"
                    ],

                "projected_row_px":
                    proj[
                        "projected_row_px"
                    ],

                "projected_col_px":
                    proj[
                        "projected_col_px"
                    ],
            })


        decision = (
            _67cd_apply_frozen_pairing(
                candidates
            )
        )

        winner = (
            decision[
                "candidate"
            ]
        )

        if (
            decision[
                "emit"
            ]
        ):
            emitted_levels += 1


        all_prediction_rows.append({

            "study_id_opaque":
                opaque,

            "level":
                level,

            "upstream_67B2_emit":
                True,

            "upstream_67B2_decision_status":
                upstream_decision,

            "upstream_selected_window_index":
                selected_window_index,

            "predicted_disc_x_mm":
                float(
                    P[0]
                ),

            "predicted_disc_y_mm":
                float(
                    P[1]
                ),

            "predicted_disc_z_mm":
                float(
                    P[2]
                ),

            "pairing_emit":
                bool(
                    decision[
                        "emit"
                    ]
                ),

            "pairing_reason":
                decision[
                    "reason"
                ],

            "selected_series_id_opaque":
                (
                    winner[
                        "series_id_opaque"
                    ]
                    if winner
                    else None
                ),

            "selected_instance_number":
                (
                    winner[
                        "instance_number"
                    ]
                    if winner
                    else None
                ),

            "selected_plane_distance_mm":
                (
                    float(
                        winner[
                            "plane_distance_mm"
                        ]
                    )
                    if winner
                    else None
                ),

            "distance_margin_mm":
                (
                    float(
                        decision[
                            "margin_mm"
                        ]
                    )
                    if (
                        decision[
                            "margin_mm"
                        ]
                        is not None
                    )
                    else None
                ),

            "selected_inside_fov":
                (
                    bool(
                        winner[
                            "inside_fov"
                        ]
                    )
                    if winner
                    else None
                ),

            "projected_row_px":
                (
                    float(
                        winner[
                            "projected_row_px"
                        ]
                    )
                    if winner
                    else None
                ),

            "projected_col_px":
                (
                    float(
                        winner[
                            "projected_col_px"
                        ]
                    )
                    if winner
                    else None
                ),
        })


    study_summary[
        "pairing_levels_emitted"
    ] = emitted_levels

    study_summary[
        "pairing_study_status"
    ] = (
        "ALL_5_EMITTED"
        if emitted_levels == 5
        else
        "PARTIAL_PAIRING"
        if emitted_levels > 0
        else
        "PAIRING_ABSTAIN"
    )


    study_summary_rows.append(
        study_summary
    )


    if (
        study_idx % 16 == 0
        or
        study_idx == 296
    ):

        elapsed = (
            time.time()
            - t0
        )

        print(
            f"67C-D prediction "
            f"{study_idx}/296 "
            f"| elapsed="
            f"{elapsed/60:.1f} min"
        )

        # Incremental crash-safe snapshots.
        pd.DataFrame(
            all_prediction_rows
        ).to_csv(
            PREDICTIONS_67C_D_PATH,
            index=False,
        )

        pd.DataFrame(
            study_summary_rows
        ).to_csv(
            STUDY_SUMMARY_67C_D_PATH,
            index=False,
        )


# ============================================================
# 11 — FINAL GT-FREE ARTIFACTS
# ============================================================

predictions_67c_d = pd.DataFrame(
    all_prediction_rows
)

study_summary_67c_d = pd.DataFrame(
    study_summary_rows
)


assert len(
    study_summary_67c_d
) == 296

assert (
    study_summary_67c_d[
        "study_id_opaque"
    ].nunique()
    == 296
)

assert len(
    predictions_67c_d
) == (
    296 * 5
)

assert (
    predictions_67c_d[
        [
            "study_id_opaque",
            "level",
        ]
    ]
    .duplicated()
    .sum()
    == 0
)


predictions_67c_d.to_csv(
    PREDICTIONS_67C_D_PATH,
    index=False,
)

study_summary_67c_d.to_csv(
    STUDY_SUMMARY_67C_D_PATH,
    index=False,
)


PREDICTIONS_67C_D_SHA256 = (
    sha256_file(
        PREDICTIONS_67C_D_PATH
    )
)

STUDY_SUMMARY_67C_D_SHA256 = (
    sha256_file(
        STUDY_SUMMARY_67C_D_PATH
    )
)


# ============================================================
# 12 — PRE-GT SUMMARY
#
# NO ACCURACY IS CALCULATED HERE.
# ============================================================

upstream_emitted_studies = int(
    study_summary_67c_d[
        "upstream_67B2_emit"
    ].astype(bool).sum()
)

pairing_attempted_levels = int(
    (
        predictions_67c_d[
            "upstream_67B2_emit"
        ].astype(bool)
    ).sum()
)

pairing_emitted_levels = int(
    (
        predictions_67c_d[
            "pairing_emit"
        ].astype(bool)
    ).sum()
)

conditional_pairing_coverage_pre_gt = (
    pairing_emitted_levels
    /
    pairing_attempted_levels
    if pairing_attempted_levels
    else None
)

full_pipeline_level_coverage_pre_gt = (
    pairing_emitted_levels
    /
    (296 * 5)
)


prediction_summary_pre_gt = {

    "stage":
        "67C-D_PREDICTION_COMPLETE_GT_STILL_LOCKED",

    "validation_studies":
        296,

    "upstream_67B2_emitted_studies":
        upstream_emitted_studies,

    "pairing_attempted_levels":
        pairing_attempted_levels,

    "pairing_emitted_levels":
        pairing_emitted_levels,

    "conditional_pairing_coverage_pre_gt":
        conditional_pairing_coverage_pre_gt,

    "full_pipeline_level_coverage_pre_gt":
        full_pipeline_level_coverage_pre_gt,

    "pairing_reason_counts":
        predictions_67c_d[
            "pairing_reason"
        ]
        .value_counts(
            dropna=False
        )
        .to_dict(),

    "study_status_counts":
        study_summary_67c_d[
            "pairing_study_status"
        ]
        .value_counts(
            dropna=False
        )
        .to_dict(),

    "predictions_sha256":
        PREDICTIONS_67C_D_SHA256,

    "study_summary_sha256":
        STUDY_SUMMARY_67C_D_SHA256,

    "67C_C_freeze_sha256":
        EXPECTED_67C_FREEZE_SHA256,

    "axial_validation_gt_accessed":
        False,

    "internal_test_accessed":
        False,

    "official_test_accessed":
        False,

    "AXIAL_PAIRING_VALIDATED_67C":
        False,
}


PRE_GT_SUMMARY_PATH_67C_D = (
    LOCKED_67C_D_DIR
    /
    "67C_D_prediction_summary_pre_gt.json"
)


PRE_GT_SUMMARY_PATH_67C_D.write_text(
    json.dumps(
        prediction_summary_pre_gt,
        indent=2,
    ),
    encoding="utf-8",
)


PRE_GT_SUMMARY_SHA256_67C_D = (
    sha256_file(
        PRE_GT_SUMMARY_PATH_67C_D
    )
)


# ============================================================
# 13 — FINAL SAFETY ASSERTIONS
# ============================================================

ACCESS_OUTER_VALIDATION_AXIAL_GT = False
RUN_LOCKED_VALIDATION = False

assert (
    ACCESS_OUTER_VALIDATION_AXIAL_GT
    is False
)

assert RUN_LOCKED_VALIDATION is False

assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False

assert AXIAL_PAIRING_VALIDATED_67C is False
assert TRAINING_PERFORMED is False


# ============================================================
# 14 — CLOSING BLOCK
# ============================================================

print()
print("=" * 68)
print("67C-D GT-FREE PREDICTION PHASE COMPLETE")
print("=" * 68)

print()

print(
    "Validation studies:",
    len(
        study_summary_67c_d
    ),
)

print(
    "67B2 upstream emitted studies:",
    upstream_emitted_studies,
    "/ 296",
)

print()

print(
    "Pairing attempted levels:",
    pairing_attempted_levels,
)

print(
    "Pairing emitted levels:",
    pairing_emitted_levels,
)

print(
    "Conditional pairing coverage:",
    conditional_pairing_coverage_pre_gt,
)

print(
    "Full pipeline level coverage:",
    full_pipeline_level_coverage_pre_gt,
)

print()

print("Pairing reasons:")

print(
    predictions_67c_d[
        "pairing_reason"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)

print()

print("Study statuses:")

print(
    study_summary_67c_d[
        "pairing_study_status"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)

print()

print(
    "Predictions:",
    PREDICTIONS_67C_D_PATH,
)

print(
    "Predictions SHA256:",
    PREDICTIONS_67C_D_SHA256,
)

print()

print(
    "Study summary:",
    STUDY_SUMMARY_67C_D_PATH,
)

print(
    "Study summary SHA256:",
    STUDY_SUMMARY_67C_D_SHA256,
)

print()

print(
    "Pre-GT summary:",
    PRE_GT_SUMMARY_PATH_67C_D,
)

print(
    "Pre-GT summary SHA256:",
    PRE_GT_SUMMARY_SHA256_67C_D,
)

print()

print(
    "Outer validation axial GT accessed:",
    ACCESS_OUTER_VALIDATION_AXIAL_GT,
)

print(
    "Internal test accessed:",
    False,
)

print(
    "Official test accessed:",
    OFFICIAL_TEST_ACCESSED,
)

print()

print(
    "AXIAL_PAIRING_VALIDATED_67C:",
    AXIAL_PAIRING_VALIDATED_67C,
)

print()

print(
    "PREDICTIONS ARE NOW FROZEN."
)

print(
    "NEXT: OPEN AXIAL VALIDATION GT ONCE "
    "AND EVALUATE THE PREDECLARED GATES."
)